# FloodCheck by Atlas Geosystems — run in Colab

This notebook builds the same project as the `floodcheck/` repository, file for file, and starts it in Colab. Search for your place or mark an area anywhere in the country, choose the rain, press **Check for floods**, and the flood map appears.

| File | What it is |
|---|---|
| `backend/floodmodel.py` | The flood model: terrain, runoff, river flow and flood depth |
| `backend/exposure.py` | Finds the buildings, roads, schools and health facilities in the flood water |
| `backend/weather.py` | Real rainfall, ground wetness and river flow for any date (past or forecast) |
| `backend/imagery.py` | Sentinel-1 radar and Sentinel-2 photos for any date, as map tiles |
| `backend/geocode.py` | Place search: village, town, school or market name (or coordinates) to a map location |
| `backend/server.py` | The web server: runs the model in the background and sends results to the page |
| `backend/static/index.html` | The dashboard page |
| `backend/static/css/floodcheck.css` | Dashboard styles (Atlas Geosystems colours) |
| `backend/static/js/floodcheck.js` | Dashboard behaviour: map, inputs, runs, results |
| `backend/static/assets/` | Atlas Geosystems logo files |

**How to use it:** choose **Runtime → Run all**. The last cell prints a link. Keep this tab open while you use the app: if the notebook stops, the app stops.

To run the project on a computer or server instead of Colab, use the repository and follow its `README.md`.

In [ ]:
!pip install -q rasterio numba scipy fastapi uvicorn pillow shapely pyarrow overturemaps rio-tiler pystac-client planetary-computer reportlab
!mkdir -p /content/floodcheck/backend/static/css /content/floodcheck/backend/static/js /content/floodcheck/backend/static/assets /content/floodcheck/results

## 1. The flood model

Same method as the original modelling notebook. The slow part (downloading data and tracing water flow) is kept in memory for the last two areas, so trying a second storm in the same area is much faster.

In [ ]:
%%writefile /content/floodcheck/backend/floodmodel.py
"""Rapid flood model: rain + terrain -> flood depth, for any area in the country.

Same methodology as the original modelling notebook, reorganised into functions so a web server can call it:
  load_terrain()        gets the terrain and flow network for an area. Reads it from PREPARED TERRAIN when the
                        area is inside a prepared region (fast, light), otherwise downloads and builds it live
                        (slow, needs a lot of memory)
  run_storm()           runs runoff, river flow and flooding for one storm (fast part)
  precompute_region()   prepares the terrain for a whole region once, ahead of time (see tools/precompute_terrain.py)

Prepared terrain: everything that depends only on the land (conditioned elevation, flow directions, upstream area,
longest flow path, land cover, soils) is computed once per region and stored as tiled GeoTIFFs. A live check then
reads only the small window it needs. Set FLOOD_TERRAIN to the folder or https:// address that holds index.json.
"""
import os, json, time, heapq, urllib.request
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin, rowcol
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling, transform_bounds
from rasterio.warp import transform as warp_transform
from rasterio.windows import from_bounds as window_from_bounds, Window
import rasterio.windows
from scipy import ndimage
from numba import njit

os.environ.update(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", AWS_NO_SIGN_REQUEST="YES",
                  GDAL_HTTP_MAX_RETRY="5", GDAL_HTTP_RETRY_DELAY="3",
                  CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.vrt")

NODATA = -9999.0
KENYA_BBOX = (33.8, -4.8, 42.0, 5.1)           # rough country bounds, used to reject areas outside it
MAX_FOCUS_DEG = float(os.environ.get("FLOOD_MAX_SIDE_DEG", "0.40"))   # largest area side a user can draw (0.40 = ~44 km)
# Where prepared terrain lives: a folder or an https:// address containing index.json (empty = <project>/terrain)
TERRAIN_STORE = os.environ.get("FLOOD_TERRAIN", "").strip() or os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "terrain")
# Build terrain live (download + flow tracing) for areas outside the prepared regions? Needs 2-4 GB of memory.
LIVE_TERRAIN = os.environ.get("FLOOD_LIVE_TERRAIN", "1").strip().lower() not in ("0", "false", "no", "off")

DEFAULTS = dict(
    res_m=30, buffer_deg=0.15, prepared_buffer_deg=0.03,
    channel_threshold_km2=1.0, max_reach_length_m=2000, min_reach_for_curve_m=300,
    channel_manning_n=0.04, min_slope=1e-4, ia_ratio=0.2,
    max_stage_m=15.0, stage_step_m=0.2, min_depth_m=0.10, min_patch_cells=10,
    default_soil_group="C",
)

# ----------------------------------------------------------------------------- grid and data

def utm_epsg_for(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone

def make_model_grid(bbox, res):
    lon_c, lat_c = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    crs = CRS.from_epsg(utm_epsg_for(lon_c, lat_c))
    x0, y0, x1, y1 = transform_bounds("EPSG:4326", crs, *bbox, densify_pts=21)
    x0, y0 = np.floor(x0 / res) * res, np.floor(y0 / res) * res
    x1, y1 = np.ceil(x1 / res) * res, np.ceil(y1 / res) * res
    return dict(crs=crs, transform=from_origin(x0, y1, res, res), res=res,
                width=int(round((x1 - x0) / res)), height=int(round((y1 - y0) / res)))

def read_to_grid(hrefs, grid, resampling, dtype, dst_nodata, retries=3, log=print):
    out = np.full((grid["height"], grid["width"]), dst_nodata, dtype=dtype)
    n_read = 0
    for href in hrefs:
        data = None
        for attempt in range(retries):
            try:
                with rasterio.open(href) as src:
                    src_nodata = src.nodata if src.nodata is not None else dst_nodata
                    with WarpedVRT(src, crs=grid["crs"], transform=grid["transform"],
                                   width=grid["width"], height=grid["height"],
                                   resampling=resampling, src_nodata=src_nodata, nodata=dst_nodata) as vrt:
                        data = vrt.read(1).astype(dtype)
                break
            except rasterio.errors.RasterioIOError as err:
                if any(k in str(err) for k in ("404", "does not exist", "No such file")) or attempt == retries - 1:
                    log(f"skipped {href.split('/')[-1]}")
                    break
                time.sleep(5 * (attempt + 1))
        if data is None:
            continue
        n_read += 1
        fill = (out == dst_nodata) & (data != dst_nodata)
        out[fill] = data[fill]
    if n_read == 0:
        raise RuntimeError("None of the data tiles could be read. Check the internet connection and the area.")
    return out

def copernicus_dem_urls(bbox):
    w, s, e, n = bbox
    urls = []
    for lat in range(int(np.floor(s)), int(np.ceil(n))):
        for lon in range(int(np.floor(w)), int(np.ceil(e))):
            name = (f"Copernicus_DSM_COG_10_{'N' if lat >= 0 else 'S'}{abs(lat):02d}_00_"
                    f"{'E' if lon >= 0 else 'W'}{abs(lon):03d}_00_DEM")
            urls.append(f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif")
    return urls

def worldcover_urls(bbox):
    w, s, e, n = bbox
    urls = []
    for lat in range(int(np.floor(s / 3) * 3), int(np.ceil(n / 3) * 3), 3):
        for lon in range(int(np.floor(w / 3) * 3), int(np.ceil(e / 3) * 3), 3):
            tile = f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}{'E' if lon >= 0 else 'W'}{abs(lon):03d}"
            urls.append("https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
                        f"v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif")
    return urls

SOILGRIDS = "/vsicurl/https://files.isric.org/soilgrids/latest/data/{v}/{v}_5-15cm_mean.vrt"
GROUP_CODE = {"A": 1, "B": 2, "C": 3, "D": 4}

def soil_groups_from_texture(clay_pct, sand_pct):
    hsg = np.full(clay_pct.shape, 3, dtype="uint8")
    hsg[clay_pct < 20] = 2
    hsg[(clay_pct < 10) & (sand_pct > 90)] = 1
    hsg[(clay_pct >= 20) & (clay_pct < 40)] = 3
    hsg[clay_pct >= 40] = 4
    return hsg

# ----------------------------------------------------------------------------- numba engine

D8_DR = np.array([-1, -1, 0, 1, 1, 1, 0, -1])
D8_DC = np.array([0, 1, 1, 1, 0, -1, -1, -1])

@njit(cache=True)
def priority_flood_d8(dem, valid, res, eps):
    nr, nc = dem.shape
    n = nr * nc
    filled = dem.astype(np.float64).ravel().copy()
    vflat = valid.ravel()
    visited = np.zeros(n, np.bool_)
    heap = [(0.0, np.int64(0), np.int64(0))]
    heap.pop()
    cnt = np.int64(0)
    for r in range(nr):
        for c in range(nc):
            i = r * nc + c
            if not vflat[i]:
                continue
            boundary = r == 0 or c == 0 or r == nr - 1 or c == nc - 1
            if not boundary:
                for k in range(8):
                    if not vflat[(r + D8_DR[k]) * nc + c + D8_DC[k]]:
                        boundary = True
                        break
            if boundary:
                visited[i] = True
                heapq.heappush(heap, (filled[i], cnt, np.int64(i))); cnt += 1
    while len(heap) > 0:
        z, _, i = heapq.heappop(heap)
        r, c = i // nc, i % nc
        for k in range(8):
            rr, cc = r + D8_DR[k], c + D8_DC[k]
            if rr < 0 or cc < 0 or rr >= nr or cc >= nc:
                continue
            j = rr * nc + cc
            if not vflat[j] or visited[j]:
                continue
            visited[j] = True
            if filled[j] <= z:
                filled[j] = z + eps
            heapq.heappush(heap, (filled[j], cnt, np.int64(j))); cnt += 1
    rec = np.full(n, -1, np.int64)
    dist = np.full(n, res)
    for r in range(nr):
        for c in range(nc):
            i = r * nc + c
            if not vflat[i]:
                continue
            best, best_drop, best_d = -1, 0.0, res
            for k in range(8):
                rr, cc = r + D8_DR[k], c + D8_DC[k]
                if rr < 0 or cc < 0 or rr >= nr or cc >= nc:
                    continue
                j = rr * nc + cc
                if not vflat[j]:
                    continue
                d = res * 1.4142135623730951 if (D8_DR[k] != 0 and D8_DC[k] != 0) else res
                drop = (filled[i] - filled[j]) / d
                if drop > best_drop:
                    best, best_drop, best_d = j, drop, d
            rec[i] = best
            dist[i] = best_d
    return filled.reshape(nr, nc), rec, dist

@njit(cache=True)
def topological_order(rec, valid_flat):
    n = rec.size
    indeg = np.zeros(n, np.int32)
    for i in range(n):
        if rec[i] >= 0:
            indeg[rec[i]] += 1
    order = np.empty(n, np.int64)
    head, tail = 0, 0
    for i in range(n):
        if valid_flat[i] and indeg[i] == 0:
            order[tail] = i; tail += 1
    while head < tail:
        i = order[head]; head += 1
        r = rec[i]
        if r >= 0:
            indeg[r] -= 1
            if indeg[r] == 0:
                order[tail] = r; tail += 1
    return order[:tail]

@njit(cache=True)
def accumulate(order, rec, dist, cell_area_km2, runoff_mm, z):
    n = rec.size
    area = np.zeros(n); vol = np.zeros(n); L = np.zeros(n); zhead = z.copy()
    for i in order:
        area[i] += cell_area_km2
        vol[i] += runoff_mm[i] * cell_area_km2
    for i in order:
        r = rec[i]
        if r >= 0:
            area[r] += area[i]; vol[r] += vol[i]
            cand = L[i] + dist[i]
            if cand > L[r]:
                L[r] = cand; zhead[r] = zhead[i]
    return area, vol, L, zhead

@njit(cache=True)
def accumulate_volume(order, rec, cell_area_km2, runoff_mm, extra_vol):
    """Runoff volume flowing through each cell (mm x km2); extra_vol is water entering from outside the window."""
    n = rec.size
    vol = extra_vol.copy()
    for i in order:
        vol[i] += runoff_mm[i] * cell_area_km2
    for i in order:
        r = rec[i]
        if r >= 0:
            vol[r] += vol[i]
    return vol

@njit(cache=True)
def build_reaches(order, rec, dist, is_ch, max_len):
    n = rec.size
    n_ch_donors = np.zeros(n, np.int32); ch_donor = np.full(n, -1, np.int64)
    for i in order:
        if is_ch[i]:
            r = rec[i]
            if r >= 0 and is_ch[r]:
                n_ch_donors[r] += 1; ch_donor[r] = i
    reach = np.full(n, -1, np.int64); rlen = np.zeros(n); nreach = 0
    for i in order:
        if not is_ch[i]:
            continue
        if n_ch_donors[i] == 1:
            d = ch_donor[i]
            if rlen[d] + dist[d] < max_len:
                reach[i] = reach[d]; rlen[i] = rlen[d] + dist[d]
                continue
        reach[i] = nreach; nreach += 1
    return reach, nreach

@njit(cache=True)
def compute_hand(order, rec, is_ch, z):
    n = rec.size
    drain = np.full(n, -1, np.int64)
    for k in range(order.size - 1, -1, -1):
        i = order[k]
        if is_ch[i]:
            drain[i] = i
        elif rec[i] >= 0:
            drain[i] = drain[rec[i]]
    hand = np.full(n, np.nan)
    for i in range(n):
        if drain[i] >= 0:
            hand[i] = max(z[i] - z[drain[i]], 0.0)
    return drain, hand

# ----------------------------------------------------------------------------- runoff tables

CN_TABLE = {10: (30, 55, 70, 77), 20: (35, 56, 70, 77), 30: (39, 61, 74, 80), 40: (67, 78, 85, 89),
            50: (89, 92, 94, 95), 60: (77, 86, 91, 94), 70: (98, 98, 98, 98), 80: (100, 100, 100, 100),
            90: (90, 90, 90, 90), 95: (90, 90, 90, 90), 100: (49, 69, 79, 84)}

def curve_numbers(landcover, soil_group, wetness):
    lut = np.zeros((256, 5)); lut[:, 1:] = CN_TABLE[30]
    for cls, row in CN_TABLE.items():
        lut[cls, 1:] = row
    cn = lut[landcover, np.clip(soil_group, 1, 4)]
    if wetness == "dry":
        cn = cn / (2.281 - 0.01281 * cn)
    elif wetness == "wet":
        cn = cn / (0.427 + 0.00573 * cn)
    return np.clip(cn, 1, 100)

def scs_runoff(rain_mm, cn, ia_ratio):
    S = 25400.0 / cn - 254.0
    Ia = ia_ratio * S
    return np.where(rain_mm > Ia, (rain_mm - Ia) ** 2 / np.maximum(rain_mm - Ia + S, 1e-9), 0.0)

# ----------------------------------------------------------------------------- pipeline

def validate_focus(focus_bbox):
    w, s, e, n = focus_bbox
    if not (w < e and s < n):
        raise ValueError("The area is empty. Draw a rectangle on the map.")
    if e - w > MAX_FOCUS_DEG or n - s > MAX_FOCUS_DEG:
        raise ValueError(f"The area is too large. Keep each side under {MAX_FOCUS_DEG}° (about 44 km).")
    kw, ks, ke, kn = KENYA_BBOX
    if w < kw or e > ke or s < ks or n > kn:
        raise ValueError("The area is outside the area the model covers. Draw it inside the country.")

def download_inputs(domain, p=DEFAULTS, log=print):
    """Elevation, land cover and soil groups for a lon/lat box, on a UTM grid."""
    grid = make_model_grid(domain, p["res_m"])
    shape = (grid["height"], grid["width"])

    log("Downloading ground heights")
    dem = read_to_grid(copernicus_dem_urls(domain), grid, Resampling.bilinear, "float32", NODATA, log=log)
    valid = dem != NODATA
    log("Downloading land cover")
    landcover = read_to_grid(worldcover_urls(domain), grid, Resampling.mode, "uint8", 0, log=log)
    log("Downloading soil data")
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="3"):
            clay = read_to_grid([SOILGRIDS.format(v="clay")], grid, Resampling.nearest, "int16", -32768, log=log)
            sand = read_to_grid([SOILGRIDS.format(v="sand")], grid, Resampling.nearest, "int16", -32768, log=log)
        missing = (clay == -32768) | (sand == -32768)
        soil_group = soil_groups_from_texture(clay / 10.0, sand / 10.0)
        soil_group[missing] = GROUP_CODE[p["default_soil_group"]]
        soil_note = None
    except Exception as err:
        soil_group = np.full(shape, GROUP_CODE[p["default_soil_group"]], dtype="uint8")
        soil_note = f"Soil data unavailable; soil group {p['default_soil_group']} used everywhere."
        log(soil_note)
    return grid, dem, valid, landcover, soil_group, soil_note

def load_terrain(focus_bbox, p=DEFAULTS, log=print):
    """Terrain and flow network for the focus area: from prepared terrain when possible, otherwise built live."""
    region = find_region(focus_bbox, p)
    if region is not None:
        return load_prepared(focus_bbox, region, p, log)
    if not LIVE_TERRAIN:
        raise ValueError("This area isn't prepared yet. Draw your area inside the prepared regions outlined on the map.")
    b = p["buffer_deg"]
    w, s, e, n = focus_bbox
    domain = (w - b, s - b, e + b, n + b)
    grid, dem, valid, landcover, soil_group, soil_note = download_inputs(domain, p, log)
    shape = (grid["height"], grid["width"])

    log("Tracing how water flows across the land")
    res = float(p["res_m"])
    dem_cond, rec, dist = priority_flood_d8(dem, valid, res, 1e-6)
    z_flat = dem_cond.ravel().copy()
    valid_flat = valid.ravel()
    order = topological_order(rec, valid_flat)
    return dict(focus_bbox=tuple(focus_bbox), domain=domain, grid=grid, shape=shape, dem=dem, valid=valid,
                landcover=landcover, soil_group=soil_group, soil_note=soil_note,
                dem_cond=dem_cond, rec=rec, dist=dist, z_flat=z_flat, valid_flat=valid_flat, order=order,
                source="live")

# ----------------------------------------------------------------------------- prepared terrain

LAYERS = dict(dem=("float32", NODATA), demcond=("float32", NODATA), fdir=("uint8", 255), facc=("float32", 0.0),
              flowlen=("float32", 0.0), zhead=("float32", NODATA), landcover=("uint8", 0), soil=("uint8", 0))
_INDEX = {"store": None, "data": None}

def _is_url(path):
    return path.startswith("http://") or path.startswith("https://")

def _join(base, *parts):
    return "/".join([base.rstrip("/")] + [p.strip("/") for p in parts]) if _is_url(base) else os.path.join(base, *parts)

def _gdal_path(path):
    return "/vsicurl/" + path if _is_url(path) else path

def terrain_index(reload=False):
    """index.json of the prepared terrain store, or None when there is none."""
    if _INDEX["store"] != TERRAIN_STORE or reload:
        _INDEX.update(store=TERRAIN_STORE, data=None)
        path = _join(TERRAIN_STORE, "index.json")
        try:
            if _is_url(path):
                with urllib.request.urlopen(path, timeout=20) as r:
                    _INDEX["data"] = json.load(r)
            elif os.path.exists(path):
                with open(path) as f:
                    _INDEX["data"] = json.load(f)
        except Exception as err:
            print(f"Prepared terrain index could not be read ({path}): {err}")
    return _INDEX["data"]

def prepared_regions():
    idx = terrain_index()
    return [dict(name=r["name"], bbox=r["bbox"]) for r in (idx or {}).get("regions", [])]

def find_region(focus_bbox, p=DEFAULTS):
    """The prepared region that holds the focus area plus its buffer with the widest margin, or None."""
    b = p["prepared_buffer_deg"]
    w, s, e, n = focus_bbox[0] - b, focus_bbox[1] - b, focus_bbox[2] + b, focus_bbox[3] + b
    best, best_margin = None, -1.0
    for r in (terrain_index() or {}).get("regions", []):
        rw, rs, re_, rn = r["bbox"]
        margin = min(w - rw, s - rs, re_ - e, rn - n)
        if margin >= 0 and margin > best_margin:
            best, best_margin = r, margin
    return best

def _codes_from_receivers(rec, nc):
    """D8 direction code (0-7, 255 = none) for each cell from its receiver index."""
    n = rec.size
    code = np.full(n, 255, dtype="uint8")
    idx = np.nonzero(rec >= 0)[0]
    dr = rec[idx] // nc - idx // nc
    dc = rec[idx] % nc - idx % nc
    lut = np.full((3, 3), 255, dtype="uint8")
    for k in range(8):
        lut[D8_DR[k] + 1, D8_DC[k] + 1] = k
    code[idx] = lut[dr + 1, dc + 1]
    return code

def precompute_region(name, region_bbox, out_dir, p=DEFAULTS, log=print, inputs=None):
    """Prepare the terrain for one region and register it in out_dir/index.json.

    inputs: optional (grid, dem, valid, landcover, soil_group, soil_note) instead of downloading (used by tests)."""
    os.makedirs(os.path.join(out_dir, name), exist_ok=True)
    grid, dem, valid, landcover, soil_group, soil_note = inputs or download_inputs(tuple(region_bbox), p, log)
    nr, nc = dem.shape
    log(f"Tracing how water flows across {nr} x {nc} cells")
    res = float(grid["res"])
    dem_cond, rec, dist = priority_flood_d8(dem, valid, res, 1e-6)
    order = topological_order(rec, valid.ravel())
    log("Measuring upstream areas and flow paths")
    area, _, flow_len, zhead = accumulate(order, rec, dist, res * res / 1e6, np.zeros(rec.size), dem_cond.ravel())
    layers = dict(dem=dem, demcond=dem_cond, fdir=_codes_from_receivers(rec, nc).reshape(nr, nc),
                  facc=area.reshape(nr, nc), flowlen=flow_len.reshape(nr, nc), zhead=zhead.reshape(nr, nc),
                  landcover=landcover, soil=soil_group)
    del rec, dist, order, area, flow_len, zhead
    for key, (dtype, nodata) in LAYERS.items():
        log(f"Writing {key}.tif")
        arr = layers[key].astype(dtype)
        if dtype == "float32":
            arr[~valid] = nodata
        with rasterio.open(os.path.join(out_dir, name, f"{key}.tif"), "w", driver="GTiff", width=nc, height=nr,
                           count=1, dtype=dtype, crs=grid["crs"], transform=grid["transform"], nodata=nodata,
                           tiled=True, blockxsize=512, blockysize=512, compress="deflate",
                           predictor=3 if dtype == "float32" else 2, BIGTIFF="IF_SAFER") as dst:
            dst.write(arr, 1)
    meta = dict(name=name, bbox=[round(float(v), 6) for v in region_bbox], res_m=res, width=nc, height=nr,
                crs=grid["crs"].to_string(), soil_note=soil_note, created=time.strftime("%Y-%m-%d"),
                channel_threshold_km2=p["channel_threshold_km2"])
    with open(os.path.join(out_dir, name, "meta.json"), "w") as f:
        json.dump(meta, f, indent=1)
    idx_path = os.path.join(out_dir, "index.json")
    idx = json.load(open(idx_path)) if os.path.exists(idx_path) else dict(version=1, regions=[])
    idx["regions"] = [r for r in idx["regions"] if r["name"] != name] + [
        dict(name=name, bbox=meta["bbox"], path=name, res_m=res)]
    with open(idx_path, "w") as f:
        json.dump(idx, f, indent=1)
    log(f"Region '{name}' ready")
    return meta

def load_prepared(focus_bbox, region, p=DEFAULTS, log=print):
    """Read the window of a prepared region that covers the focus area plus a small buffer."""
    log("Reading prepared terrain for this area")
    b = p["prepared_buffer_deg"]
    w, s, e, n = focus_bbox
    domain = (w - b, s - b, e + b, n + b)
    base = _join(TERRAIN_STORE, region.get("path", region["name"]))
    src_path = lambda key: _gdal_path(_join(base, f"{key}.tif"))
    with rasterio.open(src_path("demcond")) as src:
        crs, tr, H, W = src.crs, src.transform, src.height, src.width
        res = float(abs(tr.a))
    x0, y0, x1, y1 = transform_bounds("EPSG:4326", crs, *domain, densify_pts=21)
    win = window_from_bounds(x0, y0, x1, y1, transform=tr).round_offsets().round_lengths()
    c0, r0 = max(int(win.col_off), 0), max(int(win.row_off), 0)
    c1, r1 = min(int(win.col_off + win.width), W), min(int(win.row_off + win.height), H)
    win = Window(c0, r0, c1 - c0, r1 - r0)
    pad = Window(c0 - 1, r0 - 1, c1 - c0 + 2, r1 - r0 + 2)

    def read(key, window):
        dtype, nodata = LAYERS[key]
        with rasterio.open(src_path(key)) as src:
            return src.read(1, window=window, boundless=True, fill_value=nodata).astype(dtype)

    dem = read("dem", win)
    dem_cond = read("demcond", win).astype("float64")
    fdir_p = read("fdir", pad)
    facc_p = read("facc", pad).astype("float64")
    flow_len = read("flowlen", win).astype("float64")
    zhead = read("zhead", win).astype("float64")
    landcover = read("landcover", win)
    soil_group = read("soil", win)
    nr, nc = dem.shape
    valid = dem != NODATA
    soil_group[soil_group == 0] = GROUP_CODE[p["default_soil_group"]]

    # receivers inside the window (flow leaving the window ends there)
    code = fdir_p[1:-1, 1:-1].ravel()
    rows, cols = np.divmod(np.arange(nr * nc), nc)
    has = (code < 8) & valid.ravel()
    k = np.where(has, code, 0)
    rr, cc = rows + D8_DR[k], cols + D8_DC[k]
    inside = has & (rr >= 0) & (rr < nr) & (cc >= 0) & (cc < nc)
    rec = np.where(inside, rr * nc + cc, -1).astype(np.int64)
    rec[inside & ~valid.ravel()[np.clip(rr * nc + cc, 0, nr * nc - 1)]] = -1
    diag = (D8_DR[k] != 0) & (D8_DC[k] != 0)
    dist = np.where(diag, res * 1.4142135623730951, res)

    # upstream area entering from the ring of cells just outside the window
    inflow = np.zeros(nr * nc)
    ring = np.zeros(fdir_p.shape, bool); ring[0, :] = ring[-1, :] = ring[:, 0] = ring[:, -1] = True
    pr, pc = np.nonzero(ring & (fdir_p < 8))
    kk = fdir_p[pr, pc]
    tr_, tc_ = pr + D8_DR[kk] - 1, pc + D8_DC[kk] - 1
    ok = (tr_ >= 0) & (tr_ < nr) & (tc_ >= 0) & (tc_ < nc)
    np.add.at(inflow, tr_[ok] * nc + tc_[ok], facc_p[pr[ok], pc[ok]])

    valid_flat = valid.ravel()
    order = topological_order(rec, valid_flat)
    grid = dict(crs=crs, transform=rasterio.windows.transform(win, tr), res=res, width=nc, height=nr)
    facc = facc_p[1:-1, 1:-1].ravel()
    return dict(focus_bbox=tuple(focus_bbox), domain=domain, grid=grid, shape=(nr, nc), dem=dem, valid=valid,
                landcover=landcover, soil_group=soil_group, soil_note=None,
                dem_cond=dem_cond, rec=rec, dist=dist, z_flat=dem_cond.ravel().copy(), valid_flat=valid_flat,
                order=order, facc_flat=facc, flowlen_flat=flow_len.ravel(), zhead_flat=zhead.ravel(),
                inflow_area_flat=inflow, source=f"prepared:{region['name']}")

def snap_to_channel(T, area_km2, lat, lon, radius_m=1000):
    grid = T["grid"]
    xs, ys = warp_transform("EPSG:4326", grid["crs"], [lon], [lat])
    r0, c0 = rowcol(grid["transform"], xs[0], ys[0])
    nr, nc = T["shape"]
    if not (0 <= r0 < nr and 0 <= c0 < nc):
        raise ValueError("The river inflow point is outside the modelled area. Place it inside the rectangle.")
    k = int(radius_m // grid["res"])
    r_lo, c_lo = max(r0 - k, 0), max(c0 - k, 0)
    win = area_km2.reshape(T["shape"])[r_lo:r0 + k + 1, c_lo:c0 + k + 1]
    dr, dc = np.unravel_index(np.argmax(win), win.shape)
    return (r_lo + dr) * nc + (c_lo + dc)

def run_storm(T, rain_mm, rain_hours, wetness="normal", inflows=(), p=DEFAULTS, log=print):
    """Runoff, river flow and flooding for one storm. inflows: list of dicts {lat, lon, flood_m3s, normal_m3s}."""
    grid, shape = T["grid"], T["shape"]
    res = float(grid.get("res", p["res_m"])); cell_m2 = res * res; cell_km2 = cell_m2 / 1e6
    rec, dist, order, z_flat, valid, valid_flat = (T[k] for k in ("rec", "dist", "order", "z_flat", "valid", "valid_flat"))
    nr, nc = shape

    log("Working out how much rain runs off")
    cn = curve_numbers(T["landcover"], T["soil_group"], wetness)
    runoff_mm = np.where(valid, scs_runoff(rain_mm, cn, p["ia_ratio"]), 0.0)

    log("Working out river flows")
    if "facc_flat" in T:
        # prepared terrain: upstream area and flow paths are known for the whole region; water from upstream
        # land outside the window is assumed to run off like the average land inside it
        area_km2, flow_len, zhead = T["facc_flat"], T["flowlen_flat"], T["zhead_flat"]
        mean_mm = float(runoff_mm[valid].mean()) if valid.any() else 0.0
        vol = accumulate_volume(order, rec, cell_km2, runoff_mm.ravel(), T["inflow_area_flat"] * mean_mm)
    else:
        area_km2, vol, flow_len, zhead = accumulate(order, rec, dist, cell_km2, runoff_mm.ravel(), z_flat)
    mean_runoff = np.where(area_km2 > 0, vol / np.maximum(area_km2, 1e-12), 0.0)
    L = np.maximum(flow_len, res)
    slope = np.maximum((zhead - z_flat) / L, 0.001)
    tc_h = 0.0195 * L ** 0.77 * slope ** -0.385 / 60.0
    q_peak = 0.208 * area_km2 * mean_runoff / (rain_hours / 2.0 + 0.6 * tc_h)
    q_peak[~valid_flat] = 0.0
    is_ch = (area_km2 >= p["channel_threshold_km2"]) & valid_flat

    inflow_info = []
    for inf in inflows or []:
        cell = snap_to_channel(T, area_km2, inf["lat"], inf["lon"])
        extra = max(float(inf["flood_m3s"]) - float(inf.get("normal_m3s", 0) or 0), 0.0)
        i, n_steps = cell, 0
        while i >= 0 and n_steps < rec.size:
            q_peak[i] += extra; is_ch[i] = True; i = rec[i]; n_steps += 1
        inflow_info.append(dict(lat=inf["lat"], lon=inf["lon"], added_m3s=extra, river_km=n_steps * res / 1000))

    log("Measuring height above the rivers")
    reach, n_reach = build_reaches(order, rec, dist, is_ch, float(p["max_reach_length_m"]))
    drain, hand = compute_hand(order, rec, is_ch, z_flat)

    log("Working out how high each river rises")
    ch_idx = np.nonzero(is_ch & (reach >= 0))[0]
    rid_ch = reach[ch_idx]
    reach_len = np.maximum(np.bincount(rid_ch, weights=dist[ch_idx], minlength=n_reach), res)
    z_hi = np.full(n_reach, -np.inf); z_lo = np.full(n_reach, np.inf)
    np.maximum.at(z_hi, rid_ch, z_flat[ch_idx]); np.minimum.at(z_lo, rid_ch, z_flat[ch_idx])
    reach_slope = np.maximum((z_hi - z_lo) / reach_len, p["min_slope"])
    q_design = np.zeros(n_reach); np.maximum.at(q_design, rid_ch, q_peak[ch_idx])

    cand = np.nonzero((drain >= 0) & (hand < p["max_stage_m"]))[0]
    cand_rid = reach[drain[cand]]; cand_hand = hand[cand]
    gy, gx = np.gradient(T["dem_cond"], res)
    bed_factor = cell_m2 * np.sqrt(1.0 + (gx ** 2 + gy ** 2).ravel()[cand])
    stages = np.arange(p["stage_step_m"], p["max_stage_m"] + 1e-9, p["stage_step_m"])
    Q_curve = np.zeros((stages.size, n_reach))
    for k, h in enumerate(stages):
        m = cand_hand < h
        V = np.bincount(cand_rid[m], weights=(h - cand_hand[m]) * cell_m2, minlength=n_reach)
        Ab = np.bincount(cand_rid[m], weights=bed_factor[m], minlength=n_reach)
        A = V / reach_len; P = Ab / reach_len
        R = np.where(P > 0, A / np.maximum(P, 1e-9), 0.0)
        Q_curve[k] = A * R ** (2 / 3) * np.sqrt(reach_slope) / p["channel_manning_n"]
    Q_curve = np.maximum.accumulate(Q_curve, axis=0)
    stage = np.zeros(n_reach)
    xq = np.concatenate([[0.0], stages])
    for j in np.nonzero(q_design > 0)[0]:
        stage[j] = np.interp(q_design[j], np.concatenate([[0.0], Q_curve[:, j]]), xq)
    ch_rec = rec[ch_idx]
    leaves = (ch_rec >= 0) & (reach[np.maximum(ch_rec, 0)] != rid_ch)
    reach_down = np.full(n_reach, -1, np.int64); reach_down[rid_ch[leaves]] = reach[ch_rec[leaves]]
    short = reach_len < p["min_reach_for_curve_m"]
    for j in range(n_reach - 1, -1, -1):
        if short[j] and reach_down[j] >= 0:
            stage[j] = stage[reach_down[j]]
    capped = int((q_design > Q_curve[-1]).sum())

    log("Mapping flood depth")
    depth = np.zeros(rec.size)
    has = drain >= 0
    depth[has] = stage[reach[drain[has]]] - hand[has]
    depth = np.clip(depth, 0, None).reshape(shape); depth[~valid] = 0
    depth[depth < p["min_depth_m"]] = 0
    labels, n_lab = ndimage.label(depth > 0)
    if n_lab:
        sizes = ndimage.sum(np.ones_like(depth), labels, index=np.arange(1, n_lab + 1))
        depth[np.isin(labels, np.nonzero(sizes < p["min_patch_cells"])[0] + 1)] = 0
    permanent_water = T["landcover"] == 80
    flood = np.where(permanent_water, 0, depth).astype("float32")

    # Clip to the area the user drew
    fx0, fy0, fx1, fy1 = transform_bounds("EPSG:4326", grid["crs"], *T["focus_bbox"])
    win = window_from_bounds(fx0, fy0, fx1, fy1, transform=grid["transform"]).round_offsets().round_lengths()
    r0, c0 = max(win.row_off, 0), max(win.col_off, 0)
    r1, c1 = min(r0 + win.height, nr), min(c0 + win.width, nc)
    fwin = Window(c0, r0, c1 - c0, r1 - r0)
    f_depth = flood[r0:r1, c0:c1]
    f_water = permanent_water[r0:r1, c0:c1]
    flooded = f_depth > 0
    summary = dict(
        rain_mm=rain_mm, rain_hours=rain_hours, wetness=wetness,
        flooded_area_km2=round(float(flooded.sum() * cell_km2), 2),
        flooded_share_pct=round(float(100 * flooded.mean()), 2),
        max_depth_m=round(float(f_depth.max()), 2),
        mean_depth_m=round(float(f_depth[flooded].mean()), 2) if flooded.any() else 0.0,
        area_over_1m_km2=round(float((f_depth > 1).sum() * cell_km2), 2),
        mean_runoff_pct=round(float(100 * runoff_mm[valid].mean() / max(rain_mm, 1e-9)), 1),
        inflows=inflow_info, capped_segments=capped, soil_note=T["soil_note"], terrain=T.get("source", "live"),
    )
    return dict(depth=f_depth, permanent_water=f_water, ground=T["dem"][r0:r1, c0:c1], crs=grid["crs"],
                transform=rasterio.windows.transform(fwin, grid["transform"]), summary=summary)

# ----------------------------------------------------------------------------- outputs for the web map

DEPTH_STOPS = [(0.1, (189, 228, 244)), (0.5, (107, 174, 214)), (1.0, (49, 130, 189)),
               (2.0, (8, 81, 156)), (3.0, (8, 48, 107))]

def depth_rgba(depth, permanent_water):
    """Colour flood depth for display. Permanent water gets a flat muted colour."""
    d = depth.astype("float64")
    xs = [s[0] for s in DEPTH_STOPS]
    rgba = np.zeros(d.shape + (4,), dtype="uint8")
    for ch in range(3):
        rgba[..., ch] = np.interp(d, xs, [s[1][ch] for s in DEPTH_STOPS]).astype("uint8")
    rgba[..., 3] = np.where(d > 0, 215, 0)
    rgba[permanent_water] = (90, 110, 130, 150)
    return rgba

def image_corners(result):
    """Lon/lat of the result image's four corners (top-left, top-right, bottom-right, bottom-left)."""
    t = result["transform"]; h, w = result["depth"].shape
    xs = [t.c, t.c + t.a * w, t.c + t.a * w, t.c]
    ys = [t.f, t.f, t.f + t.e * h, t.f + t.e * h]
    lons, lats = warp_transform(result["crs"], "EPSG:4326", xs, ys)
    return [[float(lo), float(la)] for lo, la in zip(lons, lats)]

def depth_at(result, lat, lon):
    xs, ys = warp_transform("EPSG:4326", result["crs"], [lon], [lat])
    r, c = rowcol(result["transform"], xs[0], ys[0])
    h, w = result["depth"].shape
    if 0 <= r < h and 0 <= c < w:
        return float(result["depth"][r, c]), bool(result["permanent_water"][r, c])
    return None, False

def save_depth_geotiff(result, path):
    d = result["depth"]
    with rasterio.open(path, "w", driver="GTiff", height=d.shape[0], width=d.shape[1], count=1,
                       dtype="float32", crs=result["crs"], transform=result["transform"],
                       nodata=0, compress="deflate") as dst:
        dst.write(d, 1)


def profile(result, coords, step_m=30.0, max_points=4000):
    """Ground and water level along a line drawn on the map. coords: [[lon, lat], ...]."""
    lons = [c[0] for c in coords]; lats = [c[1] for c in coords]
    xs, ys = warp_transform("EPSG:4326", result["crs"], lons, lats)
    px, py, dist = [], [], []
    total = 0.0
    for i in range(len(xs) - 1):
        seg = float(np.hypot(xs[i + 1] - xs[i], ys[i + 1] - ys[i]))
        n = max(int(seg / step_m), 1)
        for k in range(n):
            t = k / n
            px.append(xs[i] + t * (xs[i + 1] - xs[i])); py.append(ys[i] + t * (ys[i + 1] - ys[i]))
            dist.append(total + t * seg)
        total += seg
    px.append(xs[-1]); py.append(ys[-1]); dist.append(total)
    if len(px) > max_points:
        keep = np.linspace(0, len(px) - 1, max_points).astype(int)
        px = [px[i] for i in keep]; py = [py[i] for i in keep]; dist = [dist[i] for i in keep]
    rows, cols = rowcol(result["transform"], px, py)
    h, w = result["depth"].shape
    ground, water, wet = [], [], []
    for r, c in zip(rows, cols):
        if 0 <= r < h and 0 <= c < w and result["ground"][r, c] != NODATA:
            g = float(result["ground"][r, c]); d = float(result["depth"][r, c])
            ground.append(round(g, 2)); water.append(round(g + d, 2) if d > 0 else None)
            wet.append(bool(result["permanent_water"][r, c]))
        else:
            ground.append(None); water.append(None); wet.append(False)
    return dict(distance_m=[round(d, 1) for d in dist], ground_m=ground, water_m=water, permanent_water=wet,
                length_m=round(total, 1))


## 2. What's in the water

Downloads building footprints and roads from Overture Maps (which combines OpenStreetMap with Microsoft and Google building data), and schools and health facilities from OpenStreetMap. Then checks which of them sit in flood water, and how deep. If any source can't be reached, the run carries on without it.

In [ ]:
%%writefile /content/floodcheck/backend/exposure.py
"""Exposure: which buildings, roads and facilities sit in flood water.

Data sources (no account needed):
  buildings  Overture Maps (includes OpenStreetMap, Microsoft and Google open building footprints)
  roads      Overture Maps transportation segments
  schools and health facilities  OpenStreetMap via the Overpass API, falling back to Overture places
"""
import json, time, urllib.parse, urllib.request
import numpy as np
import shapely
from rasterio.transform import rowcol
from rasterio.warp import transform as warp_transform

ROAD_CLASSES = {"motorway", "trunk", "primary", "secondary", "tertiary", "unclassified",
                "residential", "living_street", "service", "track"}
MAIN_ROADS = {"motorway", "trunk", "primary", "secondary", "tertiary"}
MAX_BUILDINGS_RETURNED = 25000
DEPTH_BANDS = [0.1, 0.5, 1.0, 2.0]

# ----------------------------------------------------------------------------- download

def _overture_table(kind, bbox):
    from overturemaps import core
    last_err = None
    for use_stac in (True, False):
        try:
            reader = core.record_batch_reader(kind, bbox=tuple(bbox), stac=use_stac)
            if reader is not None:
                return reader.read_all()
        except Exception as err:          # try the other access method
            last_err = err
    raise RuntimeError(f"Overture {kind} download failed: {last_err}")

def _geometries(table):
    col = table.column("geometry").combine_chunks()
    if hasattr(col, "storage"):
        col = col.storage
    return shapely.from_wkb(np.asarray(col.to_pylist(), dtype=object))

def _names(table):
    if "names" not in table.column_names:
        return [None] * table.num_rows
    return [(n or {}).get("primary") if isinstance(n, dict) else None for n in table.column("names").to_pylist()]

def fetch_buildings(bbox, log=print):
    log("Downloading building footprints")
    t = _overture_table("building", bbox)
    geoms = _geometries(t)
    keep = ~shapely.is_empty(geoms)
    return dict(geoms=geoms[keep])

def fetch_roads(bbox, log=print):
    log("Downloading roads")
    t = _overture_table("segment", bbox)
    subtype = t.column("subtype").to_pylist() if "subtype" in t.column_names else ["road"] * t.num_rows
    klass = t.column("class").to_pylist() if "class" in t.column_names else ["unclassified"] * t.num_rows
    names = _names(t)
    geoms = _geometries(t)
    keep = [i for i in range(t.num_rows) if subtype[i] == "road" and (klass[i] or "unclassified") in ROAD_CLASSES]
    return dict(geoms=geoms[keep], cls=[klass[i] or "unclassified" for i in keep], names=[names[i] for i in keep])

def _facility_kind(tags_or_category):
    s = " ".join(str(v) for v in tags_or_category).lower()
    if any(k in s for k in ("hospital", "clinic", "health", "medical", "doctor", "dispensary")):
        return "health"
    if any(k in s for k in ("school", "college", "university", "kindergarten")):
        return "school"
    return None

def fetch_facilities_osm(bbox, timeout=90):
    w, s, e, n = bbox
    q = f"""[out:json][timeout:{timeout}];
(
  nwr["amenity"~"^(school|college|university|kindergarten|hospital|clinic|doctors)$"]({s},{w},{n},{e});
  nwr["healthcare"]({s},{w},{n},{e});
);
out center tags;"""
    req = urllib.request.Request("https://overpass-api.de/api/interpreter",
                                 data=urllib.parse.urlencode({"data": q}).encode(),
                                 headers={"User-Agent": "atlas-floodcheck/0.1"})
    with urllib.request.urlopen(req, timeout=timeout + 30) as resp:
        data = json.load(resp)
    out = []
    for el in data.get("elements", []):
        lat = el.get("lat", el.get("center", {}).get("lat"))
        lon = el.get("lon", el.get("center", {}).get("lon"))
        tags = el.get("tags", {})
        kind = _facility_kind([tags.get("amenity", ""), tags.get("healthcare", "")])
        if lat is None or kind is None:
            continue
        out.append(dict(lat=lat, lon=lon, kind=kind, name=tags.get("name") or ("Health facility" if kind == "health" else "School"),
                        source="OpenStreetMap"))
    return out

def fetch_facilities_overture(bbox):
    t = _overture_table("place", bbox)
    cats = t.column("categories").to_pylist() if "categories" in t.column_names else [None] * t.num_rows
    names = _names(t)
    geoms = _geometries(t)
    out = []
    for i in range(t.num_rows):
        c = cats[i] or {}
        kind = _facility_kind([c.get("primary") or ""] + list(c.get("alternate") or []))
        if kind is None:
            continue
        p = geoms[i]
        out.append(dict(lat=shapely.get_y(p), lon=shapely.get_x(p), kind=kind,
                        name=names[i] or ("Health facility" if kind == "health" else "School"), source="Overture"))
    return out

def fetch_facilities(bbox, log=print):
    log("Downloading schools and health facilities")
    try:
        return fetch_facilities_osm(bbox)
    except Exception as err:
        log(f"OpenStreetMap facilities unavailable ({str(err)[:60]}); trying Overture")
        return fetch_facilities_overture(bbox)

def load_exposure(bbox, log=print):
    """Download everything for an area. Each part fails independently, so a missing source doesn't stop the run."""
    data, notes = {}, []
    for key, fn in (("buildings", fetch_buildings), ("roads", fetch_roads), ("facilities", fetch_facilities)):
        try:
            data[key] = fn(bbox, log=log)
        except Exception as err:
            data[key] = None
            notes.append(f"{key.capitalize()} couldn't be downloaded, so they're left out of this run.")
            log(f"{key} unavailable: {str(err)[:100]}")
    data["notes"] = notes
    return data

# ----------------------------------------------------------------------------- analysis

def sample_depth(result, lons, lats):
    lons = np.asarray(lons, dtype="float64"); lats = np.asarray(lats, dtype="float64")
    if lons.size == 0:
        return np.zeros(0)
    xs, ys = warp_transform("EPSG:4326", result["crs"], lons.tolist(), lats.tolist())
    rows, cols = rowcol(result["transform"], xs, ys)
    rows, cols = np.asarray(rows), np.asarray(cols)
    h, w = result["depth"].shape
    inside = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    out = np.zeros(lons.size)
    out[inside] = result["depth"][rows[inside], cols[inside]]
    return out

def _feature(geom, props, ndigits=6):
    g = json.loads(shapely.to_geojson(shapely.set_precision(geom, 10 ** -ndigits)))
    return {"type": "Feature", "properties": props, "geometry": g}

def analyse(result, exposure, bbox):
    """Intersect exposure data with a flood result. Returns summary counts and GeoJSON layers."""
    area = shapely.box(*bbox)
    summary, layers = {}, {}

    b = exposure.get("buildings")
    if b is not None:
        geoms = b["geoms"]
        cent = shapely.centroid(geoms)
        inside = shapely.contains(area, cent)
        geoms, cent = geoms[inside], cent[inside]
        depth = sample_depth(result, shapely.get_x(cent), shapely.get_y(cent))
        flooded = depth > 0
        summary["buildings_total"] = int(geoms.size)
        summary["buildings_flooded"] = {f"over_{d}m": int((depth > d).sum()) for d in DEPTH_BANDS}
        summary["buildings_flooded"]["any"] = int(flooded.sum())
        idx = np.nonzero(flooded)[0]
        idx = idx[np.argsort(-depth[idx])][:MAX_BUILDINGS_RETURNED]
        # polygons for close-up views, plus centre points so buildings stay visible when zoomed out
        feats = [_feature(geoms[i], {"depth": round(float(depth[i]), 2)}) for i in idx]
        cx, cy = shapely.get_x(cent), shapely.get_y(cent)
        feats += [{"type": "Feature", "properties": {"depth": round(float(depth[i]), 2)},
                   "geometry": {"type": "Point", "coordinates": [round(float(cx[i]), 6), round(float(cy[i]), 6)]}} for i in idx]
        layers["buildings"] = {"type": "FeatureCollection", "features": feats}
        summary["buildings_truncated"] = bool(flooded.sum() > MAX_BUILDINGS_RETURNED)

    r = exposure.get("roads")
    if r is not None and len(r["geoms"]):
        geoms = shapely.intersection(r["geoms"], area)
        feats, flooded_km, main_km, total_km = [], 0.0, 0.0, 0.0
        for g, cls, name in zip(geoms, r["cls"], r["names"]):
            if shapely.is_empty(g):
                continue
            # sample every ~30 m (0.00027°) along the road
            length_deg = shapely.length(g)
            n = max(int(length_deg / 0.00027), 2)
            pts = shapely.line_interpolate_point(g, np.linspace(0, 1, n), normalized=True)
            d = sample_depth(result, shapely.get_x(pts), shapely.get_y(pts))
            km = length_deg * 111.0          # close enough near the equator for a summary figure
            total_km += km
            wet = d > 0.3                    # 30 cm of water makes most roads impassable to cars
            frac = float(wet.mean())
            if frac > 0:
                flooded_km += km * frac
                if cls in MAIN_ROADS:
                    main_km += km * frac
                # return only the flooded stretches, as runs of consecutive wet sample points
                xs, ys = shapely.get_x(pts), shapely.get_y(pts)
                edges = np.diff(np.concatenate([[0], wet.astype(int), [0]]))
                for a, b_ in zip(np.nonzero(edges == 1)[0], np.nonzero(edges == -1)[0]):
                    a0, b0 = max(a - 1, 0), min(b_, len(xs) - 1)      # extend half a step each side
                    if b0 - a0 < 1:
                        continue
                    seg = shapely.LineString(np.column_stack([xs[a0:b0 + 1], ys[a0:b0 + 1]]))
                    feats.append(_feature(seg, {"class": cls, "name": name or "",
                                                "max_depth": round(float(d[a:b_].max()), 2)}))
        summary["roads_total_km"] = round(total_km, 1)
        summary["roads_flooded_km"] = round(flooded_km, 1)
        summary["main_roads_flooded_km"] = round(main_km, 1)
        layers["roads"] = {"type": "FeatureCollection", "features": feats}

    f = exposure.get("facilities")
    if f is not None:
        f = [x for x in f if bbox[0] <= x["lon"] <= bbox[2] and bbox[1] <= x["lat"] <= bbox[3]]
        depth = sample_depth(result, [x["lon"] for x in f], [x["lat"] for x in f])
        feats = []
        for x, dd in zip(f, depth):
            feats.append({"type": "Feature", "geometry": {"type": "Point", "coordinates": [x["lon"], x["lat"]]},
                          "properties": {"name": x["name"], "kind": x["kind"], "depth": round(float(dd), 2),
                                         "flooded": bool(dd > 0), "source": x["source"]}})
        summary["facilities_total"] = {"school": sum(x["kind"] == "school" for x in f),
                                       "health": sum(x["kind"] == "health" for x in f)}
        flooded = [ft["properties"] | {"lon": ft["geometry"]["coordinates"][0], "lat": ft["geometry"]["coordinates"][1]}
                   for ft in feats if ft["properties"]["flooded"]]
        summary["facilities_flooded"] = sorted(flooded, key=lambda p: -p["depth"])
        layers["facilities"] = {"type": "FeatureCollection", "features": feats}

    summary["notes"] = exposure.get("notes", [])
    return summary, layers


## 3. Real weather and river flow for any date

Rainfall comes from Open-Meteo: ERA5 reanalysis for past dates (back to 1940), and the weather forecast for recent days and the next 16 days. It's averaged over your area. Ground wetness is worked out from the rain in the 5 days before the storm, using the standard SCS rule. River flow comes from GloFAS, Europe's global flood forecasting system, back to 1984, including forecasts. None of these need an account.

In [ ]:
%%writefile /content/floodcheck/backend/weather.py
"""Real rainfall, ground wetness and river flow for any date, from Open-Meteo (free, no account).

  rainfall + wetness  ERA5 reanalysis for past dates (1940 onwards); weather forecast for the last
                      three months and the next 16 days
  river flow          GloFAS (Copernicus Global Flood Awareness System), 1984 onwards plus forecast
"""
import datetime as dt
import json, time, urllib.parse, urllib.request
import numpy as np

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
FLOOD_URL = "https://flood-api.open-meteo.com/v1/flood"
TZ = "Africa/Nairobi"
FORECAST_PAST_LIMIT_DAYS = 90
FORECAST_AHEAD_DAYS = 16

# SCS antecedent moisture thresholds, growing season (5-day rain before the storm, mm)
AMC_DRY_BELOW_MM = 36.0
AMC_WET_ABOVE_MM = 53.0

def _get_json(url, params, retries=3):
    q = urllib.parse.urlencode(params, safe=",")
    last = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(f"{url}?{q}", headers={"User-Agent": "atlas-floodcheck/0.2"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                return json.load(resp)
        except urllib.error.HTTPError as err:
            body = err.read().decode("utf-8", "ignore")
            try:
                reason = json.loads(body).get("reason", body)
            except Exception:
                reason = body
            if err.code == 429 or err.code >= 500:
                last = RuntimeError(f"The weather service is busy ({err.code}). Try again in a minute.")
                time.sleep(3 * (attempt + 1)); continue
            raise ValueError(f"The weather service rejected the request: {reason[:200]}")
        except urllib.error.URLError as err:
            last = RuntimeError(f"Can't reach the weather service: {err.reason}")
            time.sleep(3 * (attempt + 1))
    raise last

def _parse_date(s):
    try:
        return dt.date.fromisoformat(s)
    except ValueError:
        raise ValueError("Enter the date as YYYY-MM-DD.")

def _grid_points(bbox, n=3):
    w, s, e, nn = bbox
    lats = np.linspace(s, nn, n); lons = np.linspace(w, e, n)
    pts = [(float(la), float(lo)) for la in lats for lo in lons]
    return [p[0] for p in pts], [p[1] for p in pts]

def rainfall_for_event(bbox, date_str, days):
    """Area-average rainfall for the `days` days ending on `date_str`, plus the 5 days before for wetness."""
    end = _parse_date(date_str)
    if not 1 <= days <= 10:
        raise ValueError("Choose between 1 and 10 days of rain.")
    start = end - dt.timedelta(days=days - 1)
    ante_start = start - dt.timedelta(days=5)
    today = dt.date.today()
    if end > today + dt.timedelta(days=FORECAST_AHEAD_DAYS - 1):
        raise ValueError(f"Forecasts only reach {FORECAST_AHEAD_DAYS} days ahead. Pick an earlier date.")
    if ante_start < dt.date(1940, 1, 1):
        raise ValueError("Rainfall records start in 1940. Pick a later date.")

    lats, lons = _grid_points(bbox)
    common = dict(latitude=",".join(f"{v:.4f}" for v in lats), longitude=",".join(f"{v:.4f}" for v in lons),
                  hourly="precipitation", timezone=TZ)
    if ante_start >= today - dt.timedelta(days=FORECAST_PAST_LIMIT_DAYS):
        past = (today - ante_start).days + 1
        data = _get_json(FORECAST_URL, dict(common, past_days=min(max(past, 1), 92), forecast_days=FORECAST_AHEAD_DAYS))
        source = "Open-Meteo weather forecast and recent analysis"
    else:
        data = _get_json(ARCHIVE_URL, dict(common, start_date=ante_start.isoformat(), end_date=end.isoformat()))
        source = "ERA5 reanalysis (via Open-Meteo)"
    data = data if isinstance(data, list) else [data]

    times = np.array(data[0]["hourly"]["time"], dtype="datetime64[m]")
    vals = np.array([[np.nan if v is None else v for v in d["hourly"]["precipitation"]] for d in data], dtype="float64")
    mean = np.nanmean(vals, axis=0)
    day = times.astype("datetime64[D]")
    in_storm = (day >= np.datetime64(start)) & (day <= np.datetime64(end))
    in_ante = (day >= np.datetime64(ante_start)) & (day < np.datetime64(start))
    if not in_storm.any() or np.isnan(mean[in_storm]).all():
        raise ValueError("No rainfall data is available for those dates yet. Recent days can take up to a week to appear.")
    storm = np.nan_to_num(mean[in_storm])
    rain_mm = float(storm.sum())
    wet_hours = int((storm > 0.1).sum())
    ante_mm = float(np.nansum(mean[in_ante]))
    wetness = "dry" if ante_mm < AMC_DRY_BELOW_MM else "wet" if ante_mm > AMC_WET_ABOVE_MM else "normal"

    daily = []
    for k in range((end - ante_start).days + 1):
        d = ante_start + dt.timedelta(days=k)
        m = day == np.datetime64(d)
        daily.append(dict(date=d.isoformat(), rain_mm=round(float(np.nansum(mean[m])), 1), in_storm=bool(d >= start)))
    return dict(rain_mm=round(rain_mm, 1), rain_hours=max(wet_hours, 1), wetness=wetness,
                antecedent_mm=round(ante_mm, 1), start=start.isoformat(), end=end.isoformat(),
                is_forecast=bool(end > today), source=source, daily=daily,
                peak_hour_mm=round(float(storm.max()), 1))

def river_flow_for_event(lat, lon, date_str, days, search_deg=0.05):
    """GloFAS river flow near a point: peak around the event and the typical flow over the year before.

    GloFAS works on a ~5 km grid, so the nearest grid cell may sit on a smaller stream. We check the
    cells around the point and use the one carrying the most water, which is almost always the main river."""
    end = _parse_date(date_str)
    start = end - dt.timedelta(days=days - 1)
    win_start, win_end = start - dt.timedelta(days=1), end + dt.timedelta(days=4)   # flood waves arrive late
    base_start = start - dt.timedelta(days=365)
    if base_start < dt.date(1984, 1, 1):
        raise ValueError("River flow records start in 1984. Pick a later date.")
    offs = [-search_deg, 0.0, search_deg]
    lats = [lat + a for a in offs for _ in offs]; lons = [lon + b for _ in offs for b in offs]
    data = _get_json(FLOOD_URL, dict(latitude=",".join(f"{v:.4f}" for v in lats),
                                     longitude=",".join(f"{v:.4f}" for v in lons), daily="river_discharge",
                                     start_date=base_start.isoformat(), end_date=win_end.isoformat()))
    data = data if isinstance(data, list) else [data]
    best, best_mean = None, -1.0
    for d in data:
        q = np.array([np.nan if v is None else v for v in d["daily"]["river_discharge"]], dtype="float64")
        if np.isfinite(q).any() and np.nanmean(q) > best_mean:
            best, best_mean = (d, q), float(np.nanmean(q))
    if best is None:
        raise ValueError("No river flow data was found near this point. Place the inflow on a larger river.")
    d, q = best
    t = np.array(d["daily"]["time"], dtype="datetime64[D]")
    base = q[(t >= np.datetime64(base_start)) & (t < np.datetime64(start))]
    win = (t >= np.datetime64(win_start)) & (t <= np.datetime64(win_end))
    if not np.isfinite(q[win]).any():
        raise ValueError("No river flow data for those dates yet.")
    qwin = np.where(np.isfinite(q[win]), q[win], -1)
    k = int(np.argmax(qwin))
    return dict(flood_m3s=round(float(qwin[k]), 0), peak_date=str(t[win][k]),
                normal_m3s=round(float(np.nanmedian(base)), 0) if np.isfinite(base).any() else 0.0,
                grid_lat=d.get("latitude"), grid_lon=d.get("longitude"),
                series=[dict(date=str(tt), q=None if not np.isfinite(v) else round(float(v), 1))
                        for tt, v in zip(t[win], q[win])],
                source="GloFAS v4 (via Open-Meteo)")


## 4. Satellite images for any date

Sentinel-2 colour photos (10 m, from 2017) come from the Earth Search catalogue on AWS. Sentinel-1 radar images (10 m, from 2014) come from Microsoft Planetary Computer. The app finds the image nearest to the date you choose, choosing the least cloudy photo, and draws it as map tiles. Radar sees through cloud, and calm water shows up black, so it's the best way to check the model against a real flood.

In [ ]:
%%writefile /content/floodcheck/backend/imagery.py
"""Satellite images for any date, served as map tiles.

  Sentinel-2  10 m colour photos (2017 onwards), from the Element 84 Earth Search catalogue on AWS
  Sentinel-1  10 m radar (2014 onwards), terrain-corrected, from Microsoft Planetary Computer.
              Radar sees through cloud; calm water shows up black.
"""
import datetime as dt
import hashlib, io, threading, time
from collections import OrderedDict
import numpy as np
from PIL import Image

EARTH_SEARCH = "https://earth-search.aws.element84.com/v1"
PLANETARY_COMPUTER = "https://planetarycomputer.microsoft.com/api/stac/v1"
S2_WINDOW_DAYS = 15
S1_WINDOW_DAYS = 12
WATER_DB = -18.0              # radar backscatter below this is treated as open water

SELECTIONS = {}
TILE_CACHE = OrderedDict()
TILE_CACHE_MAX = 3000
_lock = threading.Lock()
EMPTY_TILE = None

def _empty_tile():
    global EMPTY_TILE
    if EMPTY_TILE is None:
        buf = io.BytesIO(); Image.new("RGBA", (256, 256), (0, 0, 0, 0)).save(buf, "PNG"); EMPTY_TILE = buf.getvalue()
    return EMPTY_TILE

def _search(url, collection, bbox, start, end, sign=False, query=None):
    import pystac_client
    last = None
    for attempt in range(4):
        try:
            client = pystac_client.Client.open(url)
            kw = dict(collections=[collection], bbox=bbox, datetime=f"{start.isoformat()}/{end.isoformat()}", max_items=200)
            if query:
                kw["query"] = query
            return list(client.search(**kw).items())
        except Exception as err:
            last = err
            if "rate limit" in str(err).lower():
                time.sleep(5 * (attempt + 1)); continue
            time.sleep(2)
    raise RuntimeError(f"The satellite image catalogue didn't respond: {str(last)[:150]}")

def _pick_date(items, target, score):
    by_day = {}
    for it in items:
        by_day.setdefault(it.datetime.date(), []).append(it)
    best = min(by_day, key=lambda d: score(d, by_day[d], target))
    return best, by_day[best]

def find_images(source, bbox, date_str):
    target = dt.date.fromisoformat(date_str)
    if source == "s2":
        items = _search(EARTH_SEARCH, "sentinel-2-l2a", bbox, target - dt.timedelta(days=S2_WINDOW_DAYS),
                        target + dt.timedelta(days=S2_WINDOW_DAYS), query={"eo:cloud_cover": {"lt": 90}})
        items = [it for it in items if "visual" in it.assets]
        if not items:
            raise ValueError(f"No Sentinel-2 photo within {S2_WINDOW_DAYS} days of that date. Try another date.")
        # prefer clear images, then closeness to the chosen date
        score = lambda d, its, t: np.mean([it.properties.get("eo:cloud_cover", 100) for it in its]) + 2.0 * abs((d - t).days)
        day, chosen = _pick_date(items, target, score)
        hrefs = [it.assets["visual"].href for it in chosen]
        info = dict(cloud_pct=round(float(np.mean([it.properties.get("eo:cloud_cover", 0) for it in chosen])), 0))
    elif source == "s1":
        items = _search(PLANETARY_COMPUTER, "sentinel-1-rtc", bbox, target - dt.timedelta(days=S1_WINDOW_DAYS),
                        target + dt.timedelta(days=S1_WINDOW_DAYS))
        items = [it for it in items if "vv" in it.assets]
        if not items:
            raise ValueError(f"No Sentinel-1 radar image within {S1_WINDOW_DAYS} days of that date. Try another date.")
        score = lambda d, its, t: abs((d - t).days)
        day, chosen = _pick_date(items, target, score)
        hrefs = [it.assets["vv"].href for it in chosen]
        info = dict(orbit=chosen[0].properties.get("sat:orbit_state"))
    else:
        raise ValueError("Unknown image source.")
    sel_id = hashlib.sha1(f"{source}|{day}|{'|'.join(sorted(hrefs))}".encode()).hexdigest()[:12]
    SELECTIONS[sel_id] = dict(source=source, hrefs=hrefs, date=day.isoformat())
    return dict(id=sel_id, source=source, date=day.isoformat(), days_from_target=(day - target).days,
                scenes=len(hrefs), tiles=f"api/imagery/{sel_id}/{{z}}/{{x}}/{{y}}.png", **info)

def _read_tile(href, x, y, z, **kw):
    from rio_tiler.io import Reader
    with Reader(href) as src:
        return src.tile(x, y, z, tilesize=256, **kw)

def render_tile(sel_id, z, x, y, water=False):
    key = (sel_id, z, x, y, water)
    with _lock:
        if key in TILE_CACHE:
            TILE_CACHE.move_to_end(key); return TILE_CACHE[key]
    sel = SELECTIONS.get(sel_id)
    if sel is None:
        raise KeyError(sel_id)
    from rio_tiler.mosaic import mosaic_reader
    from rio_tiler.errors import TileOutsideBounds, EmptyMosaicError
    hrefs = sel["hrefs"]
    if sel["source"] == "s1":
        import planetary_computer
        hrefs = [planetary_computer.sign(h) for h in hrefs]
    try:
        img, _ = mosaic_reader(hrefs, _read_tile, x, y, z, allowed_exceptions=(TileOutsideBounds,))
    except (EmptyMosaicError, TileOutsideBounds):
        return _empty_tile()
    mask = np.asarray(img.mask) > 0
    if sel["source"] == "s2":
        rgb = np.clip(np.asarray(img.data[:3]), 0, 255).astype("uint8").transpose(1, 2, 0)
        rgba = np.dstack([rgb, np.where(mask, 255, 0).astype("uint8")])
    else:
        vv = np.asarray(img.data[0], dtype="float32")
        db = 10 * np.log10(np.clip(vv, 1e-5, None))
        g = (np.clip((db + 25.0) / 25.0, 0, 1) * 255).astype("uint8")
        rgba = np.dstack([g, g, g, np.where(mask, 255, 0).astype("uint8")])
        if water:
            wm = mask & (db < WATER_DB)
            rgba[wm] = (0, 170, 255, 255)
    buf = io.BytesIO(); Image.fromarray(rgba, "RGBA").save(buf, "PNG", optimize=False)
    out = buf.getvalue()
    with _lock:
        TILE_CACHE[key] = out
        while len(TILE_CACHE) > TILE_CACHE_MAX:
            TILE_CACHE.popitem(last=False)
    return out


## 4b. Place search

Lets users find their village, town, school or market by name (or type coordinates) and sets the area around it. Uses OpenStreetMap Nominatim through the server, limited to the covered country, one request per second, with repeated searches cached.

In [ ]:
%%writefile /content/floodcheck/backend/geocode.py
"""Place search: turns a village, town, school or market name into a map location.

Uses OpenStreetMap Nominatim (free, no account), limited to the country extent the model covers.
Nominatim's usage policy asks for at most one request per second and a real User-Agent, so
requests go through the server (not the browser), are spaced out, and repeated searches are cached.
Typing coordinates ("-1.2921, 36.8219") skips the online search.
"""
import json, re, threading, time, urllib.error, urllib.parse, urllib.request
from collections import OrderedDict

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "atlas-floodcheck/1.0 (community flood screening tool)"
MIN_INTERVAL_S = 1.1
CACHE_MAX = 500

_cache = OrderedDict()
_lock = threading.Lock()
_last_call = [0.0]
_COORDS = re.compile(r"^\s*(-?\d{1,2}(?:\.\d+)?)\s*[,;\s]\s*(-?\d{1,3}(?:\.\d+)?)\s*$")

def _inside(lat, lon, bbox):
    w, s, e, n = bbox
    return w <= lon <= e and s <= lat <= n

def parse_coordinates(q):
    """'lat, lon' typed by the user, or None."""
    m = _COORDS.match(q or "")
    if not m:
        return None
    lat, lon = float(m.group(1)), float(m.group(2))
    if -90 <= lat <= 90 and -180 <= lon <= 180:
        return lat, lon
    return None

def search(q, bbox, limit=6):
    """Places matching q inside bbox (west, south, east, north), best match first."""
    q = (q or "").strip()
    if len(q) < 2:
        raise ValueError("Type at least two letters to search.")
    if len(q) > 120:
        raise ValueError("The search is too long. Use a shorter place name.")

    coords = parse_coordinates(q)
    if coords:
        lat, lon = coords
        if not _inside(lat, lon, bbox):
            raise ValueError("Those coordinates are outside the area the model covers.")
        return [dict(name=f"{lat:.5f}, {lon:.5f}", label="Coordinates you typed", lat=lat, lon=lon,
                     bbox=None, kind="coordinates")]

    key = (q.lower(), tuple(bbox), limit)
    with _lock:
        if key in _cache:
            _cache.move_to_end(key)
            return _cache[key]

    w, s, e, n = bbox
    params = dict(q=q, format="jsonv2", limit=limit, viewbox=f"{w},{n},{e},{s}", bounded=1, addressdetails=0)
    url = f"{NOMINATIM_URL}?{urllib.parse.urlencode(params)}"
    with _lock:                                   # one request at a time, spaced out
        wait = MIN_INTERVAL_S - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept-Language": "en"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                rows = json.load(resp)
        except urllib.error.HTTPError as err:
            raise RuntimeError(f"The place search service is busy ({err.code}). Try again in a moment.")
        except (urllib.error.URLError, TimeoutError) as err:
            raise RuntimeError("Can't reach the place search service. Check the internet connection.")
        finally:
            _last_call[0] = time.time()

    out = []
    for r in rows:
        try:
            lat, lon = float(r["lat"]), float(r["lon"])
        except (KeyError, TypeError, ValueError):
            continue
        bb = r.get("boundingbox")                 # [south, north, west, east] as strings
        box = [float(bb[2]), float(bb[0]), float(bb[3]), float(bb[1])] if bb and len(bb) == 4 else None
        label = r.get("display_name", "")
        out.append(dict(name=r.get("name") or label.split(",")[0], label=label, lat=lat, lon=lon,
                        bbox=box, kind=(r.get("type") or r.get("category") or "place").replace("_", " ")))
    with _lock:
        _cache[key] = out
        while len(_cache) > CACHE_MAX:
            _cache.popitem(last=False)
    return out


In [ ]:
%%writefile /content/floodcheck/backend/report.py
"""PDF flood report for a finished run: map, risk level, key numbers, what is in the water, advice.

Built with reportlab. The background map comes from Esri World Street Map, requested in the run's own
projection so it lines up exactly with the flood depth grid; if that fails, a hillshade of the ground is used.
"""
import io, time, urllib.parse, urllib.request
import numpy as np
from PIL import Image
from rasterio.warp import transform as warp_transform
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

import floodmodel as fm

NAVY, BLUE, TEAL, GREEN, MAGENTA, GREY = (colors.HexColor(c) for c in
                                          ("#0B1650", "#1E86B8", "#4BB8C2", "#3E9A44", "#C4307A", "#4A5578"))
LEVEL_COLOURS = [colors.HexColor(c) for c in ("#3E9A44", "#D9921A", "#C4307A", "#7E1046")]
ESRI_EXPORT = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/export"

# Keep in step with VERDICTS / riskLevel() in static/js/floodcheck.js
VERDICTS = [
    ("Low risk", "This rain is not expected to flood much land in the area.",
     ["Keep drains, culverts and furrows clear", "Keep watching the weather forecast"]),
    ("Some flooding likely", "Low land near rivers and streams may go under water.",
     ["Do not walk or drive through moving water", "Move animals, seed and stored food off low ground",
      "Check on neighbours who live near the river"]),
    ("Serious flooding likely", "Homes, roads or important places are in the flood water in this check.",
     ["Agree where your family would go on higher ground", "Keep documents, medicines and a phone charger ready to carry",
      "Do not cross flooded roads or bridges", "Tell local leaders and share the warning"]),
    ("Severe flooding likely", "Many homes or important places are in deep water in this check.",
     ["Be ready to leave early, before the water rises", "Move to higher ground when told to",
      "Never walk, swim or drive through flood water", "Follow instructions from local authorities"]),
]

def risk_level(s):
    e = s.get("exposure") or {}; b = e.get("buildings_flooded") or {}; fac = e.get("facilities_flooded") or []
    share = s.get("flooded_share_pct") or 0
    if not (s.get("flooded_area_km2") or 0) > 0:
        return 0
    lv = 0
    if share >= 1 or b.get("any", 0) > 0 or e.get("roads_flooded_km", 0) >= 0.5:
        lv = 1
    if share >= 5 or b.get("over_0.5m", 0) >= 20 or fac or e.get("main_roads_flooded_km", 0) >= 1:
        lv = 2
    if share >= 15 or b.get("over_1.0m", 0) >= 100 or any(f.get("depth", 0) > 1 for f in fac):
        lv = 3
    return lv

def _num(v, d=1):
    if isinstance(v, float):
        return f"{v:,.{d}f}".rstrip("0").rstrip(".") if d else f"{v:,.0f}"
    return f"{v:,}"

# ----------------------------------------------------------------------------- map image

def _bounds(result):
    t = result["transform"]; h, w = result["depth"].shape
    return t.c, t.f + t.e * h, t.c + t.a * w, t.f          # x0, y0, x1, y1 in the run's projection

def _basemap(result, size):
    """Street map for the run's extent, in its own projection. None if it can't be fetched."""
    x0, y0, x1, y1 = _bounds(result)
    epsg = result["crs"].to_epsg()
    params = dict(bbox=f"{x0},{y0},{x1},{y1}", bboxSR=epsg, imageSR=epsg, size=f"{size[0]},{size[1]}",
                  format="png", transparent="false", f="image", dpi=110)
    try:
        req = urllib.request.Request(f"{ESRI_EXPORT}?{urllib.parse.urlencode(params)}",
                                     headers={"User-Agent": "atlas-floodcheck/1.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            img = Image.open(io.BytesIO(r.read())).convert("RGB")
        return img.resize(size) if img.size != size else img
    except Exception as err:
        print(f"Report basemap unavailable: {err}")
        return None

def _hillshade(ground, res, size):
    g = np.where(ground == fm.NODATA, np.nan, ground.astype("float64"))
    g = np.where(np.isnan(g), np.nanmean(g) if np.isfinite(np.nanmean(g)) else 0, g)
    gy, gx = np.gradient(g, res)
    slope = np.arctan(np.hypot(gx, gy)); aspect = np.arctan2(-gx, gy)
    az, alt = np.radians(315), np.radians(45)
    hs = np.sin(alt) * np.cos(slope) + np.cos(alt) * np.sin(slope) * np.cos(az - aspect)
    hs = np.clip(150 + 90 * hs, 0, 255).astype("uint8")
    return Image.fromarray(np.dstack([hs, hs, hs + 0])).resize(size)

def map_image(result, facilities=(), max_px=1500):
    h, w = result["depth"].shape
    scale = max_px / max(h, w)
    size = (max(int(w * scale), 1), max(int(h * scale), 1))
    base = _basemap(result, size)
    credit = "Background: Esri, HERE, Garmin, © OpenStreetMap contributors"
    if base is None:
        base = _hillshade(result["ground"], abs(result["transform"].a), size)
        credit = "Background: shaded relief from Copernicus DEM"
    flood = Image.fromarray(fm.depth_rgba(result["depth"], result["permanent_water"]), "RGBA").resize(size, Image.NEAREST)
    img = base.convert("RGBA"); img.alpha_composite(flood)
    if facilities:
        from PIL import ImageDraw
        d = ImageDraw.Draw(img)
        t = result["transform"]
        xs, ys = warp_transform("EPSG:4326", result["crs"], [f["lon"] for f in facilities], [f["lat"] for f in facilities])
        for f, x, y in zip(facilities, xs, ys):
            px, py = (x - t.c) / t.a * scale, (y - t.f) / t.e * scale
            r = 14
            col = (62, 154, 68, 255) if f.get("kind") == "health" else (30, 134, 184, 255)
            d.ellipse([px - r, py - r, px + r, py + r], fill=col, outline=(255, 255, 255, 255), width=3)
    return img.convert("RGB"), credit, scale

# ----------------------------------------------------------------------------- PDF

class _Doc:
    def __init__(self, buf, logo_path):
        self.c = canvas.Canvas(buf, pagesize=A4, pageCompression=1)
        self.W, self.H = A4
        self.M = 16 * mm
        self.logo = logo_path
        self.page = 0
        self.y = 0

    def new_page(self, title):
        if self.page:
            self.footer(); self.c.showPage()
        self.page += 1
        c, W, H, M = self.c, self.W, self.H, self.M
        try:
            c.drawImage(ImageReader(self.logo), M, H - M - 13 * mm, width=36 * mm, height=13 * mm, mask="auto",
                        preserveAspectRatio=True, anchor="w")
        except Exception:
            pass
        c.setFillColor(NAVY); c.setFont("Helvetica-Bold", 17)
        c.drawRightString(W - M, H - M - 6 * mm, title)
        c.setFont("Helvetica", 9); c.setFillColor(GREY)
        c.drawRightString(W - M, H - M - 11 * mm, time.strftime("Generated %d %b %Y, %H:%M UTC", time.gmtime()))
        x = M
        for col in (TEAL, BLUE, NAVY, GREEN, MAGENTA):          # the five figures from the logo
            c.setFillColor(col); c.rect(x, H - M - 16 * mm, (W - 2 * M) / 5, 1.6 * mm, stroke=0, fill=1)
            x += (W - 2 * M) / 5
        self.y = H - M - 22 * mm

    def footer(self):
        c = self.c
        c.setFont("Helvetica", 8); c.setFillColor(GREY)
        c.drawString(self.M, 10 * mm, "FloodCheck by Atlas Geosystems · rapid flood screening from global satellite data")
        c.drawRightString(self.W - self.M, 10 * mm, f"Page {self.page}")

    def heading(self, text):
        self.y -= 3 * mm
        self.c.setFillColor(NAVY); self.c.setFont("Helvetica-Bold", 12.5)
        self.c.drawString(self.M, self.y, text)
        self.c.setStrokeColor(TEAL); self.c.setLineWidth(1)
        self.c.line(self.M, self.y - 2 * mm, self.W - self.M, self.y - 2 * mm)
        self.y -= 8 * mm

    def para(self, text, size=10, colour=colors.black, bold=False, indent=0, gap=1.5 * mm, width=None):
        c = self.c
        font = "Helvetica-Bold" if bold else "Helvetica"
        maxw = (width or (self.W - 2 * self.M)) - indent
        words, line = text.split(), ""
        c.setFont(font, size); c.setFillColor(colour)
        for wd in words:
            test = (line + " " + wd).strip()
            if c.stringWidth(test, font, size) > maxw and line:
                c.drawString(self.M + indent, self.y, line); self.y -= size * 1.35; line = wd
            else:
                line = test
        if line:
            c.drawString(self.M + indent, self.y, line); self.y -= size * 1.35
        self.y -= gap

    def bullets(self, items, colour=NAVY, size=10):
        for it in items:
            self.c.setFillColor(colour); self.c.circle(self.M + 2 * mm, self.y + 1.2 * mm, 1.1 * mm, stroke=0, fill=1)
            self.para(it, size=size, indent=6 * mm, gap=0.5 * mm)
        self.y -= 1.5 * mm

    def table(self, rows, widths, header=True, size=9.5):
        c, x0 = self.c, self.M
        rh = size * 1.9
        for i, row in enumerate(rows):
            if header and i == 0:
                c.setFillColor(NAVY); c.rect(x0, self.y - rh + 3, sum(widths), rh, stroke=0, fill=1)
                c.setFillColor(colors.white); c.setFont("Helvetica-Bold", size)
            else:
                if i % 2 == 0:
                    c.setFillColor(colors.HexColor("#F3F6F9")); c.rect(x0, self.y - rh + 3, sum(widths), rh, stroke=0, fill=1)
                c.setFillColor(colors.black); c.setFont("Helvetica", size)
            x = x0
            for cell, w in zip(row, widths):
                txt = str(cell)
                while c.stringWidth(txt, "Helvetica", size) > w - 8 and len(txt) > 4:
                    txt = txt[:-2]
                if txt != str(cell):
                    txt = txt[:-1] + "…"
                c.drawString(x + 4, self.y - rh / 2 + 1.5, txt)
                x += w
            self.y -= rh
        self.y -= 3 * mm

def build_pdf(run, logo_path):
    """Return the PDF report for a finished run as bytes."""
    result, s, req = run["result"], run["summary"], run["request"]
    e = s.get("exposure") or {}
    lv = risk_level(s); title, text, todo = VERDICTS[lv]
    w, so, ea, n = req.bbox
    mid = (so + n) / 2
    wk, hk = (ea - w) * 111.32 * np.cos(np.radians(mid)), (n - so) * 110.57

    buf = io.BytesIO()
    d = _Doc(buf, logo_path); c = d.c
    c.setTitle("FloodCheck flood report"); c.setAuthor("Atlas Geosystems FloodCheck")
    d.new_page("Flood report")

    # area line
    d.para(f"Area: {w:.4f}, {so:.4f} to {ea:.4f}, {n:.4f} (west, south, east, north; WGS84)  ·  "
           f"about {wk:.0f} × {hk:.0f} km", size=9, colour=GREY, gap=2 * mm)
    d.para(f"Storm: {_num(float(s['rain_mm']), 0)} mm of rain over {_num(float(s['rain_hours']))} hours on "
           f"{s['wetness']} ground" + (f", plus {len(s.get('inflows') or [])} river inflow(s)" if s.get("inflows") else ""),
           size=9, colour=GREY, gap=3 * mm)

    # verdict box
    box_h = 20 * mm
    c.setFillColor(LEVEL_COLOURS[lv]); c.roundRect(d.M, d.y - box_h + 4 * mm, d.W - 2 * d.M, box_h, 3 * mm, stroke=0, fill=1)
    c.setFillColor(colors.white); c.setFont("Helvetica", 9)
    c.drawString(d.M + 5 * mm, d.y - 1 * mm, "FLOOD RISK FOR THIS RAIN")
    c.setFont("Helvetica-Bold", 18); c.drawString(d.M + 5 * mm, d.y - 9 * mm, title)
    c.setFont("Helvetica", 9.5); c.drawString(d.M + 5 * mm, d.y - 14.5 * mm, text)
    for i, lab in enumerate(("Low", "Some", "Serious", "Severe")):         # 4-step meter
        bx = d.W - d.M - 4 * 21 * mm - 4 * mm + i * 21 * mm
        c.setFillColor(colors.white if i <= lv else colors.Color(1, 1, 1, alpha=0.35))
        c.roundRect(bx, d.y - 6 * mm, 19 * mm, 2.2 * mm, 1 * mm, stroke=0, fill=1)
        c.setFillColor(colors.white); c.setFont("Helvetica", 7.5); c.drawString(bx, d.y - 10.5 * mm, lab)
    d.y -= box_h + 2 * mm

    # map
    fac = (e.get("facilities_flooded") or [])[:40]
    img, credit, _ = map_image(result, fac)
    max_w, max_h = d.W - 2 * d.M, 112 * mm
    sc = min(max_w / img.width, max_h / img.height)
    mw, mh = img.width * sc, img.height * sc
    mx = d.M + (max_w - mw) / 2
    my = d.y - mh
    c.drawImage(ImageReader(img), mx, my, width=mw, height=mh)
    c.setStrokeColor(MAGENTA); c.setLineWidth(1.5); c.setDash(4, 3); c.rect(mx, my, mw, mh, stroke=1, fill=0); c.setDash()
    # north arrow
    ax, ay = mx + mw - 9 * mm, my + mh - 14 * mm
    c.setFillColor(colors.white); c.circle(ax, ay + 3 * mm, 5.5 * mm, stroke=0, fill=1)
    c.setFillColor(NAVY); p = c.beginPath(); p.moveTo(ax, ay + 7 * mm); p.lineTo(ax - 2.5 * mm, ay); p.lineTo(ax + 2.5 * mm, ay); p.close()
    c.drawPath(p, stroke=0, fill=1); c.setFont("Helvetica-Bold", 8); c.drawCentredString(ax, ay - 3.2 * mm, "N")
    # scale bar
    width_m = result["depth"].shape[1] * abs(result["transform"].a)
    target = width_m / 4
    nice = max(v for v in (100, 200, 500, 1000, 2000, 5000, 10000, 20000) if v <= max(target, 100))
    bar = nice / width_m * mw
    c.setFillColor(colors.white); c.rect(mx + 3 * mm, my + 3 * mm, bar + 6 * mm, 9 * mm, stroke=0, fill=1)
    c.setFillColor(NAVY); c.rect(mx + 6 * mm, my + 6 * mm, bar, 1.6 * mm, stroke=0, fill=1)
    c.setFont("Helvetica", 7.5); c.drawString(mx + 6 * mm, my + 8.6 * mm, f"{nice / 1000:g} km" if nice >= 1000 else f"{nice} m")
    d.y = my - 4 * mm
    # legend + credit
    c.setFont("Helvetica-Bold", 8.5); c.setFillColor(NAVY); c.drawString(d.M, d.y, "Water depth")
    stops = [(0.1, "#BDE4F4"), (0.5, "#6BAED6"), (1.0, "#3182BD"), (2.0, "#08519C"), (3.0, "#08306B")]
    lx = d.M + 22 * mm
    for i, (v, col) in enumerate(stops):
        c.setFillColor(colors.HexColor(col)); c.rect(lx + i * 15 * mm, d.y - 1 * mm, 15 * mm, 3.5 * mm, stroke=0, fill=1)
        c.setFillColor(GREY); c.setFont("Helvetica", 7); c.drawString(lx + i * 15 * mm, d.y - 4.5 * mm, f"{v:g} m" + ("+" if i == 4 else ""))
    c.setFillColor(colors.HexColor("#5A6E82")); c.rect(lx + 80 * mm, d.y - 1 * mm, 6 * mm, 3.5 * mm, stroke=0, fill=1)
    c.setFillColor(GREY); c.setFont("Helvetica", 7.5); c.drawString(lx + 88 * mm, d.y, "River or lake (always wet)")
    if fac:
        c.setFillColor(BLUE); c.circle(lx + 128 * mm, d.y + 0.7 * mm, 1.5 * mm, stroke=0, fill=1)
        c.setFillColor(GREY); c.drawString(lx + 131 * mm, d.y, "School")
        c.setFillColor(GREEN); c.circle(lx + 143 * mm, d.y + 0.7 * mm, 1.5 * mm, stroke=0, fill=1)
        c.setFillColor(GREY); c.drawString(lx + 146 * mm, d.y, "Health")
    c.setFont("Helvetica", 6.5); c.drawRightString(d.W - d.M, d.y - 8 * mm, credit)
    d.y -= 14 * mm

    # key numbers
    facts = [("Under water", f"{_num(float(s['flooded_area_km2']))} km²", f"{_num(float(s['flooded_share_pct']))}% of the area"),
             ("Deepest water", f"{_num(float(s['max_depth_m']))} m", ""),
             ("Average depth", f"{_num(float(s['mean_depth_m']))} m", ""),
             ("Deeper than 1 m", f"{_num(float(s['area_over_1m_km2']))} km²", "")]
    bw = (d.W - 2 * d.M - 3 * 4 * mm) / 4
    for i, (lab, val, sub) in enumerate(facts):
        bx = d.M + i * (bw + 4 * mm)
        c.setFillColor(colors.HexColor("#F3F6F9")); c.rect(bx, d.y - 16 * mm, bw, 18 * mm, stroke=0, fill=1)
        c.setFillColor((BLUE, TEAL, NAVY, GREEN)[i]); c.rect(bx, d.y - 16 * mm, 1.4 * mm, 18 * mm, stroke=0, fill=1)
        c.setFillColor(GREY); c.setFont("Helvetica", 8); c.drawString(bx + 4 * mm, d.y - 3 * mm, lab)
        c.setFillColor(NAVY); c.setFont("Helvetica-Bold", 14); c.drawString(bx + 4 * mm, d.y - 9.5 * mm, val)
        if sub:
            c.setFillColor(GREY); c.setFont("Helvetica", 7.5); c.drawString(bx + 4 * mm, d.y - 13.8 * mm, sub)
    d.y -= 22 * mm

    # ---- page 2
    d.new_page("Flood report")
    d.heading("What to do")
    d.para(text, size=10)
    d.bullets(todo, colour=LEVEL_COLOURS[lv])
    d.para("This is a simple guide from this model run. Always follow official weather and flood warnings.",
           size=8.5, colour=GREY, gap=3 * mm)

    d.heading("Homes, roads and services in the water")
    b = e.get("buildings_flooded")
    if b is not None:
        d.para(f"{_num(b.get('any', 0))} of {_num(e.get('buildings_total', 0))} buildings have flood water around them.", size=10)
        rows = [["Water depth around the building", "Buildings"]] + [
            [f"More than {k.split('_')[1].rstrip('m')} m", _num(b.get(k, 0))] for k in ("over_0.1m", "over_0.5m", "over_1.0m", "over_2.0m") if k in b]
        d.table(rows, [110 * mm, 40 * mm])
    if "roads_flooded_km" in e:
        d.para(f"Roads under water: {_num(float(e['roads_flooded_km']))} km of {_num(float(e.get('roads_total_km', 0)), 0)} km"
               f" (main roads: {_num(float(e.get('main_roads_flooded_km', 0)))} km).", size=10, gap=3 * mm)
    allfac = e.get("facilities_flooded") or []
    tot = e.get("facilities_total") or {}
    if tot or allfac:
        d.para(f"Schools and health facilities in the water: {len(allfac)}"
               + (f" (of {sum(tot.values())} in the area)" if tot else ""), size=10, gap=2 * mm)
    if allfac:
        rows = [["Name", "Type", "Water depth"]] + [[f.get("name") or "Unnamed", f.get("kind", "").capitalize(), f"{f['depth']:.1f} m"]
                                                    for f in allfac[:15]]
        d.table(rows, [100 * mm, 30 * mm, 30 * mm])
        if len(allfac) > 15:
            d.para(f"… and {len(allfac) - 15} more.", size=9, colour=GREY)
    if not e:
        d.para("Buildings, roads and facilities were not available for this run.", size=10, colour=GREY)

    d.heading("Inputs and method")
    rows = [["Input", "Value"],
            ["Total rain", f"{_num(float(s['rain_mm']), 0)} mm"], ["Falling over", f"{_num(float(s['rain_hours']))} hours"],
            ["Ground before the rain", s["wetness"].capitalize()],
            ["Rain that ran off", f"{_num(float(s['mean_runoff_pct']))}%"],
            ["Terrain", "Prepared ahead of time" if str(s.get("terrain", "")).startswith("prepared") else "Built for this check"]]
    for inf in s.get("inflows") or []:
        rows.append(["River inflow", f"{_num(float(inf['added_m3s']), 0)} m³/s added, followed for {_num(float(inf['river_km']))} km"])
    d.table(rows, [70 * mm, 100 * mm])
    notes = list(e.get("notes") or []) + ([s["soil_note"]] if s.get("soil_note") else [])
    if s.get("capped_segments"):
        notes.append(f"{s['capped_segments']} river stretches carried more water than the model's highest level; depths there may be underestimated.")
    if notes:
        d.bullets(notes, colour=GREY, size=9)
    d.para("Method: rainfall runoff from land cover and soils (SCS curve number), river flow from upstream area and "
           "travel time, and flood depth from height above the nearest river (HAND), on a 30 m grid.", size=8.5, colour=GREY)
    d.para("Data: Copernicus DEM GLO-30, ESA WorldCover 2021, ISRIC SoilGrids, Overture Maps and OpenStreetMap "
           "(buildings, roads, facilities), Open-Meteo / GloFAS (rain and river flow).", size=8.5, colour=GREY)
    d.para("A quick estimate from global satellite data. It shows where flooding is likely, not an engineering flood study, "
           "and it can miss walls, culverts and drains. Check the map against what the community has seen in past floods.",
           size=8.5, colour=GREY)
    d.footer(); c.save()
    return buf.getvalue()


## 5. The web server

It accepts a run request from the page, runs the model in the background (one run at a time), reports progress, and serves the flood map image, a GeoTIFF download, and depth lookups when you click the map.

In [ ]:
%%writefile /content/floodcheck/backend/server.py
"""Web server for the flood model: serves the map page and runs flood simulations in the background."""
import io, json, os, queue, threading, time, traceback, uuid
from collections import OrderedDict

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, Response
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from PIL import Image

import floodmodel as fm
import exposure as ex
import weather as wx
import imagery as im
import geocode as geo
import report

STATIC_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "static")
RESULTS_DIR = os.environ.get("FLOOD_RESULTS_DIR", "./results")
os.makedirs(RESULTS_DIR, exist_ok=True)

app = FastAPI(title="FloodCheck rapid flood model")
app.mount("/static", StaticFiles(directory=STATIC_DIR), name="static")   # logo and other page assets

# Allow other websites (e.g. a separate Angular frontend) to call this API.
# FLOOD_CORS_ORIGINS="http://localhost:4200,https://example.github.io"  ("*" allows any site)
_cors = [o.strip().rstrip("/") for o in os.environ.get("FLOOD_CORS_ORIGINS", "").split(",") if o.strip()]
if _cors:
    from fastapi.middleware.cors import CORSMiddleware
    app.add_middleware(CORSMiddleware, allow_origins=_cors, allow_methods=["GET", "POST"], allow_headers=["*"])

class Inflow(BaseModel):
    lat: float
    lon: float
    flood_m3s: float = Field(gt=0)
    normal_m3s: float = Field(default=0, ge=0)

class RunRequest(BaseModel):
    bbox: list[float] = Field(min_length=4, max_length=4)   # west, south, east, north
    rain_mm: float = Field(gt=0, le=1000)
    rain_hours: float = Field(gt=0, le=240)
    wetness: str = Field(default="normal", pattern="^(dry|normal|wet)$")
    inflows: list[Inflow] = []

RUNS = {}                       # run id -> state dict
TERRAIN = OrderedDict()         # cached terrain for recently used areas (the slow part)
TERRAIN_CACHE_SIZE = max(int(os.environ.get("FLOOD_TERRAIN_CACHE", "2")), 0)   # areas kept in memory (1 on small servers)
JOBS = queue.Queue()
KEEP_RUNS = max(int(os.environ.get("FLOOD_KEEP_RUNS", "10")), 1)   # finished runs kept for the map, layers and downloads

def prune_runs():
    """Forget the oldest finished runs (and their GeoTIFFs) so memory doesn't grow with every check."""
    finished = sorted((r.get("finished", 0), rid) for rid, r in RUNS.items() if r["status"] in ("done", "failed"))
    for _, rid in finished[:max(len(finished) - KEEP_RUNS, 0)]:
        old = RUNS.pop(rid, None)
        if old and old.get("tif_path"):
            try:
                os.remove(old["tif_path"])
            except OSError:
                pass

def terrain_key(bbox):
    return tuple(round(v, 4) for v in bbox)

def worker():
    while True:
        run_id = JOBS.get()
        run = RUNS[run_id]
        req = run["request"]
        run["status"] = "running"; run["started"] = time.time()

        def log(msg):
            run["step"] = msg
            run["log"].append(msg)

        try:
            key = terrain_key(req.bbox)
            if key in TERRAIN:
                TERRAIN.move_to_end(key)
                log("Reusing downloaded data for this area")
                terrain = TERRAIN[key]
            else:
                terrain = fm.load_terrain(req.bbox, log=log)
                terrain["exposure"] = ex.load_exposure(req.bbox, log=log)
                if TERRAIN_CACHE_SIZE:
                    TERRAIN[key] = terrain
                while len(TERRAIN) > TERRAIN_CACHE_SIZE:
                    TERRAIN.popitem(last=False)
            result = fm.run_storm(terrain, req.rain_mm, req.rain_hours, req.wetness,
                                  [i.model_dump() for i in req.inflows], log=log)
            log("Checking buildings, roads and facilities")
            exp_summary, exp_layers = ex.analyse(result, terrain["exposure"], req.bbox)
            result["summary"]["exposure"] = exp_summary
            run["layers"] = {k: json.dumps(v, separators=(",", ":")) for k, v in exp_layers.items()}
            log("Drawing the map")
            buf = io.BytesIO()
            Image.fromarray(fm.depth_rgba(result["depth"], result["permanent_water"]), "RGBA").save(buf, "PNG", optimize=True)
            run["png"] = buf.getvalue()
            run["tif_path"] = os.path.join(RESULTS_DIR, f"{run_id}.tif")
            fm.save_depth_geotiff(result, run["tif_path"])
            run["result"] = result
            run["corners"] = fm.image_corners(result)
            run["summary"] = result["summary"]
            run["status"] = "done"
        except Exception as err:
            traceback.print_exc()
            run["status"] = "failed"
            run["error"] = str(err) if isinstance(err, (ValueError, RuntimeError)) else \
                f"The model stopped with an internal error: {err}"
        finally:
            run["finished"] = time.time()
            prune_runs()
            JOBS.task_done()

threading.Thread(target=worker, daemon=True).start()

@app.get("/")
def index():
    return FileResponse(os.path.join(STATIC_DIR, "index.html"))

@app.get("/api/config")
def config():
    out = dict(kenya_bbox=fm.KENYA_BBOX, max_side_deg=fm.MAX_FOCUS_DEG,
               prepared_regions=fm.prepared_regions(), live_terrain=fm.LIVE_TERRAIN)
    tiles = os.environ.get("FLOOD_STREET_TILES", "").strip()       # optional own street map provider
    if tiles:
        out["street_map"] = dict(tiles=[t.strip() for t in tiles.split(",") if t.strip()],
                                 attribution=os.environ.get("FLOOD_STREET_ATTRIBUTION", "© OpenStreetMap contributors"))
    return out

@app.post("/api/runs")
def create_run(req: RunRequest):
    try:
        fm.validate_focus(req.bbox)
        if not fm.LIVE_TERRAIN and fm.find_region(req.bbox) is None:
            raise ValueError("This area isn't prepared yet. Draw your area inside the prepared regions outlined on the map.")
    except ValueError as err:
        raise HTTPException(status_code=422, detail=str(err))
    run_id = uuid.uuid4().hex[:10]
    cached = terrain_key(req.bbox) in TERRAIN
    RUNS[run_id] = dict(status="queued", step="Waiting to start", log=[], request=req,
                        created=time.time(), cached_terrain=cached)
    JOBS.put(run_id)
    return dict(id=run_id, queue_position=JOBS.qsize(), cached_terrain=cached)

def get_run(run_id):
    run = RUNS.get(run_id)
    if run is None:
        raise HTTPException(status_code=404, detail="This run no longer exists. Run the model again.")
    return run

@app.get("/api/runs/{run_id}")
def run_status(run_id: str):
    run = get_run(run_id)
    out = dict(id=run_id, status=run["status"], step=run["step"], log=run["log"],
               elapsed_s=round(time.time() - run.get("started", time.time()), 1),
               cached_terrain=run["cached_terrain"])
    if run["status"] == "done":
        out.update(summary=run["summary"], corners=run["corners"],
                   image_url=f"api/runs/{run_id}/depth.png", geotiff_url=f"api/runs/{run_id}/depth.tif", report_url=f"api/runs/{run_id}/report.pdf",
                   layers={k: f"api/runs/{run_id}/layers/{k}.geojson" for k in run.get("layers", {})})
    if run["status"] == "failed":
        out["error"] = run["error"]
    return out

@app.get("/api/runs/{run_id}/depth.png")
def run_png(run_id: str):
    run = get_run(run_id)
    if "png" not in run:
        raise HTTPException(status_code=409, detail="This run hasn't finished yet.")
    return Response(run["png"], media_type="image/png", headers={"Cache-Control": "max-age=3600"})

@app.get("/api/runs/{run_id}/depth.tif")
def run_tif(run_id: str):
    run = get_run(run_id)
    if "tif_path" not in run:
        raise HTTPException(status_code=409, detail="This run hasn't finished yet.")
    return FileResponse(run["tif_path"], media_type="image/tiff", filename=f"flood_depth_{run_id}.tif")

@app.get("/api/runs/{run_id}/report.pdf")
def run_report(run_id: str):
    """Printable PDF report: map, flood risk, key numbers, what is in the water, advice."""
    run = get_run(run_id)
    if run.get("status") != "done":
        raise HTTPException(status_code=409, detail="This run hasn't finished yet.")
    if "pdf" not in run:
        try:
            run["pdf"] = report.build_pdf(run, os.path.join(STATIC_DIR, "assets", "logo-full.png"))
        except Exception as err:
            traceback.print_exc()
            raise HTTPException(status_code=500, detail=f"The report couldn't be made: {err}")
    stamp = time.strftime("%Y%m%d", time.localtime(run.get("finished", time.time())))
    return Response(run["pdf"], media_type="application/pdf",
                    headers={"Content-Disposition": f'attachment; filename="floodcheck_report_{stamp}_{run_id}.pdf"'})

@app.get("/api/runs/{run_id}/depth")
def run_depth_at(run_id: str, lat: float, lon: float):
    run = get_run(run_id)
    if "result" not in run:
        raise HTTPException(status_code=409, detail="This run hasn't finished yet.")
    depth, water = fm.depth_at(run["result"], lat, lon)
    return dict(depth_m=depth, permanent_water=water)

@app.get("/api/runs/{run_id}/layers/{name}.geojson")
def run_layer(run_id: str, name: str):
    run = get_run(run_id)
    if name not in run.get("layers", {}):
        raise HTTPException(status_code=404, detail="This layer isn't available for this run.")
    return Response(run["layers"][name], media_type="application/geo+json")

class ProfileRequest(BaseModel):
    coords: list[list[float]] = Field(min_length=2, max_length=50)

@app.post("/api/runs/{run_id}/profile")
def run_profile(run_id: str, req: ProfileRequest):
    run = get_run(run_id)
    if "result" not in run:
        raise HTTPException(status_code=409, detail="This run hasn't finished yet.")
    return fm.profile(run["result"], req.coords)

# ----------------------------------------------------------------------------- real weather, river flow, imagery

def _bbox_param(bbox: str):
    try:
        b = [float(v) for v in bbox.split(",")]
        assert len(b) == 4 and b[0] < b[2] and b[1] < b[3]
        return b
    except Exception:
        raise HTTPException(status_code=422, detail="The area is invalid. Draw it again.")

def _friendly(fn, *args):
    try:
        return fn(*args)
    except ValueError as err:
        raise HTTPException(status_code=422, detail=str(err))
    except RuntimeError as err:
        raise HTTPException(status_code=503, detail=str(err))

@app.get("/api/weather")
def weather(bbox: str, date: str, days: int = 1):
    return _friendly(wx.rainfall_for_event, _bbox_param(bbox), date, days)

@app.get("/api/river-flow")
def river_flow(lat: float, lon: float, date: str, days: int = 1):
    return _friendly(wx.river_flow_for_event, lat, lon, date, days)

@app.get("/api/imagery/search")
def imagery_search(source: str, bbox: str, date: str):
    return _friendly(im.find_images, source, _bbox_param(bbox), date)

@app.get("/api/imagery/{sel_id}/{z}/{x}/{y}.png")
def imagery_tile(sel_id: str, z: int, x: int, y: int, water: bool = False):
    try:
        png = im.render_tile(sel_id, z, x, y, water)
    except KeyError:
        raise HTTPException(status_code=404, detail="Image selection expired. Choose the date again.")
    except Exception as err:
        print("tile error", sel_id, z, x, y, err)
        png = im._empty_tile()
    return Response(png, media_type="image/png", headers={"Cache-Control": "max-age=86400"})

# ----------------------------------------------------------------------------- place search

@app.get("/api/search")
def place_search(q: str):
    """Find a village, town, school or market by name (or typed coordinates) inside the covered area."""
    return dict(results=_friendly(geo.search, q, fm.KENYA_BBOX), attribution="Search © OpenStreetMap contributors (Nominatim)")


## 6. The dashboard

The Atlas Geosystems FloodCheck dashboard, built for local users: three plain steps, quick rain choices, a flood risk card with simple advice, and a depth legend in everyday terms. It keeps every tool of the original map page: depth on click, layers, river inflow, before/after satellite images, 3D view, cross-sections and GeoTIFF download.

The page, its styles, its script and the logo are separate files, exactly as in the repository.

In [ ]:
%%writefile /content/floodcheck/backend/static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>FloodCheck · Atlas Geosystems</title>
<link rel="icon" type="image/png" href="static/assets/logo-mark.png">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:wght@400;700&family=Montserrat:wght@600;700;800&display=swap" rel="stylesheet">
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<link href="static/css/floodcheck.css" rel="stylesheet">
</head>
<body>
<div class="app">
  <header class="brandbar">
    <img src="static/assets/logo-full.png" alt="Atlas Geosystems — Community Centered Geosystems">
    <span class="brand-divider" aria-hidden="true"></span>
    <h1 class="app-name">FloodCheck<small>See where flood water could reach in your community</small></h1>
  </header>

  <aside class="panel" aria-label="Flood check controls" data-sheet="half">
    <button type="button" class="sheet-handle" id="sheet-handle" aria-controls="panel-scroll" aria-expanded="true">
      <span class="grabber" aria-hidden="true"></span><span id="sheet-label">Show more map</span>
    </button>
    <nav class="tabs" role="tablist" aria-label="FloodCheck sections">
      <button role="tab" class="tab" id="tab-area" data-tab="area" aria-selected="true" aria-controls="pane-area">Area</button>
      <button role="tab" class="tab" id="tab-rain" data-tab="rain" aria-selected="false" aria-controls="pane-rain">Rain</button>
      <button role="tab" class="tab" id="tab-results" data-tab="results" aria-selected="false" aria-controls="pane-results">Results <span class="tab-badge" id="results-badge" hidden>1</span></button>
      <button role="tab" class="tab" id="tab-report" data-tab="report" aria-selected="false" aria-controls="pane-report">Report</button>
    </nav>
    <div class="panel-scroll" id="panel-scroll">

      <!-- ===================== AREA ===================== -->
      <div class="pane" id="pane-area" role="tabpanel" aria-labelledby="tab-area">
        <section class="sec" id="step-area">
          <h2 class="sec-title">Set your area</h2>
          <p class="sec-sub">The flood check covers the box you mark on the map. Keep each side under about <span id="max-km">44</span> km.</p>
          <button class="btn btn-primary btn-wide" id="draw-btn" aria-pressed="false"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M4 4h4M16 4h4v4M20 16v4h-4M8 20H4v-4M4 10v4M20 10v4M10 4h4M10 20h4" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"/></svg>Draw my area on the map</button>
          <button class="btn btn-wide" id="example-btn"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M12 3l1.8 5.2L19 10l-5.2 1.8L12 17l-1.8-5.2L5 10l5.2-1.8z" fill="none" stroke="currentColor" stroke-width="2" stroke-linejoin="round"/></svg>Use an example area</button>

          <label for="place-q" class="sec-label">Search for a place</label>
          <div class="search-row" role="search">
            <input type="search" id="place-q" placeholder="Village, town, school or market" autocomplete="off" enterkeyhint="search" aria-describedby="place-help">
            <button class="btn btn-search" id="place-btn" type="button" aria-label="Search"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><circle cx="11" cy="11" r="6.5" fill="none" stroke="currentColor" stroke-width="2.2"/><path d="M16 16l4.5 4.5" stroke="currentColor" stroke-width="2.2" stroke-linecap="round"/></svg><span>Search</span></button>
          </div>
          <p class="hint" id="place-help">Or type coordinates, for example -0.45, 39.65.</p>
          <ul class="place-results" id="place-results" aria-live="polite"></ul>

          <div class="coord-card" id="area-readout" aria-live="polite">No area marked yet.</div>
          <p class="hint regions-note" id="regions-note" hidden>Areas outlined in teal on the map are prepared ahead of time, so checks there are much faster.</p>
          <button class="btn btn-next btn-wide" id="next-rain" data-goto="rain" hidden>Next: choose the rain <svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M5 12h13M13 6l6 6-6 6" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/></svg></button>
        </section>
      </div>

      <!-- ===================== RAIN ===================== -->
      <div class="pane" id="pane-rain" role="tabpanel" aria-labelledby="tab-rain" hidden>
        <section class="sec" id="step-storm">
          <h2 class="sec-title">How much rain?</h2>
          <p class="sec-sub">Use real rainfall for a date, or choose an amount yourself.</p>
          <div class="seg seg-2" role="radiogroup" aria-label="Where the rain amount comes from">
            <input type="radio" name="rainsrc" id="rs-real" value="real" checked><label for="rs-real">Real weather</label>
            <input type="radio" name="rainsrc" id="rs-own" value="own"><label for="rs-own">Choose myself</label>
          </div>
          <div class="real-weather" id="real-weather">
            <p class="hint">Past dates use measured rain; dates in the next 16 days use the forecast.</p>
            <div class="real-grid">
              <div class="field"><label for="wx-date" class="sec-label">Last day of rain</label><input type="date" id="wx-date" min="1940-01-10"></div>
              <div class="field"><label for="wx-days" class="sec-label">Days of rain</label>
                <select id="wx-days"><option>1</option><option>2</option><option selected>3</option><option>5</option><option>7</option></select></div>
            </div>
            <button class="btn btn-accent btn-wide" id="wx-btn"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M7 15a4 4 0 0 1-.5-8 5.5 5.5 0 0 1 10.6 1.5A3.5 3.5 0 0 1 17 15z" fill="none" stroke="currentColor" stroke-width="2" stroke-linejoin="round"/><path d="M9 18l-1 2.5M13 18l-1 2.5M17 18l-1 2.5" stroke="currentColor" stroke-width="2" stroke-linecap="round"/></svg>Get rainfall</button>
            <div id="wx-result" aria-live="polite"></div>
          </div>
          <div id="own-presets">
            <span class="sec-label">Quick choice</span>
            <div class="presets" role="group" aria-label="Quick rain amounts">
              <button type="button" data-mm="30" data-h="6"><b>30 mm</b>Light</button>
              <button type="button" data-mm="75" data-h="12"><b>75 mm</b>Heavy</button>
              <button type="button" data-mm="150" data-h="24"><b>150 mm</b>Very heavy</button>
              <button type="button" data-mm="250" data-h="48"><b>250 mm</b>Extreme</button>
            </div>
          </div>
          <div class="field" style="margin-top:12px">
            <label for="rain-range" class="sec-label">Total rain</label>
            <div class="num-row">
              <input type="range" id="rain-range" min="10" max="400" step="5" value="100" aria-label="Total rain in millimetres">
              <div class="unit-input"><input type="number" id="rain-mm" min="1" max="1000" value="100" aria-label="Total rain"><span>mm</span></div>
            </div>
          </div>
          <div class="field" style="margin-bottom:0">
            <label for="rain-hours" class="sec-label">Falling over</label>
            <div class="unit-input" style="width:150px"><input type="number" id="rain-hours" min="0.5" max="240" step="0.5" value="24"><span>hours</span></div>
          </div>
        </section>

        <section class="sec">
          <h2 class="sec-title" id="wet-label">Ground before the rain</h2>
          <p class="sec-sub">Wet ground soaks up less water, so floods are bigger.</p>
          <div class="seg" role="radiogroup" aria-labelledby="wet-label">
            <input type="radio" name="wetness" id="wet-dry" value="dry"><label for="wet-dry">Dry</label>
            <input type="radio" name="wetness" id="wet-normal" value="normal" checked><label for="wet-normal">Normal</label>
            <input type="radio" name="wetness" id="wet-wet" value="wet"><label for="wet-wet">Wet</label>
          </div>
        </section>

        <section class="sec" id="step-inflow">
          <h2 class="sec-title">Big river coming in <span class="optional">(optional)</span></h2>
          <p class="sec-sub">A large river can bring flood water from far away. Tap where it enters your area, then get its real flow or type it in.</p>
          <div id="inflow-list"></div>
          <button class="btn btn-wide" id="inflow-btn" aria-pressed="false"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M3 9c3-2 6 2 9 0s6-2 9 0M3 15c3-2 6 2 9 0s6-2 9 0" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"/></svg>Mark where the river enters</button>
        </section>
      </div>

      <!-- ===================== RESULTS ===================== -->
      <div class="pane" id="pane-results" role="tabpanel" aria-labelledby="tab-results" hidden>
        <section class="sec empty" id="results-empty">
          <h2 class="sec-title">No results yet</h2>
          <p class="sec-sub" id="results-empty-text">Set your area, choose the rain, then press <strong>Check for floods</strong>.</p>
        </section>

        <section class="status" id="status" hidden aria-live="polite">
          <h2 id="status-title">Checking for floods</h2>
          <div id="status-text"></div>
          <ul class="log" id="status-log"></ul>
        </section>

        <section class="results" id="results" hidden aria-live="polite">
          <div class="verdict" id="verdict" data-level="0">
            <div class="verdict-top">
              <p class="verdict-kicker">Flood risk for this rain</p>
              <p class="verdict-title" id="verdict-title"></p>
              <div class="meter" aria-hidden="true"><span></span><span></span><span></span><span></span></div>
              <div class="meter-labels" aria-hidden="true"><span>Low</span><span>Some</span><span>Serious</span><span>Severe</span></div>
            </div>
            <div class="verdict-body">
              <p id="verdict-text"></p>
              <h3>What to do</h3>
              <ul class="todo" id="verdict-do"></ul>
              <p class="verdict-note">This is a simple guide from this model run. Always follow official weather and flood warnings.</p>
            </div>
          </div>

          <div class="headline-block">
            <div class="headline" id="res-area"></div>
            <p class="headline-sub" id="res-sub"></p>
            <dl class="facts">
              <div><dt>Deepest water</dt><dd id="res-max"></dd></div>
              <div><dt>Average depth</dt><dd id="res-mean"></dd></div>
              <div><dt>Deeper than 1 m</dt><dd id="res-deep"></dd></div>
              <div><dt>Rain that ran off</dt><dd id="res-runoff"></dd></div>
            </dl>
            <p class="hint" style="margin:10px 0 0">Tap anywhere on the map to see how deep the water is there.</p>
          </div>

          <div class="card" id="expo" hidden>
            <h3>Homes, roads and services in the water</h3>
            <p class="hint" id="expo-buildings-head"></p>
            <ul class="band-list" id="expo-bands"></ul>
            <p class="hint" id="expo-roads" style="margin-top:10px"></p>
            <p class="hint" id="expo-fac-head" style="margin:10px 0 0"></p>
            <ul class="fac-list" id="expo-fac"></ul>
          </div>

          <details class="card profile">
            <summary>Look across a river (cross-section)</summary>
            <p class="hint">Draw a line across a river to see the ground and the flood water along it.</p>
            <button class="btn" id="profile-btn" aria-pressed="false"><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><circle cx="5" cy="18" r="2" fill="currentColor"/><circle cx="19" cy="6" r="2" fill="currentColor"/><path d="M6.5 16.5l11-9" stroke="currentColor" stroke-width="2" stroke-dasharray="3 2.5"/></svg>Draw a line</button>
            <div id="profile-chart"></div>
            <div class="profile-readout" id="profile-readout" aria-live="polite"></div>
          </details>
          <ul class="notes" id="res-notes"></ul>
          <button class="btn btn-next btn-wide" data-goto="report">Get the report <svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M5 12h13M13 6l6 6-6 6" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/></svg></button>
        </section>
      </div>

      <!-- ===================== REPORT ===================== -->
      <div class="pane" id="pane-report" role="tabpanel" aria-labelledby="tab-report" hidden>
        <section class="sec empty" id="report-empty">
          <h2 class="sec-title">No report yet</h2>
          <p class="sec-sub" id="report-empty-text">Run a flood check first. The report will then be ready to download here.</p>
        </section>
        <section class="sec" id="report-ready" hidden>
          <h2 class="sec-title">Download</h2>
          <p class="sec-sub">A printable PDF with the flood map, the risk level, what is in the water and what to do. Good for sharing with leaders and planners.</p>
          <a class="btn btn-primary btn-wide" id="res-pdf" href="#" download><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M7 3h7l5 5v13H7z" fill="none" stroke="currentColor" stroke-width="2" stroke-linejoin="round"/><path d="M14 3v5h5M10 13v5M10 18h4M10 13h3" stroke="currentColor" stroke-width="2" stroke-linecap="round"/></svg>Download PDF report</a>
          <span class="sec-label">For GIS software</span>
          <a class="btn btn-wide" id="res-download" href="#" download><svg class="ic" viewBox="0 0 24 24" aria-hidden="true"><path d="M12 4l9 5-9 5-9-5zM3 14l9 5 9-5" fill="none" stroke="currentColor" stroke-width="2" stroke-linejoin="round"/></svg>Flood depth (GeoTIFF)</a>
          <p class="hint" style="margin-top:8px">The GeoTIFF holds water depth in metres for every 30 m cell. Open it in QGIS or ArcGIS.</p>
        </section>
        <p class="disclaimer">A quick estimate from global satellite data. It shows where flooding is likely, not an engineering flood study, and it can miss walls, culverts and drains. Local knowledge matters: check the map against what your community has seen in past floods.</p>
      </div>
    </div>

    <div class="run-bar">
      <button class="run-btn" id="run-btn" disabled>Check for floods</button>
      <p class="run-note" id="run-note">Mark your area to start.</p>
    </div>
  </aside>

  <div id="map" role="region" aria-label="Map">
    <div class="map-overlay-left">
    <div class="map-tools">
      <div class="tool-group" role="group" aria-label="Background map">
        <button id="bm-map" aria-pressed="true">Map</button>
        <button id="bm-sat" aria-pressed="false">Satellite</button>
      </div>
      <div class="tool-group"><button id="terrain-btn" aria-pressed="false">3D view</button></div>
      <div class="tool-group"><button id="compare-btn" aria-pressed="false"><span class="lbl-long">Before / after photos</span><span class="lbl-short">Compare</span></button></div>
    </div>
    <details class="layers-card" id="layers-card" open hidden>
      <summary>What the map shows</summary>
      <div id="layer-rows"></div>
    </details>
    </div>
    <div class="compare-bar" id="compare-bar" hidden>
      <div class="cmp-side">
        <label for="cmp-left-src">Left side</label>
        <select id="cmp-left-src"><option value="s2" selected>Sentinel-2 photo (10 m)</option><option value="s1">Sentinel-1 radar (10 m)</option><option value="modis">MODIS daily photo (250 m)</option></select>
        <input type="date" id="cmp-left" value="2023-10-15" min="2014-10-01" aria-label="Left image date">
        <p class="cmp-status" id="cmp-left-status" aria-live="polite"></p>
      </div>
      <div class="cmp-side">
        <label for="cmp-right-src">Right side</label>
        <select id="cmp-right-src"><option value="s2">Sentinel-2 photo (10 m)</option><option value="s1" selected>Sentinel-1 radar (10 m)</option><option value="modis">MODIS daily photo (250 m)</option></select>
        <input type="date" id="cmp-right" value="2023-11-25" min="2014-10-01" aria-label="Right image date">
        <p class="cmp-status" id="cmp-right-status" aria-live="polite"></p>
      </div>
      <div class="cmp-opts">
        <label class="check"><input type="checkbox" id="cmp-flood" checked> Model flood on right</label>
        <label class="check"><input type="checkbox" id="cmp-water" checked> Show radar water in blue</label>
      </div>
      <p class="cmp-note">The nearest image to each date is used. Photos can be hidden by cloud; radar sees through it, and calm water shows up black (or blue).</p>
    </div>
    <div id="map-after" hidden></div>
    <div class="swipe" id="swipe" hidden><button id="swipe-handle" aria-label="Drag to compare the two images">‹›</button></div>
    <div class="map-hint" id="map-hint" hidden></div>
    <div class="legend" id="legend" hidden>
      <div class="legend-title">How deep is the water?</div>
      <div class="legend-ramp" aria-hidden="true"></div>
      <div class="legend-scale"><span>0.1 m</span><span>1 m</span><span>3 m+</span></div>
      <div class="legend-words"><span>Ankle</span><span>Waist</span><span>Over head</span></div>
      <div class="legend-water"><i aria-hidden="true"></i>River or lake (always wet)</div>
      <label>See-through <input type="range" id="opacity" min="0.2" max="1" step="0.05" value="0.85" style="width:100%"></label>
    </div>
  </div>
</div>

<script src="static/js/floodcheck.js"></script>
</body>
</html>


In [ ]:
%%writefile /content/floodcheck/backend/static/css/floodcheck.css
/* FloodCheck dashboard styles: Atlas Geosystems palette */
:root {
  /* Atlas Geosystems palette, taken from the logo */
  --navy: #0B1650;         /* pin + wordmark */
  --navy-soft: #2B3770;
  --blue: #1E86B8;         /* globe blue, figure 2 */
  --teal: #4BB8C2;         /* figure 1 */
  --green: #3E9A44;        /* figure 4, "Geosystems" */
  --magenta: #C4307A;      /* figure 5 */
  --amber: #D9921A;
  --paper: #F3F6F9;        /* cool panel ground */
  --card: #FFFFFF;
  --rule: #D5DCE6;
  --ink: #0B1650;
  --ink-soft: #4A5578;
  --error: #A3204F;
  --radius-s: 6px;
  --radius-m: 12px;
  --font: "Atkinson Hyperlegible", "Segoe UI", system-ui, -apple-system, sans-serif;
  --display: "Montserrat", "Segoe UI", system-ui, sans-serif;
}
* { box-sizing: border-box; }
html, body { height: 100%; margin: 0; overflow: hidden; overflow: clip; overscroll-behavior: none; }
body { font-family: var(--font); color: var(--ink); background: var(--paper); font-size: 16px; line-height: 1.5; }
button, input, select { font: inherit; color: inherit; }
:focus-visible { outline: 3px solid var(--blue); outline-offset: 2px; }

/* The page itself never scrolls: only the panel scrolls, and the map fills the rest. */
.app { display: grid; grid-template-columns: clamp(320px, 30vw, 410px) minmax(0, 1fr); grid-template-rows: auto minmax(0, 1fr);
       height: 100vh; height: 100dvh; overflow: hidden; }

/* ---------- brand header ---------- */
.brandbar { grid-column: 1 / -1; position: relative; display: flex; align-items: center; gap: 18px; padding: 8px 20px 12px; background: #fff;
            padding-top: calc(8px + env(safe-area-inset-top, 0px)); }
.brandbar::after { /* five figures from the logo, as one ribbon */
  content: ""; position: absolute; left: 0; right: 0; bottom: 0; height: 5px;
  background: linear-gradient(90deg, var(--teal) 0 20%, var(--blue) 20% 40%, var(--navy) 40% 60%, var(--green) 60% 80%, var(--magenta) 80% 100%); }
.brandbar img { height: 52px; width: auto; display: block; }
.brand-divider { width: 1px; align-self: stretch; margin: 6px 0; background: var(--rule); }
.app-name { font-family: var(--display); font-weight: 800; font-size: 22px; letter-spacing: -0.01em; line-height: 1.1; margin: 0; }
.app-name small { display: block; font-family: var(--font); font-weight: 400; font-size: 14px; color: var(--ink-soft); letter-spacing: 0; margin-top: 2px; }
.brand-help { margin-left: auto; }

#map { position: relative; height: 100%; min-height: 0; min-width: 0; }

/* ---------- panel ---------- */
.panel { display: flex; flex-direction: column; min-height: 0; min-width: 0; overflow: hidden; background: var(--paper); border-right: 1px solid var(--rule); }
.panel-scroll { position: relative; /* keeps the hidden radio inputs inside the scroll area */ flex: 1 1 auto; min-height: 0; overflow-y: auto; overflow-x: hidden; overscroll-behavior: contain; padding: 18px 18px 8px; }
.panel-scroll * { overflow-wrap: anywhere; }
.sheet-handle { display: none; }
.lbl-short { display: none; }
.intro { margin: 0 0 4px; color: var(--ink-soft); font-size: 15px; }

.step { margin-top: 14px; padding: 16px 16px 14px; background: var(--card); border: 1px solid var(--rule); border-radius: var(--radius-m); }
.step-head { display: grid; grid-template-columns: 36px 1fr; gap: 10px; align-items: center; }
.step-num { width: 34px; height: 34px; border-radius: 50%; display: grid; place-items: center; font-family: var(--display); font-weight: 800; font-size: 16px; color: #fff; background: var(--c, var(--navy)); }
#step-area { --c: var(--teal); }
#step-storm { --c: var(--blue); }
#step-inflow { --c: var(--green); }
.step.is-done .step-num { font-size: 0; }
.step.is-done .step-num::before { content: "✓"; font-size: 18px; }
.step h2 { margin: 0; font-family: var(--display); font-size: 17px; font-weight: 700; }
.step-body { margin-top: 10px; }
.hint { color: var(--ink-soft); font-size: 14px; margin: 0 0 10px; }
.optional { font-family: var(--font); font-weight: 400; color: var(--ink-soft); font-size: 14px; }

/* Buttons: every clickable thing has a filled colour so it reads as a button.
   .btn = secondary (light blue fill), .btn-primary = navy, .btn-accent = globe blue, .btn-next = teal. */
.btn { color: var(--navy); text-decoration: none; display: inline-flex; align-items: center; justify-content: center; gap: 8px;
       border: 1.5px solid #BBD3EA; background: #E8F2FB; border-radius: 10px; padding: 9px 14px; cursor: pointer; font-weight: 700;
       transition: background 0.15s ease, border-color 0.15s ease, box-shadow 0.15s ease; }
.btn:hover { background: #D6E8F7; border-color: #93BCE0; }
.btn:active { transform: translateY(1px); }
.btn .ic { width: 18px; height: 18px; flex: none; }
.btn-primary { background: var(--navy); border-color: var(--navy); color: #fff; box-shadow: 0 2px 6px rgba(11, 22, 80, 0.18); }
.btn-primary:hover { background: var(--navy-soft); border-color: var(--navy-soft); }
.btn-accent { background: var(--blue); border-color: var(--blue); color: #fff; }
.btn-accent:hover { background: #176E98; border-color: #176E98; }
.btn-next { background: #E3F5F6; border-color: #9ED9DE; color: #11606A; margin-top: 14px; }
.btn-next:hover { background: #CDEDEF; border-color: var(--teal); }
.btn-wide { width: 100%; padding: 11px 14px; }
.btn-wide + .btn-wide { margin-top: 10px; }
.btn[aria-pressed="true"] { background: var(--teal); border-color: var(--teal); color: #fff; }
.btn.is-busy { opacity: 0.75; cursor: progress; }
.btn-big { width: 100%; justify-content: center; padding: 11px 14px; }
.btn-link { border: 0; background: none; padding: 0; color: var(--blue); text-decoration: underline; text-underline-offset: 3px; cursor: pointer; font-weight: 700; }
.btn-row { display: flex; gap: 12px; align-items: center; flex-wrap: wrap; }

.readout { margin-top: 10px; font-size: 14px; }
.readout strong { font-family: var(--display); font-size: 18px; }
.readout.is-bad { color: var(--error); }

.field { margin-bottom: 14px; }
.field label, .field .label { display: block; font-weight: 700; font-size: 15px; margin-bottom: 4px; }
.field .sub { font-weight: 400; color: var(--ink-soft); font-size: 14px; }
.num-row { display: grid; grid-template-columns: 1fr 116px; gap: 10px; align-items: center; }
input[type="range"] { width: 100%; accent-color: var(--blue); }
.unit-input { display: flex; align-items: center; border: 1.5px solid var(--rule); border-radius: var(--radius-s); background: #fff; }
.unit-input input { width: 100%; border: 0; padding: 7px 4px 7px 10px; background: transparent; min-width: 0; outline: none; -moz-appearance: textfield; font-weight: 700; }
.unit-input input::-webkit-outer-spin-button, .unit-input input::-webkit-inner-spin-button { -webkit-appearance: none; margin: 0; }
.unit-input span { white-space: nowrap; padding-right: 10px; color: var(--ink-soft); font-size: 13px; }
.unit-input:focus-within { border-color: var(--blue); box-shadow: 0 0 0 2px var(--blue); }

.presets { display: grid; grid-template-columns: repeat(4, 1fr); gap: 6px; margin: 8px 0 2px; }
.presets button { border: 1.5px solid var(--rule); background: #fff; border-radius: var(--radius-s); padding: 6px 2px; cursor: pointer; font-size: 13px; line-height: 1.2; }
.presets button b { display: block; font-family: var(--display); font-size: 14px; }
.presets button:hover { border-color: var(--blue); }
.presets button.is-on { border-color: var(--blue); background: #E6F3FA; }

.seg { position: relative; display: grid; grid-template-columns: repeat(3, 1fr); border: 2px solid var(--navy); border-radius: var(--radius-s); overflow: hidden; }
.seg input { position: absolute; opacity: 0; pointer-events: none; }
.seg label { text-align: center; padding: 7px 4px; cursor: pointer; font-weight: 400; margin: 0; border-left: 2px solid var(--navy); }
.seg label:first-of-type { border-left: 0; }
.seg input:checked + label { background: var(--navy); color: #fff; font-weight: 700; }
.seg input:focus-visible + label { outline: 3px solid var(--blue); outline-offset: -3px; }
.seg-2 { grid-template-columns: 1fr 1fr; margin-bottom: 12px; }

.inflow { border: 1.5px solid var(--rule); border-radius: var(--radius-s); padding: 10px 12px; margin-bottom: 10px; background: var(--paper); }
.inflow-top { display: flex; justify-content: space-between; align-items: baseline; margin-bottom: 8px; font-size: 14px; }
.inflow-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
.inflow-grid .field { margin: 0; }

.run-bar { padding: 12px 18px calc(12px + env(safe-area-inset-bottom, 0px)); border-top: 1px solid var(--rule); background: #fff; }
.run-btn { width: 100%; border: 0; border-radius: var(--radius-s); background: var(--green); color: #fff; font-family: var(--display); font-weight: 800; font-size: 17px; padding: 13px; cursor: pointer; }
.run-btn:hover:not(:disabled) { background: #2F7F35; }
.run-btn:disabled { background: #C9D2DE; color: #F5F7FA; cursor: not-allowed; }
.run-note { margin: 8px 0 0; font-size: 14px; color: var(--ink-soft); min-height: 1.2em; }

/* ---------- status ---------- */
.status { margin-top: 14px; padding: 14px 16px; border-radius: var(--radius-m); background: #E6F3FA; border: 1px solid #BFDDEE; }
.status h2 { margin: 0 0 6px; font-family: var(--display); font-size: 16px; }
.log { list-style: none; padding: 0; margin: 8px 0 0; font-size: 14px; }
.log li { padding: 2px 0 2px 24px; position: relative; color: var(--ink-soft); }
.log li::before { content: "✓"; position: absolute; left: 2px; color: var(--green); font-weight: 700; }
.log li.is-current { color: var(--ink); font-weight: 700; }
.log li.is-current::before { content: ""; width: 11px; height: 11px; left: 3px; top: 7px; border: 2.5px solid var(--blue); border-top-color: transparent; border-radius: 50%; animation: spin 0.9s linear infinite; }
@keyframes spin { to { transform: rotate(360deg); } }
.status.is-error { background: #FBE8F0; border-color: #EDB9CF; color: var(--error); }

/* ---------- results ---------- */
.results { margin-top: 14px; }
.verdict { --lv: var(--green); border-radius: var(--radius-m); overflow: hidden; background: var(--card); border: 1px solid var(--rule); }
.verdict[data-level="1"] { --lv: var(--amber); }
.verdict[data-level="2"] { --lv: var(--magenta); }
.verdict[data-level="3"] { --lv: #7E1046; }
.verdict-top { background: var(--lv); color: #fff; padding: 14px 16px 12px; }
.verdict-kicker { font-size: 14px; opacity: 0.92; margin: 0; }
.verdict-title { font-family: var(--display); font-weight: 800; font-size: 26px; line-height: 1.12; margin: 2px 0 0; letter-spacing: -0.01em; }
.meter { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; margin-top: 12px; }
.meter span { height: 8px; border-radius: 4px; background: rgba(255,255,255,0.35); }
.meter span.is-on { background: #fff; }
.meter-labels { display: grid; grid-template-columns: repeat(4, 1fr); gap: 4px; font-size: 12px; margin-top: 3px; opacity: 0.9; }
.verdict-body { padding: 12px 16px 14px; }
.verdict-body p { margin: 0 0 8px; }
.verdict-body h3 { font-family: var(--display); font-size: 15px; margin: 10px 0 6px; }
.todo { list-style: none; margin: 0; padding: 0; }
.todo li { position: relative; padding: 3px 0 3px 22px; font-size: 15px; }
.todo li::before { content: ""; position: absolute; left: 4px; top: 11px; width: 8px; height: 8px; border-radius: 50%; background: var(--lv); }
.verdict-note { font-size: 13px; color: var(--ink-soft); margin: 10px 0 0 !important; }

.headline-block { margin-top: 14px; padding: 14px 16px; background: var(--card); border: 1px solid var(--rule); border-radius: var(--radius-m); }
.headline { font-family: var(--display); font-size: 30px; font-weight: 800; line-height: 1.05; letter-spacing: -0.02em; color: var(--blue); }
.headline-sub { margin: 6px 0 12px; color: var(--ink-soft); font-size: 15px; }
.facts { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin: 0; }
.facts div { background: var(--paper); border-radius: var(--radius-s); padding: 9px 11px; border-left: 4px solid var(--blue); }
.facts div:nth-child(2) { border-left-color: var(--teal); }
.facts div:nth-child(3) { border-left-color: var(--navy); }
.facts div:nth-child(4) { border-left-color: var(--green); }
.facts dt { font-size: 13px; color: var(--ink-soft); }
.facts dd { margin: 2px 0 0; font-family: var(--display); font-weight: 700; font-size: 19px; }

.card { margin-top: 14px; padding: 14px 16px; background: var(--card); border: 1px solid var(--rule); border-radius: var(--radius-m); }
.card h3, .card summary { margin: 0 0 8px; font-family: var(--display); font-size: 16px; font-weight: 700; }
.card summary { cursor: pointer; margin: 0; }
details.card[open] summary { margin-bottom: 8px; }
.band-list { list-style: none; margin: 0; padding: 0; font-size: 14px; }
.band-list li { display: grid; grid-template-columns: 84px 1fr 56px; gap: 8px; align-items: center; padding: 3px 0; }
.band-bar { height: 12px; border-radius: 3px; }
.band-list .n { text-align: right; font-weight: 700; }
.fac-list { list-style: none; margin: 8px 0 0; padding: 0; font-size: 14px; }
.fac-list button { width: 100%; text-align: left; background: var(--paper); border: 1.5px solid var(--rule); border-radius: var(--radius-s); padding: 7px 9px; margin-bottom: 6px; cursor: pointer; display: flex; justify-content: space-between; gap: 8px; }
.fac-list button:hover { border-color: var(--navy); }
.profile svg { width: 100%; height: auto; display: block; background: #fff; border: 1px solid var(--rule); border-radius: var(--radius-s); margin-top: 10px; touch-action: none; }
.profile-readout { font-size: 13px; color: var(--ink-soft); min-height: 1.3em; margin-top: 4px; }
.notes { font-size: 13px; color: var(--ink-soft); margin: 12px 0 0; padding-left: 18px; }
.results-actions { display: flex; gap: 12px; flex-wrap: wrap; margin-top: 14px; align-items: center; }
.disclaimer { font-size: 13px; color: var(--ink-soft); margin: 14px 0 8px; }
.real-grid { display: grid; grid-template-columns: 1fr 110px; gap: 10px; }
.real-weather { border: 1.5px solid #BFDDEE; border-radius: var(--radius-s); padding: 10px 12px; margin-bottom: 14px; background: #F1F8FC; }
.real-grid select, .real-grid input[type="date"], .cmp-side select, .cmp-side input[type="date"] { border: 1.5px solid var(--rule); border-radius: var(--radius-s); padding: 6px 7px; background: #fff; width: 100%; }
.is-error-text { color: var(--error) !important; }

/* ---------- map overlays ---------- */
.hint-action { margin-left: 10px; border: 2px solid #fff; background: #fff; color: var(--navy); border-radius: var(--radius-s); padding: 3px 10px; font-weight: 700; cursor: pointer; pointer-events: auto; }
.map-hint:has(.hint-action) { pointer-events: auto; }
.regions-note { margin-top: 6px; border-left: 4px solid var(--teal); padding-left: 8px; }
.map-hint { position: absolute; top: 60px; left: 50%; transform: translateX(-50%); background: var(--navy); color: #fff; padding: 10px 16px; border-radius: var(--radius-m); font-size: 15px; font-weight: 700; z-index: 4; pointer-events: none; max-width: calc(100% - 24px); text-align: center; box-shadow: 0 4px 14px rgba(11,22,80,0.25); }
.legend { position: absolute; right: 12px; bottom: 34px; z-index: 2; background: #fff; border: 1px solid var(--rule); border-radius: var(--radius-m); padding: 10px 12px; font-size: 13px; width: 210px; }
.legend-title { font-family: var(--display); font-weight: 700; margin-bottom: 6px; }
.legend-ramp { height: 12px; border-radius: 3px; background: linear-gradient(90deg, #BDE4F4 0%, #6BAED6 16%, #3182BD 31%, #08519C 65%, #08306B 100%); }
.legend-scale { display: flex; justify-content: space-between; color: var(--ink-soft); margin-top: 3px; }
.legend-words { display: flex; justify-content: space-between; color: var(--ink-soft); font-size: 12px; }
.legend-water { display: flex; align-items: center; gap: 6px; margin-top: 8px; }
.legend-water i { width: 14px; height: 10px; background: rgba(90,110,130,0.7); display: inline-block; border-radius: 2px; }
.legend label { display: block; margin-top: 8px; }
.maplibregl-popup-content { font-family: var(--font); font-size: 14px; padding: 10px 12px; border-radius: 8px; }

/* map buttons and the layers card stack in one column, so they never overlap */
.map-overlay-left { position: absolute; top: 12px; left: 12px; z-index: 3; display: flex; flex-direction: column; align-items: flex-start; gap: 8px;
                    max-width: calc(100% - 70px); max-height: calc(100% - 24px); pointer-events: none; }
.map-overlay-left > * { pointer-events: auto; }
.map-tools { display: flex; gap: 8px; flex-wrap: wrap; }
.tool-group { display: flex; border: 2px solid var(--navy); border-radius: var(--radius-s); overflow: hidden; background: #E8F2FB; box-shadow: 0 2px 6px rgba(11,22,80,0.12); }
.tool-group button { border: 0; border-left: 2px solid var(--navy); background: transparent; padding: 7px 12px; cursor: pointer; font-weight: 700; font-size: 14px; }
.tool-group button:first-child { border-left: 0; }
.tool-group button[aria-pressed="true"] { background: var(--navy); color: #fff; }

.layers-card { background: #fff; border: 1px solid var(--rule); border-radius: var(--radius-m); padding: 10px 12px; width: 270px; max-width: 100%; font-size: 14px;
               max-height: min(60vh, calc(100% - 60px)); overflow-y: auto; }
.layers-card summary { font-family: var(--display); font-weight: 700; cursor: pointer; }
.layers-card[open] summary { margin-bottom: 6px; }
.layers-card:not([open]) { width: auto; }
.layer-row { display: grid; grid-template-columns: 18px 20px 1fr auto; gap: 8px; align-items: center; padding: 4px 0; cursor: pointer; }
.layer-row input { margin: 0; accent-color: var(--navy); }
.layer-row .count { color: var(--ink-soft); font-size: 13px; }
.sw { width: 18px; height: 12px; border-radius: 2px; display: inline-block; }
.sw-line { height: 4px; border-radius: 2px; }
.sw-dot { width: 12px; height: 12px; border-radius: 50%; margin-left: 3px; }

#map-after { position: absolute; inset: 0; z-index: 1; pointer-events: none; }
.swipe { position: absolute; top: 0; bottom: 0; width: 0; z-index: 4; }
.swipe::before { content: ""; position: absolute; top: 0; bottom: 0; left: -1.5px; width: 3px; background: #fff; box-shadow: 0 0 0 1px var(--navy); }
.swipe button { position: absolute; top: 50%; left: -20px; width: 40px; height: 40px; margin-top: -20px; border-radius: 50%; border: 2px solid var(--navy); background: #fff; cursor: ew-resize; font-weight: 700; touch-action: none; }
.compare-bar { position: absolute; bottom: 34px; left: 50%; transform: translateX(-50%); z-index: 5; background: #fff; border: 1px solid var(--rule); border-radius: var(--radius-m); padding: 10px 12px; display: grid; grid-template-columns: 1fr 1fr; gap: 8px 16px; font-size: 14px; width: min(560px, calc(100% - 260px)); }
.cmp-side { display: grid; gap: 4px; align-content: start; }
.cmp-side > label { font-weight: 700; }
.cmp-status { margin: 0; color: var(--ink-soft); font-size: 13px; min-height: 2.6em; }
.cmp-opts { grid-column: 1 / -1; display: flex; gap: 16px; flex-wrap: wrap; }
.compare-bar .check { font-weight: 400; display: flex; gap: 6px; align-items: center; }
.cmp-note { grid-column: 1 / -1; margin: 0; color: var(--ink-soft); font-size: 12px; }

/* ---------- place search ---------- */
.search-label { display: block; font-weight: 700; font-size: 15px; margin-bottom: 4px; }
.search-row { display: grid; grid-template-columns: 1fr auto; gap: 8px; }
.search-row input { min-width: 0; border: 1.5px solid var(--rule); border-radius: var(--radius-s); padding: 9px 10px; background: #fff; }
.search-row input:focus { outline: none; border-color: var(--blue); box-shadow: 0 0 0 2px var(--blue); }
.btn-search { background: var(--navy); border-color: var(--navy); color: #fff; padding: 9px 12px; }
.btn-search:hover { background: var(--navy-soft); border-color: var(--navy-soft); }
#place-help { margin: 6px 0 0; font-size: 13px; }
.place-results { list-style: none; margin: 8px 0 0; padding: 0; }
.place-results:empty { display: none; }
.place-results li + li { margin-top: 6px; }
.place-results button { width: 100%; text-align: left; background: var(--paper); border: 1.5px solid var(--rule); border-radius: var(--radius-s); padding: 8px 10px; cursor: pointer; display: grid; gap: 1px; }
.place-results button:hover, .place-results button:focus-visible { border-color: var(--teal); background: #EAF7F8; }
.place-results .pr-name { font-weight: 700; }
.place-results .pr-label { font-size: 13px; color: var(--ink-soft); overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.place-results .pr-msg { font-size: 14px; color: var(--ink-soft); padding: 4px 0; }
.place-results .pr-msg.is-error-text { color: var(--error); }
.place-results .pr-attr { font-size: 11px; color: var(--ink-soft); }
.search-or { margin: 12px 0 8px; }
.place-pin { width: 22px; height: 22px; border-radius: 50% 50% 50% 0; transform: rotate(-45deg); background: var(--magenta); border: 3px solid #fff; box-shadow: 0 0 0 1.5px var(--navy); }

/* ---------- tabs and sections ---------- */
.tabs { display: grid; grid-template-columns: repeat(4, 1fr); flex: none; background: #fff; border-bottom: 1px solid var(--rule); }
.tab { position: relative; border: 0; background: none; padding: 13px 4px 11px; font: inherit; font-size: 15px; font-weight: 700;
       color: var(--ink-soft); cursor: pointer; display: inline-flex; justify-content: center; align-items: center; gap: 6px; }
.tab:hover { color: var(--navy); background: #F5F8FB; }
.tab[aria-selected="true"] { color: var(--navy); }
.tab[aria-selected="true"]::after { content: ""; position: absolute; left: 10%; right: 10%; bottom: -1px; height: 3px; border-radius: 3px 3px 0 0; background: var(--magenta); }
.tab.is-done::before { content: "✓"; color: var(--green); font-size: 13px; }
.tab-badge { min-width: 19px; height: 19px; border-radius: 10px; background: var(--magenta); color: #fff; font-size: 12px; line-height: 19px; padding: 0 5px; }
.pane { padding-bottom: 6px; }
.sec { margin-top: 12px; padding: 16px; background: var(--card); border: 1px solid var(--rule); border-radius: 14px; box-shadow: 0 1px 3px rgba(11, 22, 80, 0.05); }
.pane > .sec:first-child, .pane > .status:first-child { margin-top: 0; }
.sec-title { margin: 0; font-family: var(--display); font-size: 17px; font-weight: 700; }
.sec-sub { margin: 4px 0 14px; color: var(--ink-soft); font-size: 14px; }
.sec-label { display: block; margin: 16px 0 6px; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: var(--ink-soft); }
.field .sec-label { margin-top: 0; }
.sec.empty { text-align: center; padding: 26px 18px; border-style: dashed; box-shadow: none; }
.sec.empty .sec-sub { margin-bottom: 0; }
.search-row .btn-search span { display: inline; }

/* the area box, shown like a coordinate card */
.coord-card { margin-top: 14px; padding: 12px 14px; border-radius: 10px; background: #EAF3FB; border: 1px solid #C9DDF0; font-size: 14px; }
.coord-head { display: flex; justify-content: space-between; align-items: center; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: var(--blue); }
.coord-head .btn-link { font-size: 13px; letter-spacing: 0; text-transform: none; text-decoration: none; }
.coord-empty { margin: 6px 0 0; color: var(--ink-soft); }
.coord-place { margin: 6px 0 0; font-weight: 700; }
.coord-val { margin: 6px 0 2px; font-family: ui-monospace, "SFMono-Regular", Consolas, monospace; font-size: 13.5px; font-weight: 700; color: var(--ink); }
.coord-sep { display: inline-block; width: 30px; margin: 0; font-family: var(--font); font-weight: 400; color: var(--ink-soft); }
.coord-size { margin: 0; color: var(--ink-soft); font-size: 13px; }
.coord-size strong { color: var(--ink); font-family: var(--display); }
.coord-prob { margin: 6px 0 0; color: var(--error); font-weight: 700; }
.coord-card.is-bad { background: #FBE8F0; border-color: #EDB9CF; }

[hidden] { display: none !important; }

/* ---------- tablets and small laptops ---------- */
@media (max-width: 1100px) {
  .app-name small { font-size: 13px; }
  .tool-group button { padding: 6px 10px; font-size: 13px; }
  .compare-bar { width: calc(100% - 250px); }
}
@media (max-width: 900px) and (min-width: 761px) {
  .brand-divider, .app-name small { display: none; }
  .legend { width: 180px; }
  .compare-bar { left: 12px; transform: none; width: calc(100% - 220px); grid-template-columns: 1fr; }
}
/* short screens (landscape phones, small laptop windows) */
@media (max-height: 640px) and (min-width: 761px) {
  .brandbar { padding-top: 4px; padding-bottom: 8px; }
  .brandbar img { height: 38px; }
  .legend-words, .legend-water { display: none; }
}

/* ---------- phones: full-screen map with the panel as a bottom sheet ---------- */
@media (max-width: 760px) {
  .app { display: block; position: relative; --hdr: 54px; --sheet-h: 52dvh; }
  .brandbar { gap: 10px; padding: 6px 12px 9px; padding-top: calc(6px + env(safe-area-inset-top, 0px)); }
  .brandbar img { height: 34px; }
  .app-name { font-size: 18px; }
  .app-name small, .brand-divider { display: none; }
  #map { position: absolute; top: var(--hdr); left: 0; right: 0; bottom: 0; height: auto; }

  .panel { position: absolute; left: 0; right: 0; bottom: 0; z-index: 6; height: var(--sheet-h); max-height: calc(100% - var(--hdr));
           border-right: 0; border-radius: 16px 16px 0 0; box-shadow: 0 -6px 24px rgba(11, 22, 80, 0.18);
           transition: height 0.25s ease; }
  .panel.is-dragging { transition: none; }
  .panel[data-sheet="peek"] .panel-scroll { visibility: hidden; }
  .sheet-handle { display: flex; flex-direction: column; align-items: center; gap: 4px; width: 100%; flex: none;
                  border: 0; background: #fff; padding: 8px 0 6px; cursor: grab; touch-action: none;
                  font-size: 13px; font-weight: 700; color: var(--ink-soft); border-bottom: 1px solid var(--rule); }
  .sheet-handle .grabber { width: 44px; height: 5px; border-radius: 3px; background: #C9D2DE; }
  .panel-scroll { padding: 12px 14px 8px; }
  .intro { font-size: 14px; }
  .step { padding: 14px 14px 12px; }
  .run-bar { padding: 10px 14px calc(10px + env(safe-area-inset-bottom, 0px)); }
  .run-btn { padding: 11px; }
  .run-note { margin-top: 6px; font-size: 13px; }

  /* map overlays sit above the sheet and stay small */
  .map-overlay-left { top: 8px; left: 8px; max-width: calc(100% - 60px); max-height: calc(100% - var(--sheet-h) - 16px); }
  .map-tools { flex-wrap: nowrap; overflow-x: auto; max-width: 100%; scrollbar-width: none; padding-bottom: 2px; }
  .map-tools::-webkit-scrollbar { display: none; }
  .tool-group { flex: none; }
  .lbl-long { display: none; } .lbl-short { display: inline; }
  .tool-group button { padding: 6px 9px; font-size: 13px; white-space: nowrap; }
  .layers-card { width: 230px; font-size: 13px; max-height: none; }
  .map-hint { top: 52px; font-size: 14px; padding: 8px 12px; }
  .legend { width: 150px; padding: 8px 10px; bottom: calc(var(--sheet-h) + 38px); right: 8px; transition: bottom 0.25s ease; }
  .legend-water, .legend label, .legend-words { display: none; }
  .compare-bar { width: calc(100% - 16px); left: 8px; transform: none; bottom: calc(var(--sheet-h) + 10px); gap: 6px 10px;
                 grid-template-columns: 1fr 1fr; padding: 8px 10px; font-size: 13px; }
  .cmp-note, .cmp-status { display: none; }
  .maplibregl-ctrl-bottom-left, .maplibregl-ctrl-bottom-right { bottom: var(--sheet-h); transition: bottom 0.25s ease; }
  .maplibregl-ctrl-top-right { top: 0; }
  .presets { grid-template-columns: repeat(2, 1fr); }
  .tab { font-size: 14px; padding: 10px 2px 9px; }
  .panel[data-sheet="peek"] .tabs { display: none; }
  .sec { padding: 14px; }
  .facts dd { font-size: 17px; }
  .headline { font-size: 26px; }
  .verdict-title { font-size: 22px; }
  .real-grid { grid-template-columns: 1fr 96px; }
}
@media (max-width: 760px) and (prefers-reduced-motion: reduce) { .panel, .legend { transition: none; } }
@media (prefers-reduced-motion: reduce) { .log li.is-current::before { animation: none; } }


In [ ]:
%%writefile /content/floodcheck/backend/static/js/floodcheck.js
// FloodCheck dashboard: map, inputs, model runs and results.
// Talks to the API in backend/server.py.
(() => {
  const KENYA = [33.8, -4.8, 42.0, 5.1];
  let MAX_SIDE = 0.40;
  let REGIONS = [], LIVE_TERRAIN = true;          // prepared terrain regions from the server
  const PREP_BUFFER = 0.03;                       // must match prepared_buffer_deg in floodmodel.py
  const EXAMPLE_BOX = [39.52, -0.60, 39.80, -0.32];   // demo area
  const TERRAIN_TILES = "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png";
  const gibsTiles = (date) => [`https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/MODIS_Terra_CorrectedReflectance_TrueColor/default/${date}/GoogleMapsCompatible_Level9/{z}/{y}/{x}.jpg`];
  const GIBS_ATTR = "Imagery: NASA GIBS / MODIS Terra";

  // ---------- layer configuration (one entry per result layer, like a layers.json) ----------
  const DEPTH_COLOR = ["interpolate", ["linear"], ["get", "depth"], 0.1, "#F2C14E", 1, "#E07A1F", 2, "#B3261E"];
  const LAYERS = [
    { id: "flood", label: "Flood water", swatch: "linear-gradient(90deg,#BDE4F4,#08306B)", kind: "image" },
    { id: "buildings", label: "Homes and buildings in water", swatch: "#E07A1F", kind: "fill",
      paint: { "fill-color": DEPTH_COLOR, "fill-opacity": 0.95, "fill-outline-color": "#5A2A0A" },
      filter: ["==", "$type", "Polygon"], minzoom: 14,
      extra: { suffix: "pts", type: "circle", filter: ["==", "$type", "Point"], maxzoom: 14,
               paint: { "circle-color": DEPTH_COLOR, "circle-radius": ["interpolate", ["linear"], ["zoom"], 9, 1.5, 14, 4],
                        "circle-stroke-color": "#5A2A0A", "circle-stroke-width": 0.5 } },
      count: (e) => e.buildings_flooded ? `${fmt(e.buildings_flooded.any, 0)}` : "–",
      popup: (p) => `<strong>Building</strong><br>About ${fmt(p.depth, 2)} m of water` },
    { id: "roads", label: "Roads under water", swatch: "#C4307A", swatchClass: "sw-line", kind: "line",
      paint: { "line-color": "#C4307A", "line-width": ["interpolate", ["linear"], ["zoom"], 9, 2, 15, 6] },
      count: (e) => e.roads_flooded_km != null ? `${fmt(e.roads_flooded_km)} km` : "–",
      popup: (p) => `<strong>${esc(p.name) || "Road"}</strong> (${esc(p.class)})<br>Up to ${fmt(p.max_depth, 2)} m of water` },
    { id: "facilities", label: "Schools and health", swatch: "#1E86B8", swatchClass: "sw-dot", kind: "circle",
      paint: { "circle-radius": ["case", ["get", "flooded"], 8, 6],
               "circle-color": ["match", ["get", "kind"], "health", "#3E9A44", "#1E86B8"],
               "circle-stroke-color": ["case", ["get", "flooded"], "#B3261E", "#FFFFFF"],
               "circle-stroke-width": ["case", ["get", "flooded"], 3, 1.5] },
      count: (e) => e.facilities_flooded ? `${e.facilities_flooded.length} in water` : "–",
      popup: (p) => `<strong>${esc(p.name)}</strong><br>${p.kind === "health" ? "Health facility" : "School"} · ` +
                    (p.flooded ? `about ${fmt(p.depth, 2)} m of water` : "not flooded") + `<br><span style="color:#4A5578">${esc(p.source)}</span>` },
  ];

  const $ = (id) => document.getElementById(id);
  const esc = (s) => String(s ?? "").replace(/[&<>"']/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c]));
  const phoneQuery = matchMedia("(max-width: 760px)");
  const isPhone = () => phoneQuery.matches;
  const sheet = { name: "half" };
  const state = { bbox: null, mode: null, firstCorner: null, inflows: [], runId: null, polling: null,
                  result: null, layerOn: {}, profilePts: [], terrain: false, compare: false };

  // ---------- street map providers ----------
  // Street maps that work without an account. Not used: tile.openstreetmap.org (blocks apps, 403
  // "Access blocked") and CARTO (now draws "API KEY REQUIRED" over every tile without a key).
  // If one provider fails, the next is used automatically. A server can put its own provider
  // first with FLOOD_STREET_TILES (see README).
  const ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services";
  const STREET_MAPS = [
    { id: "esri-streets", tiles: [`${ESRI}/World_Street_Map/MapServer/tile/{z}/{y}/{x}`],
      maxzoom: 19, attribution: "Map © Esri, HERE, Garmin, © OpenStreetMap contributors" },
    { id: "esri-topo", tiles: [`${ESRI}/World_Topo_Map/MapServer/tile/{z}/{y}/{x}`],
      maxzoom: 19, attribution: "Map © Esri, HERE, Garmin, USGS, © OpenStreetMap contributors" },
  ];
  let streetIdx = 0, streetErrors = 0;
  const streetSource = (s) => ({ type: "raster", tiles: s.tiles, tileSize: 256, maxzoom: s.maxzoom, attribution: s.attribution });

  // ---------- map ----------
  const baseStyle = (extra = {}) => ({
    version: 8,
    sources: {
      streets: streetSource(STREET_MAPS[0]),
      sat: { type: "raster", tiles: ["https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"],
             tileSize: 256, maxzoom: 18, attribution: "Imagery © Esri, Maxar, Earthstar Geographics" },
      ...extra,
    },
    layers: [
      { id: "streets", type: "raster", source: "streets" },
      { id: "sat", type: "raster", source: "sat", layout: { visibility: "none" } },
    ],
  });
  const map = new maplibregl.Map({ container: "map", center: [38.2, 0.2], zoom: 5.6, maxBounds: [[30, -9], [46, 9]],
    maxPitch: 75, attributionControl: false, style: baseStyle({ dem: { type: "raster-dem", tiles: [TERRAIN_TILES], encoding: "terrarium", tileSize: 256, maxzoom: 14 } }) });
  map.addControl(new maplibregl.NavigationControl({ showCompass: true, visualizePitch: true }), "top-right");
  map.addControl(new maplibregl.ScaleControl({ unit: "metric" }), "bottom-left");
  map.addControl(new maplibregl.AttributionControl({ compact: matchMedia("(max-width: 760px)").matches }), "bottom-right");
  map.once("load", () => setTimeout(() => {          // start the phone credit line collapsed (tap (i) to read it)
    document.querySelectorAll(".maplibregl-ctrl-attrib.maplibregl-compact").forEach(el => {
      el.classList.remove("maplibregl-compact-show"); el.removeAttribute("open");
    });
  }, 0));

  const emptyFC = { type: "FeatureCollection", features: [] };
  const whenReady = (fn) => (map.getSource("box") ? fn() : map.once("load", fn));
  map.on("load", () => {
    map.addSource("box", { type: "geojson", data: emptyFC });
    map.addLayer({ id: "box-fill", type: "fill", source: "box", paint: { "fill-color": "#C4307A", "fill-opacity": 0.06 } });
    map.addLayer({ id: "box-line", type: "line", source: "box", paint: { "line-color": "#C4307A", "line-width": 3, "line-dasharray": [2, 1.5] } });
    map.addSource("profile", { type: "geojson", data: emptyFC });
    map.addLayer({ id: "profile-line", type: "line", source: "profile", paint: { "line-color": "#0B1650", "line-width": 3, "line-dasharray": [1, 1] } });
    map.addLayer({ id: "profile-pts", type: "circle", source: "profile", filter: ["==", "$type", "Point"],
                   paint: { "circle-radius": 5, "circle-color": "#fff", "circle-stroke-color": "#0B1650", "circle-stroke-width": 2 } });
    fetch("api/config").then(r => r.ok ? r.json() : null).then(c => {
      if (!c) return;
      MAX_SIDE = c.max_side_deg;
      $("max-km").textContent = fmt(MAX_SIDE * 111, 0);
      REGIONS = c.prepared_regions || []; LIVE_TERRAIN = c.live_terrain !== false;
      if (REGIONS.length) {
        map.addSource("regions", { type: "geojson", data: { type: "FeatureCollection",
          features: REGIONS.map(r => ({ ...boxFeature(r.bbox).features[0], properties: { name: r.name } })) } });
        map.addLayer({ id: "regions-line", type: "line", source: "regions",
          paint: { "line-color": "#4BB8C2", "line-width": 2, "line-dasharray": [3, 2], "line-opacity": 0.9 } }, "box-fill");
        if (!LIVE_TERRAIN && !state.bbox) {
          const u = REGIONS.reduce((a, r) => [Math.min(a[0], r.bbox[0]), Math.min(a[1], r.bbox[1]), Math.max(a[2], r.bbox[2]), Math.max(a[3], r.bbox[3])], [180, 90, -180, -90]);
          fitBox(u);
        }
        $("regions-note").hidden = false;
      }
      if (state.bbox) setBox(state.bbox);
      if (c.street_map && c.street_map.tiles && c.street_map.tiles.length) {   // server-chosen provider goes first
        STREET_MAPS.unshift({ id: "server", maxzoom: 19, ...c.street_map });
        useStreetMap(0);
      }
    }).catch(() => {});
  });

  function useStreetMap(i) {
    streetIdx = i; streetErrors = 0;
    const visible = map.getLayer("streets") ? map.getLayoutProperty("streets", "visibility") : "visible";
    if (map.getLayer("streets")) map.removeLayer("streets");
    if (map.getSource("streets")) map.removeSource("streets");
    map.addSource("streets", streetSource(STREET_MAPS[i]));
    map.addLayer({ id: "streets", type: "raster", source: "streets", layout: { visibility: visible || "visible" } }, "sat");
  }
  // switch to the next provider if this one keeps failing (blocked, down or over its limit)
  map.on("error", (e) => {
    if (e.sourceId !== "streets") return;
    if (++streetErrors >= 4 && streetIdx < STREET_MAPS.length - 1) {
      console.warn(`Street map "${STREET_MAPS[streetIdx].id}" is failing; switching to "${STREET_MAPS[streetIdx + 1].id}".`);
      useStreetMap(streetIdx + 1);
    }
  });

  function setBasemap(sat) {
    map.setLayoutProperty("streets", "visibility", sat ? "none" : "visible");
    map.setLayoutProperty("sat", "visibility", sat ? "visible" : "none");
    $("bm-map").setAttribute("aria-pressed", String(!sat));
    $("bm-sat").setAttribute("aria-pressed", String(sat));
  }
  $("bm-map").onclick = () => setBasemap(false);
  $("bm-sat").onclick = () => setBasemap(true);

  // ---------- 3D terrain ----------
  function setTerrain(on) {
    state.terrain = on;
    $("terrain-btn").setAttribute("aria-pressed", String(on));
    const t = on ? { source: "dem", exaggeration: 2 } : null;
    map.setTerrain(t);
    if (after && afterReady) after.setTerrain(t);
    map.easeTo(on ? { pitch: 60, duration: reduced() ? 0 : 900 } : { pitch: 0, bearing: 0, duration: reduced() ? 0 : 600 });
  }
  $("terrain-btn").onclick = () => setTerrain(!state.terrain);
  const reduced = () => matchMedia("(prefers-reduced-motion: reduce)").matches;

  // ---------- helpers ----------
  function sideKm(b) {
    const midLat = (b[1] + b[3]) / 2;
    return [(b[2] - b[0]) * 111.32 * Math.cos(midLat * Math.PI / 180), (b[3] - b[1]) * 110.57];
  }
  function boxFeature(b) {
    return { type: "FeatureCollection", features: [{ type: "Feature", properties: {}, geometry: { type: "Polygon",
      coordinates: [[[b[0], b[1]], [b[2], b[1]], [b[2], b[3]], [b[0], b[3]], [b[0], b[1]]]] } }] };
  }
  const normBox = (a, b) => [Math.min(a[0], b[0]), Math.min(a[1], b[1]), Math.max(a[0], b[0]), Math.max(a[1], b[1])];
  function boxProblem(b) {
    if (b[2] - b[0] > MAX_SIDE || b[3] - b[1] > MAX_SIDE) return `Too big: keep each side under about ${fmt(MAX_SIDE * 111, 0)} km.`;
    if (b[0] < KENYA[0] || b[2] > KENYA[2] || b[1] < KENYA[1] || b[3] > KENYA[3]) return "Outside the area the model covers: draw it inside the country.";
    if (!LIVE_TERRAIN && REGIONS.length && !inPreparedRegion(b)) return "Outside the prepared areas: draw your area inside one of the teal dashed outlines.";
    return null;
  }
  const inPreparedRegion = (b) => REGIONS.some(r => b[0] - PREP_BUFFER >= r.bbox[0] && b[1] - PREP_BUFFER >= r.bbox[1] &&
                                                   b[2] + PREP_BUFFER <= r.bbox[2] && b[3] + PREP_BUFFER <= r.bbox[3]);
  const inBox = (lng, lat, b) => b && lng >= b[0] && lng <= b[2] && lat >= b[1] && lat <= b[3];
  const fmt = (n, d = 1) => Number(n).toLocaleString("en-GB", { maximumFractionDigits: d, minimumFractionDigits: 0 });

  function setHint(text, action) {
    const h = $("map-hint"); h.textContent = text || ""; h.hidden = !text;
    if (text && action) {
      const b = document.createElement("button"); b.type = "button"; b.className = "hint-action"; b.textContent = action.label;
      b.onclick = action.run; h.appendChild(b);
    }
  }
  function setMode(mode) {
    if (state.mode === "profile" && mode !== "profile") setTimeout(() => map.doubleClickZoom.enable(), 400);
    state.mode = mode; state.firstCorner = null;
    $("draw-btn").setAttribute("aria-pressed", String(mode === "draw"));
    $("inflow-btn").setAttribute("aria-pressed", String(mode === "inflow"));
    $("profile-btn").setAttribute("aria-pressed", String(mode === "profile"));
    map.getCanvas().style.cursor = mode ? "crosshair" : "";
    setHint(mode === "draw" ? "Click the first corner of your area" :
            mode === "inflow" ? "Click the point where the river enters your area" :
            mode === "profile" ? "Tap points along your line, then press Finish." : "");
    if (mode === "profile") { map.doubleClickZoom.disable(); state.profilePts = []; drawProfileLine(); }
    if (mode) sheetTo("peek");                       // on phones, get the panel out of the way of the map
  }
  function finishProfile() {
    const pts = state.profilePts.slice(); setMode(null);
    state.profilePts = pts; drawProfileLine();
    if (pts.length >= 2) runProfile(pts);
    sheetTo("half");
  }

  function setBox(b, placeName) {
    const changed = JSON.stringify(b) !== JSON.stringify(state.bbox);
    state.bbox = b;
    whenReady(() => map.getSource("box").setData(b ? boxFeature(b) : emptyFC));
    const r = $("area-readout");
    if (!b) {
      r.innerHTML = `<div class="coord-head"><span>Your area (WGS84)</span></div><p class="coord-empty">No area marked yet.</p>`;
      r.classList.remove("is-bad", "is-set");
      $("next-rain").hidden = true;
    } else {
      const [wk, hk] = sideKm(b); const prob = boxProblem(b);
      const c = (v) => v.toFixed(5) + "°";
      r.innerHTML = `<div class="coord-head"><span>Your area (WGS84)</span><button type="button" class="btn-link" id="area-clear">Clear</button></div>
        ${placeName ? `<p class="coord-place">Around ${esc(placeName)}</p>` : ""}
        <p class="coord-val"><span class="coord-sep">NW</span>${c(b[3])}, ${c(b[0])}</p>
        <p class="coord-val"><span class="coord-sep">SE</span>${c(b[1])}, ${c(b[2])}</p>
        <p class="coord-size"><strong>${fmt(wk, 0)} × ${fmt(hk, 0)} km</strong> · latitude, longitude of the corners</p>
        ${prob ? `<p class="coord-prob">${esc(prob)}</p>` : ""}`;
      r.classList.toggle("is-bad", !!prob); r.classList.add("is-set");
      $("area-clear").onclick = () => { setMode(null); if (placeMarker) { placeMarker.remove(); placeMarker = null; } setBox(null); };
      $("step-area").classList.toggle("is-done", !prob);
      $("next-rain").hidden = !!prob;
    }
    if (changed) onAreaChanged();
    updateRunButton();
  }

  // a new area means the old results, report and rainfall no longer apply
  function onAreaChanged() {
    if (state.result) clearResults("Your area changed. Press <strong>Check for floods</strong> to see results for the new area.");
    if (state.weather) {
      state.weather = null;
      if (state.bbox && !boxProblem(state.bbox)) {                 // fetch the rainfall again for the new area
        $("wx-result").innerHTML = `<p class="hint">Getting rainfall for the new area…</p>`;
        setTimeout(() => $("wx-btn").click(), 0);
      } else {
        $("wx-result").innerHTML = `<p class="hint">The area changed. Press <strong>Get rainfall</strong> again for the new area.</p>`;
      }
    }
  }
  function clearResults(message) {
    if (state.polling) return;                                     // a check is running: let it finish
    state.result = null;
    whenReady(() => removeResultLayers(map));
    if (typeof after !== "undefined" && after) removeResultLayers(after);
    $("results").hidden = true; $("status").hidden = true;
    $("legend").hidden = true; $("layers-card").hidden = true;
    $("results-badge").hidden = true;
    $("results-empty").hidden = false; $("results-empty-text").innerHTML = message;
    $("report-ready").hidden = true; $("report-empty").hidden = false; $("report-empty-text").innerHTML = message;
    $("tab-results").classList.remove("is-done");
  }

  function updateRunButton() {
    const btn = $("run-btn"), note = $("run-note");
    const running = !!state.polling;
    let msg = "";
    const realMode = document.querySelector('input[name="rainsrc"]:checked').value === "real";
    if (!state.bbox) msg = "Mark your area to start.";
    else if (boxProblem(state.bbox)) msg = boxProblem(state.bbox);
    else if (state.inflows.some(f => !(f.flood > 0))) msg = "Enter the flood flow for each river inflow, or remove it.";
    else if (state.inflows.some(f => !inBox(f.lng, f.lat, state.bbox))) msg = "A river inflow point is outside your area. Move it inside.";
    else if (realMode && !state.weather) msg = "Press “Get rainfall” first, or switch to “Choose myself”.";
    else if (!(+$("rain-mm").value > 0) || !(+$("rain-hours").value > 0)) msg = "Enter the rain amount and how long it falls.";
    btn.disabled = !!msg || running;
    $("tab-area").classList.toggle("is-done", !!state.bbox && !boxProblem(state.bbox));
    $("tab-rain").classList.toggle("is-done", realMode ? !!state.weather : (+$("rain-mm").value > 0 && +$("rain-hours").value > 0));
    note.textContent = running ? "Checking… this can take a few minutes." : (msg || "Ready. A check takes from a few seconds to a few minutes, depending on the area.");
  }

  // ---------- area drawing ----------
  $("draw-btn").onclick = () => setMode(state.mode === "draw" ? null : "draw");
  $("example-btn").onclick = () => {
    setMode(null);
    let ex = EXAMPLE_BOX;
    if (REGIONS.length && !inPreparedRegion(ex)) {         // use the middle of a prepared region instead
      const r = REGIONS[0].bbox, cx = (r[0] + r[2]) / 2, cy = (r[1] + r[3]) / 2, h = Math.min(0.05, MAX_SIDE / 2);
      ex = [cx - h, cy - h, cx + h, cy + h];
    }
    setBox(ex);
    fitBox(state.bbox);
  };

  // ---------- place search ----------
  let placeMarker = null, searchSeq = 0;
  function areaAround(r) {
    // use the place's own outline when it fits, otherwise a square around it (~17 km a side)
    const side = 0.15, half = (v) => v / 2;
    if (r.bbox) {
      const w = r.bbox[2] - r.bbox[0], h = r.bbox[3] - r.bbox[1];
      if (w >= 0.05 && h >= 0.05 && w <= MAX_SIDE * 0.95 && h <= MAX_SIDE * 0.95) return r.bbox.slice();
      const s = Math.min(Math.max(w, h, side), MAX_SIDE * 0.95);
      return [r.lon - half(s), r.lat - half(s), r.lon + half(s), r.lat + half(s)];
    }
    return [r.lon - half(side), r.lat - half(side), r.lon + half(side), r.lat + half(side)];
  }
  function choosePlace(r) {
    setMode(null);
    $("place-results").innerHTML = "";
    $("place-q").value = r.name;
    if (placeMarker) placeMarker.remove();
    const el = document.createElement("div"); el.className = "place-pin"; el.setAttribute("aria-label", r.name);
    placeMarker = new maplibregl.Marker({ element: el, anchor: "bottom" }).setLngLat([r.lon, r.lat])
      .setPopup(new maplibregl.Popup({ offset: 18 }).setHTML(`<strong>${esc(r.name)}</strong>`)).addTo(map);
    const b = areaAround(r);
    setBox(b, r.name);
    fitBox(b);
  }
  async function runSearch() {
    const q = $("place-q").value.trim(), list = $("place-results");
    if (q.length < 2) { list.innerHTML = `<li class="pr-msg">Type at least two letters.</li>`; return; }
    const seq = ++searchSeq;
    list.innerHTML = `<li class="pr-msg">Searching…</li>`;
    try {
      const r = await fetch(`api/search?${new URLSearchParams({ q })}`); const j = await r.json();
      if (seq !== searchSeq) return;
      if (!r.ok) throw new Error(j.detail || "The search failed. Try again.");
      const res = j.results || [];
      if (!res.length) { list.innerHTML = `<li class="pr-msg">No places found for “${esc(q)}”. Check the spelling, try a nearby town, or draw the area on the map.</li>`; return; }
      if (res.length === 1) { choosePlace(res[0]); return; }
      list.innerHTML = res.map((p, i) => `<li><button type="button" data-i="${i}"><span class="pr-name">${esc(p.name)}</span><span class="pr-label">${esc(p.label)}</span></button></li>`).join("")
        + `<li class="pr-attr">${esc(j.attribution || "")}</li>`;
      list.querySelectorAll("button").forEach(btn => btn.onclick = () => choosePlace(res[+btn.dataset.i]));
      list.querySelector("button").focus();
    } catch (err) {
      if (seq !== searchSeq) return;
      list.innerHTML = `<li class="pr-msg is-error-text">${esc(err.message.includes("fetch") ? "Can't reach the FloodCheck server." : err.message)}</li>`;
    }
  }
  $("place-btn").onclick = runSearch;
  $("place-q").addEventListener("keydown", (e) => { if (e.key === "Enter") { e.preventDefault(); runSearch(); } });

  map.on("click", (e) => {
    const p = [e.lngLat.lng, e.lngLat.lat];
    if (state.mode === "draw") {
      if (!state.firstCorner) { state.firstCorner = p; setHint("Now click the opposite corner"); }
      else { const b = normBox(state.firstCorner, p); setMode(null); setBox(b); sheetTo("half"); }
      return;
    }
    if (state.mode === "inflow") { addInflow(p[0], p[1]); setMode(null); sheetTo("half"); return; }
    if (state.mode === "profile") {
      state.profilePts.push(p); drawProfileLine();
      if (state.profilePts.length >= 2) setHint(`${state.profilePts.length} points. Tap more, or finish the line.`, { label: "Finish", run: finishProfile });
      return;
    }
    if (!state.result) return;
    const hitLayers = resultLayerIds();
    const hits = hitLayers.length ? map.queryRenderedFeatures(e.point, { layers: hitLayers }) : [];
    if (hits.length) {
      const cfg = LAYERS.find(l => hits[0].layer.id.startsWith(`res-${l.id}`));
      showPopup(e.lngLat, cfg.popup(hits[0].properties));
    } else queryDepth(e.lngLat);
  });
  map.on("dblclick", () => {
    if (state.mode !== "profile") return;
    // the double-click also fired two clicks: drop the duplicate point
    if (state.profilePts.length > 2) state.profilePts.pop();
    finishProfile();
  });
  map.on("mousemove", (e) => {
    if (state.mode === "draw" && state.firstCorner)
      map.getSource("box").setData(boxFeature(normBox(state.firstCorner, [e.lngLat.lng, e.lngLat.lat])));
    else if (!state.mode && state.result) {
      const ids = resultLayerIds();
      const over = ids.length && map.queryRenderedFeatures(e.point, { layers: ids }).length;
      map.getCanvas().style.cursor = over ? "pointer" : "";
    }
  });
  function resultLayerIds() {
    return LAYERS.filter(l => l.popup).flatMap(l => [`res-${l.id}`, `res-${l.id}-pts`]).filter(id => map.getLayer(id));
  }
  document.addEventListener("keydown", (e) => {
    if (e.key === "Escape" && state.mode) { setMode(null); setBox(state.bbox); }
  });

  // ---------- inflows ----------
  let inflowSeq = 0;
  function addInflow(lng, lat) {
    const f = { id: ++inflowSeq, lng, lat, flood: "", normal: "" };
    const el = document.createElement("div");
    el.style.cssText = "width:18px;height:18px;border-radius:50%;background:#08519C;border:3px solid #fff;box-shadow:0 0 0 1.5px #0B1650;cursor:grab";
    el.setAttribute("aria-label", `River inflow ${f.id}`);
    f.marker = new maplibregl.Marker({ element: el, draggable: true }).setLngLat([lng, lat]).addTo(map);
    f.marker.on("dragend", () => { const ll = f.marker.getLngLat(); f.lng = ll.lng; f.lat = ll.lat; renderInflows(); });
    state.inflows.push(f); renderInflows();
  }
  function renderInflows() {
    const list = $("inflow-list"); list.innerHTML = "";
    state.inflows.forEach((f, idx) => {
      const div = document.createElement("div"); div.className = "inflow";
      div.innerHTML = `
        <div class="inflow-top"><strong>River inflow ${idx + 1}</strong>
          <button class="btn-link" data-remove="${f.id}">Remove</button></div>
        <div class="inflow-grid">
          <div class="field"><label for="fl-${f.id}">Flood flow</label>
            <div class="unit-input"><input type="number" min="1" id="fl-${f.id}" value="${f.flood}" placeholder="e.g. 1500"><span>m³/s</span></div></div>
          <div class="field"><label for="nm-${f.id}">Normal flow</label>
            <div class="unit-input"><input type="number" min="0" id="nm-${f.id}" value="${f.normal}" placeholder="e.g. 150"><span>m³/s</span></div></div>
        </div>
        <p class="hint" style="margin:8px 0 0">${f.lat.toFixed(4)}, ${f.lng.toFixed(4)}. Drag the blue dot to move it.</p>
        <button class="btn-link" data-flow="${f.id}" style="margin-top:6px">Get real flow for the storm dates</button>
        <p class="hint flow-note" style="margin:6px 0 0">${esc(f.flowNote || "")}</p>`;
      list.appendChild(div);
      div.querySelector(`#fl-${f.id}`).oninput = (e) => { f.flood = e.target.value === "" ? "" : +e.target.value; updateRunButton(); };
      div.querySelector(`#nm-${f.id}`).oninput = (e) => { f.normal = e.target.value === "" ? "" : +e.target.value; };
      div.querySelector("[data-remove]").onclick = () => { f.marker.remove(); state.inflows = state.inflows.filter(x => x !== f); renderInflows(); };
      div.querySelector("[data-flow]").onclick = () => getRiverFlow(f);
    });
    $("step-inflow").classList.toggle("is-done", state.inflows.length > 0 && state.inflows.every(f => f.flood > 0));
    updateRunButton();
  }
  $("inflow-btn").onclick = () => setMode(state.mode === "inflow" ? null : "inflow");

  // ---------- storm inputs ----------
  $("rain-range").oninput = (e) => { $("rain-mm").value = e.target.value; updateRunButton(); };
  $("rain-mm").oninput = (e) => { $("rain-range").value = Math.min(400, +e.target.value || 10); updateRunButton(); };
  $("rain-hours").oninput = updateRunButton;

  // ---------- real weather for a date ----------
  const todayISO = () => new Date(Date.now() - new Date().getTimezoneOffset() * 60000).toISOString().slice(0, 10);
  const addDays = (iso, n) => { const d = new Date(iso + "T12:00:00Z"); d.setUTCDate(d.getUTCDate() + n); return d.toISOString().slice(0, 10); };
  const niceDate = (iso) => new Date(iso + "T12:00:00Z").toLocaleDateString("en-GB", { day: "numeric", month: "short", year: "numeric", timeZone: "UTC" });
  $("wx-date").value = todayISO();
  $("wx-date").max = addDays(todayISO(), 15);
  function syncRainSource() {
    const real = document.querySelector('input[name="rainsrc"]:checked').value === "real";
    $("real-weather").hidden = !real; $("own-presets").hidden = real;
    updateRunButton();
  }
  document.querySelectorAll('input[name="rainsrc"]').forEach(r => r.onchange = syncRainSource);
  const presetBtns = document.querySelectorAll(".presets button");
  presetBtns.forEach(b => b.onclick = () => {
    $("rain-mm").value = b.dataset.mm; $("rain-range").value = b.dataset.mm; $("rain-hours").value = b.dataset.h;
    presetBtns.forEach(x => x.classList.toggle("is-on", x === b)); updateRunButton();
  });
  ["rain-mm", "rain-range", "rain-hours"].forEach(id => $(id).addEventListener("input", () => presetBtns.forEach(x => x.classList.remove("is-on"))));
  state.weather = null;
  syncRainSource();
  $("wx-btn").onclick = async () => {
    const out = $("wx-result");
    if (!state.bbox) { out.innerHTML = `<p class="hint is-error-text">Mark your area first (step 1), so the rainfall is for the right place.</p>`; return; }
    out.innerHTML = `<p class="hint">Getting rainfall…</p>`;
    try {
      const q = new URLSearchParams({ bbox: state.bbox.join(","), date: $("wx-date").value, days: $("wx-days").value });
      const r = await fetch(`api/weather?${q}`); const j = await r.json();
      if (!r.ok) throw new Error(j.detail || "The rainfall request failed.");
      state.weather = j;
      $("rain-mm").value = Math.max(1, Math.round(j.rain_mm)); $("rain-range").value = Math.min(400, Math.max(10, j.rain_mm));
      $("rain-hours").value = j.rain_hours;
      document.getElementById(`wet-${j.wetness}`).checked = true;
      out.innerHTML = weatherChart(j) + `<p class="hint" style="margin:6px 0 0">
        <strong style="color:#0B1650">${fmt(j.rain_mm)} mm</strong> ${j.is_forecast ? "expected" : "fell"} over about ${j.rain_hours} wet hours, ${niceDate(j.start)} to ${niceDate(j.end)}${j.is_forecast ? " (forecast)" : ""}.
        The ground was <strong style="color:#0B1650">${j.wetness}</strong>: ${fmt(j.antecedent_mm)} mm fell in the 5 days before.
        ${j.rain_mm < 5 ? "<br>Very little rain fell on these dates; try another date." : ""}<br>
        <span style="font-size:12px">${esc(j.source)}. Values are averaged over your area, so local downpours can be higher.</span></p>`;
      updateRunButton();
    } catch (err) { out.innerHTML = `<p class="hint is-error-text">${esc(err.message)}</p>`; }
  };
  function weatherChart(j) {
    const W = 300, H = 90, pad = 16, n = j.daily.length, bw = (W - 8) / n;
    const mx = Math.max(5, ...j.daily.map(d => d.rain_mm));
    const bars = j.daily.map((d, i) => {
      const h = (d.rain_mm / mx) * (H - pad - 14);
      return `<rect x="${4 + i * bw + 1}" y="${H - pad - h}" width="${Math.max(bw - 2, 1)}" height="${h}" fill="${d.in_storm ? "#1E86B8" : "#C9D2DE"}"><title>${niceDate(d.date)}: ${d.rain_mm} mm</title></rect>`;
    }).join("");
    return `<svg viewBox="0 0 ${W} ${H}" style="width:100%;margin-top:10px;background:#fff;border:1px solid #D5DCE6;border-radius:8px" role="img" aria-label="Daily rainfall">
      ${bars}<text x="4" y="12" font-size="10" fill="#4A5578">${fmt(mx, 0)} mm/day</text>
      <text x="4" y="${H - 4}" font-size="10" fill="#4A5578">${niceDate(j.daily[0].date)}</text>
      <text x="${W - 4}" y="${H - 4}" font-size="10" fill="#4A5578" text-anchor="end">${niceDate(j.daily[n - 1].date)}</text>
      <text x="${W - 4}" y="12" font-size="10" fill="#1E86B8" text-anchor="end">rain days used</text></svg>`;
  }
  async function getRiverFlow(f) {
    f.flowNote = "Getting river flow…"; renderInflows();
    try {
      const q = new URLSearchParams({ lat: f.lat, lon: f.lng, date: $("wx-date").value, days: $("wx-days").value });
      const r = await fetch(`api/river-flow?${q}`); const j = await r.json();
      if (!r.ok) throw new Error(j.detail || "The river flow request failed.");
      f.flood = j.flood_m3s; f.normal = j.normal_m3s;
      f.flowNote = `Peak ${fmt(j.flood_m3s, 0)} m³/s on ${niceDate(j.peak_date)}; typical ${fmt(j.normal_m3s, 0)} m³/s over the year before. ${j.source}. Uses the dates under Real weather.`;
    } catch (err) { f.flowNote = err.message; }
    renderInflows();
  }

  // ---------- running ----------
  $("run-btn").onclick = async () => {
    const body = {
      bbox: state.bbox, rain_mm: +$("rain-mm").value, rain_hours: +$("rain-hours").value,
      wetness: document.querySelector('input[name="wetness"]:checked').value,
      inflows: state.inflows.map(f => ({ lat: f.lat, lon: f.lng, flood_m3s: +f.flood, normal_m3s: +(f.normal || 0) })),
    };
    statusFollow.on = true; statusFollow.lastLen = -1;   // follow the progress list until the user scrolls away
    showTab("results");
    $("results-empty").hidden = true; $("results-badge").hidden = true;
    showStatus("Starting the flood check", "", []);
    $("results").hidden = true;
    try {
      const r = await fetch("api/runs", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(body) });
      const j = await r.json();
      if (!r.ok) throw new Error(typeof j.detail === "string" ? j.detail : "The server rejected the request. Check the inputs.");
      state.runId = j.id;
      state.polling = setInterval(poll, 1500);
      updateRunButton(); poll();
    } catch (err) {
      showError(err.message.includes("fetch") ? "Can't reach the FloodCheck server. Check your internet connection and try again." : err.message);
    }
  };

  function showStatus(title, text, log, current) {
    const s = $("status"); s.hidden = false; s.classList.remove("is-error");
    if (sheet.name === "peek") sheetTo("half");
    $("status-title").textContent = title; $("status-text").textContent = text;
    const ul = $("status-log"); ul.innerHTML = "";
    log.forEach((m, i) => {
      const li = document.createElement("li"); li.textContent = m;
      if (current && i === log.length - 1) li.className = "is-current";
      ul.appendChild(li);
    });
    followStatus();
  }

  // ---------- keep the progress list in view while a check runs ----------
  const statusFollow = { on: false, lastLen: -1 };
  function followStatus() {
    if (!statusFollow.on) return;
    const box = $("panel-scroll"), s = $("status");
    const n = $("status-log").children.length;
    if (n === statusFollow.lastLen && s.dataset.followed === "1") return;   // nothing new to show
    statusFollow.lastLen = n; s.dataset.followed = "1";
    requestAnimationFrame(() => {
      const top = s.offsetTop - 8;                                          // panel-scroll is the offset parent
      const bottom = s.offsetTop + s.offsetHeight - box.clientHeight + 12;  // status bottom at the panel's bottom edge
      // show the newest step: the bottom of the status card, without hiding its title when it fits
      const target = bottom <= top ? Math.min(Math.max(box.scrollTop, bottom), top) : bottom;
      if (Math.abs(target - box.scrollTop) > 2) box.scrollTo({ top: target, behavior: reduced() ? "auto" : "smooth" });
    });
  }
  // the user taking over the scroll stops the auto-follow for this check
  ["wheel", "touchmove", "keydown"].forEach(ev => $("panel-scroll").addEventListener(ev, (e) => {
    if (ev === "keydown" && !["ArrowUp", "ArrowDown", "PageUp", "PageDown", "Home", "End", " "].includes(e.key)) return;
    statusFollow.on = false;
  }, { passive: true }));
  function showError(msg) {
    stopPolling();
    statusFollow.on = true; statusFollow.lastLen = -1; $("status").dataset.followed = "";
    const s = $("status"); s.hidden = false; s.classList.add("is-error");
    $("status-title").textContent = "The flood check didn't finish";
    $("status-text").textContent = msg; $("status-log").innerHTML = "";
    followStatus();
  }
  function stopPolling() { if (state.polling) clearInterval(state.polling); state.polling = null; updateRunButton(); }

  let pollBusy = false;
  async function poll() {
    if (!state.runId || pollBusy || !state.polling) return;
    pollBusy = true;
    let j;
    try {
      const r = await fetch(`api/runs/${state.runId}`); j = await r.json();
      if (!r.ok) throw new Error(j.detail || "Lost track of the run.");
    } catch (err) { pollBusy = false; showError("Lost contact with the FloodCheck server. It may have restarted (for example if it ran out of memory). Please try again, or try a smaller area."); return; }
    pollBusy = false;
    if (!state.polling) return;
    if (j.status === "queued") showStatus("Waiting for another check to finish", "", [], false);
    else if (j.status === "running") showStatus("Checking for floods", `${fmt(j.elapsed_s, 0)} seconds so far`, j.log, true);
    else if (j.status === "failed") showError(j.error);
    else if (j.status === "done") { stopPolling(); showStatus("Done", `Took ${fmt(j.elapsed_s, 0)} seconds`, j.log, false); showResult(j); }
  }

  // ---------- results on the map ----------
  function removeResultLayers(m) {
    ["res-facilities", "res-roads", "res-buildings-pts", "res-buildings", "flood"].forEach(id => { if (m.getLayer(id)) m.removeLayer(id); });
    ["res-facilities", "res-roads", "res-buildings", "flood"].forEach(id => { if (m.getSource(id)) m.removeSource(id); });
  }
  async function showResult(j) {
    state.result = j;
    removeResultLayers(map);
    const url = j.image_url + "?t=" + Date.now();
    map.addSource("flood", { type: "image", url, coordinates: j.corners });
    map.addLayer({ id: "flood", type: "raster", source: "flood",
                   paint: { "raster-opacity": +$("opacity").value, "raster-resampling": "nearest" } }, "box-fill");
    for (const cfg of LAYERS.filter(l => l.kind !== "image")) {
      if (!j.layers || !j.layers[cfg.id]) continue;
      map.addSource(`res-${cfg.id}`, { type: "geojson", data: j.layers[cfg.id] });
      const vis = { visibility: state.layerOn[cfg.id] === false ? "none" : "visible" };
      const def = { id: `res-${cfg.id}`, type: cfg.kind, source: `res-${cfg.id}`, paint: cfg.paint, layout: vis };
      if (cfg.filter) def.filter = cfg.filter;
      if (cfg.minzoom != null) def.minzoom = cfg.minzoom;
      map.addLayer(def, "profile-line");
      if (cfg.extra) {
        const x = cfg.extra, xdef = { id: `res-${cfg.id}-${x.suffix}`, type: x.type, source: `res-${cfg.id}`, paint: x.paint, layout: vis, filter: x.filter };
        if (x.maxzoom != null) xdef.maxzoom = x.maxzoom;
        map.addLayer(xdef, "profile-line");
      }
    }
    if (state.layerOn.flood === false) map.setLayoutProperty("flood", "visibility", "none");
    $("legend").hidden = false;
    renderLayersCard(j);
    renderSummary(j.summary);
    if (state.compare) { setResultVisibility(true); syncAfterFlood(); }
    if (state.profilePts.length >= 2) runProfile(state.profilePts);
  }
  $("opacity").oninput = (e) => {
    if (map.getLayer("flood")) map.setPaintProperty("flood", "raster-opacity", +e.target.value);
    if (after && after.getLayer("flood")) after.setPaintProperty("flood", "raster-opacity", +e.target.value);
  };

  function renderLayersCard(j) {
    const e = (j.summary && j.summary.exposure) || {};
    const rows = $("layer-rows"); rows.innerHTML = "";
    LAYERS.forEach(cfg => {
      const available = cfg.kind === "image" || (j.layers && j.layers[cfg.id]);
      const row = document.createElement("label"); row.className = "layer-row";
      const on = state.layerOn[cfg.id] !== false && available;
      row.innerHTML = `<input type="checkbox" ${on ? "checked" : ""} ${available ? "" : "disabled"}>
        <span class="sw ${cfg.swatchClass || ""}" style="background:${cfg.swatch}"></span>
        <span>${cfg.label}</span><span class="count">${available ? (cfg.count ? cfg.count(e) : "") : "not available"}</span>`;
      row.querySelector("input").onchange = (ev) => {
        state.layerOn[cfg.id] = ev.target.checked;
        const ids = cfg.kind === "image" ? ["flood"] : [`res-${cfg.id}`, `res-${cfg.id}-pts`];
        ids.forEach(id => { if (map.getLayer(id)) map.setLayoutProperty(id, "visibility", ev.target.checked ? "visible" : "none"); });
      };
      rows.appendChild(row);
    });
    const card = $("layers-card");
    if (card.hidden && isPhone()) card.removeAttribute("open");
    card.hidden = false;
  }

  // ---------- plain-language risk verdict ----------
  const VERDICTS = [
    { title: "Low risk", text: "This rain is not expected to flood much land in your area.",
      todo: ["Keep drains, culverts and furrows clear", "Keep watching the weather forecast"] },
    { title: "Some flooding likely", text: "Low land near rivers and streams may go under water.",
      todo: ["Do not walk or drive through moving water", "Move animals, seed and stored food off low ground", "Check on neighbours who live near the river"] },
    { title: "Serious flooding likely", text: "Homes, roads or important places are in the flood water in this check.",
      todo: ["Agree where your family would go on higher ground", "Keep documents, medicines and a phone charger ready to carry", "Do not cross flooded roads or bridges", "Tell local leaders and share the warning"] },
    { title: "Severe flooding likely", text: "Many homes or important places are in deep water in this check.",
      todo: ["Be ready to leave early, before the water rises", "Move to higher ground when told to", "Never walk, swim or drive through flood water", "Follow instructions from local authorities"] },
  ];
  function riskLevel(s) {
    const e = s.exposure || {}, b = e.buildings_flooded || {}, fac = e.facilities_flooded || [];
    const share = s.flooded_share_pct || 0;
    if (!(s.flooded_area_km2 > 0)) return 0;
    let lv = 0;
    if (share >= 1 || (b.any || 0) > 0 || (e.roads_flooded_km || 0) >= 0.5) lv = 1;
    if (share >= 5 || (b["over_0.5m"] || 0) >= 20 || fac.length > 0 || (e.main_roads_flooded_km || 0) >= 1) lv = 2;
    if (share >= 15 || (b["over_1.0m"] || 0) >= 100 || fac.some(f => f.depth > 1)) lv = 3;
    return lv;
  }
  function renderVerdict(s) {
    const lv = riskLevel(s), v = VERDICTS[lv], e = s.exposure || {};
    $("verdict").dataset.level = lv;
    $("verdict-title").textContent = v.title;
    let text = v.text;
    if (e.buildings_flooded && e.buildings_flooded.any) text += ` About ${fmt(e.buildings_flooded.any, 0)} buildings have water around them.`;
    $("verdict-text").textContent = text;
    $("verdict-do").innerHTML = v.todo.map(t => `<li>${esc(t)}</li>`).join("");
    $("verdict").querySelectorAll(".meter span").forEach((el, i) => el.classList.toggle("is-on", i <= lv));
  }

  function renderSummary(s) {
    renderVerdict(s);
    $("res-area").textContent = `${fmt(s.flooded_area_km2, s.flooded_area_km2 >= 10 ? 0 : 1)} km² under water`;
    $("res-sub").textContent = `That is ${fmt(s.flooded_share_pct)}% of your area, for ${fmt(s.rain_mm, 0)} mm of rain over ${fmt(s.rain_hours)} hours on ${s.wetness} ground.`;
    $("res-max").textContent = `${fmt(s.max_depth_m)} m`;
    $("res-mean").textContent = `${fmt(s.mean_depth_m, 2)} m`;
    $("res-deep").textContent = `${fmt(s.area_over_1m_km2)} km²`;
    $("res-runoff").textContent = `${fmt(s.mean_runoff_pct, 0)}%`;

    const e = s.exposure || {};
    const notes = [];
    (s.inflows || []).forEach((f, i) => notes.push(`River inflow ${i + 1}: added ${fmt(f.added_m3s, 0)} m³/s along ${fmt(f.river_km, 0)} km of river.`));
    if (s.capped_segments) notes.push(`${s.capped_segments} river segment(s) reached the 15 m limit. Check the inflow values.`);
    if (s.soil_note) notes.push(s.soil_note);
    const wxUsed = state.weather && Math.round(state.weather.rain_mm) === Math.round(s.rain_mm);
    if (wxUsed) notes.push(`Rainfall from ${state.weather.source}, ${niceDate(state.weather.start)} to ${niceDate(state.weather.end)}${state.weather.is_forecast ? " (forecast)" : ""}.`);
    (e.notes || []).forEach(n => notes.push(n));
    if (e.buildings_truncated) notes.push("Only the 25,000 most deeply flooded buildings are drawn on the map; the counts include all of them.");
    $("res-notes").innerHTML = notes.map(n => `<li>${esc(n)}</li>`).join("");

    // exposure block
    const hasExpo = e.buildings_flooded || e.roads_flooded_km != null || e.facilities_flooded;
    $("expo").hidden = !hasExpo;
    if (e.buildings_flooded) {
      const b = e.buildings_flooded;
      $("expo-buildings-head").innerHTML = `<strong style="color:#0B1650;font-size:20px">${fmt(b.any, 0)}</strong> of ${fmt(e.buildings_total, 0)} buildings have flood water around them. How deep:`;
      const bands = [["0.1–0.5 m", b["over_0.1m"] - b["over_0.5m"]], ["0.5–1 m", b["over_0.5m"] - b["over_1.0m"]],
                     ["1–2 m", b["over_1.0m"] - b["over_2.0m"]], ["Over 2 m", b["over_2.0m"]]];
      const maxN = Math.max(1, ...bands.map(x => x[1]));
      const cols = ["#F2C14E", "#E8A13A", "#E07A1F", "#B3261E"];
      $("expo-bands").innerHTML = bands.map(([lab, n], i) =>
        `<li><span>${lab}</span><span class="band-bar" style="width:${Math.max(2, 100 * n / maxN)}%;background:${cols[i]}"></span><span class="n">${fmt(n, 0)}</span></li>`).join("");
    } else { $("expo-buildings-head").textContent = ""; $("expo-bands").innerHTML = ""; }
    $("expo-roads").innerHTML = e.roads_flooded_km != null ?
      `<strong style="color:#0B1650">${fmt(e.roads_flooded_km)} km</strong> of road under more than 30 cm of water, including ${fmt(e.main_roads_flooded_km)} km of main roads.` : "";
    const fac = e.facilities_flooded || [];
    if (e.facilities_total) {
      const tot = e.facilities_total;
      $("expo-fac-head").innerHTML = fac.length ?
        `<strong style="color:#0B1650">${fac.length}</strong> of ${tot.school + tot.health} schools and health facilities in flood water:` :
        `None of the ${tot.school + tot.health} schools and health facilities mapped here are flooded.`;
      $("expo-fac").innerHTML = fac.slice(0, 30).map((f, i) =>
        `<li><button data-i="${i}"><span>${esc(f.name)}</span><span>${fmt(f.depth, 1)} m</span></button></li>`).join("");
      $("expo-fac").querySelectorAll("button").forEach(btn => btn.onclick = () => {
        const f = fac[+btn.dataset.i];
        map.flyTo({ center: [f.lon, f.lat], zoom: 15, duration: reduced() ? 0 : 1000 });
        showPopup({ lng: f.lon, lat: f.lat }, LAYERS.find(l => l.id === "facilities").popup(f));
      });
    } else { $("expo-fac-head").textContent = ""; $("expo-fac").innerHTML = ""; }

    $("res-download").href = state.result.geotiff_url;
    $("res-pdf").href = state.result.report_url || "#";
    $("report-ready").hidden = false; $("report-empty").hidden = true;
    $("results-empty").hidden = true;
    $("results-badge").hidden = false; $("tab-results").classList.add("is-done");
    $("status").hidden = true;                       // the progress list has done its job; results go to the top
    $("results").hidden = false;
    sheetTo("half");
    statusFollow.on = false; statusFollow.lastLen = -1; $("status").dataset.followed = "";
    $("panel-scroll").scrollTop = 0;
  }

  let popup;
  function showPopup(ll, html) {
    if (popup) popup.remove();
    popup = new maplibregl.Popup({ closeButton: true, maxWidth: "260px" }).setLngLat(ll).setHTML(html).addTo(map);
  }
  async function queryDepth(ll) {
    if (!inBox(ll.lng, ll.lat, state.bbox)) return;
    try {
      const r = await fetch(`api/runs/${state.runId}/depth?lat=${ll.lat}&lon=${ll.lng}`); const j = await r.json();
      showPopup(ll, j.permanent_water ? "River or lake (always wet)" :
                    j.depth_m > 0 ? `About <strong>${fmt(j.depth_m, 2)} m</strong> of water` : "Dry in this check");
    } catch (err) { console.warn("Depth lookup failed:", err); }
  }

  // ---------- cross-section ----------
  $("profile-btn").onclick = () => setMode(state.mode === "profile" ? null : "profile");
  // (the cross-section card stays open while you use it)
  function drawProfileLine() {
    const pts = state.profilePts;
    const feats = pts.map(p => ({ type: "Feature", properties: {}, geometry: { type: "Point", coordinates: p } }));
    if (pts.length >= 2) feats.push({ type: "Feature", properties: {}, geometry: { type: "LineString", coordinates: pts } });
    whenReady(() => map.getSource("profile").setData({ type: "FeatureCollection", features: feats }));
  }
  async function runProfile(pts) {
    $("profile-readout").textContent = "Measuring along the line…";
    try {
      const r = await fetch(`api/runs/${state.runId}/profile`, { method: "POST", headers: { "Content-Type": "application/json" },
                                                                   body: JSON.stringify({ coords: pts }) });
      const j = await r.json();
      if (!r.ok) throw new Error(j.detail || "The cross-section failed.");
      if (!j.ground_m.some(g => g != null)) {
        $("profile-chart").innerHTML = ""; $("profile-readout").textContent = "The line is outside your area. Draw it inside the rectangle.";
        return;
      }
      drawProfileChart(j);
    } catch (err) { $("profile-readout").textContent = err.message; }
  }
  function drawProfileChart(p) {
    const W = 320, H = 190, m = { l: 40, r: 8, t: 10, b: 26 };
    const xs = p.distance_m, g = p.ground_m, w = p.water_m;
    const vals = g.filter(v => v != null).concat(w.filter(v => v != null));
    let lo = Math.min(...vals), hi = Math.max(...vals);
    const pad = Math.max((hi - lo) * 0.12, 0.5); lo -= pad; hi += pad;
    const X = (d) => m.l + (d / p.length_m) * (W - m.l - m.r);
    const Y = (v) => H - m.b - ((v - lo) / (hi - lo)) * (H - m.t - m.b);
    let groundPath = "", started = false;
    xs.forEach((d, i) => { if (g[i] == null) { started = false; return; }
      groundPath += `${started ? "L" : "M"}${X(d).toFixed(1)},${Y(g[i]).toFixed(1)}`; started = true; });
    // ground fill
    const valid = xs.map((d, i) => [d, g[i]]).filter(x => x[1] != null);
    const groundFill = valid.length ? `M${X(valid[0][0])},${H - m.b}` + valid.map(([d, v]) => `L${X(d).toFixed(1)},${Y(v).toFixed(1)}`).join("") +
                       `L${X(valid[valid.length - 1][0])},${H - m.b}Z` : "";
    // water polygons for each wet run
    let water = "";
    let run = [];
    const flush = () => {
      if (run.length >= 1) {
        const top = run.map(i => `${X(xs[i]).toFixed(1)},${Y(w[i]).toFixed(1)}`);
        const bot = run.slice().reverse().map(i => `${X(xs[i]).toFixed(1)},${Y(g[i]).toFixed(1)}`);
        if (run.length === 1) { const i = run[0]; top.push(`${(X(xs[i]) + 1.5).toFixed(1)},${Y(w[i]).toFixed(1)}`); bot.unshift(`${(X(xs[i]) + 1.5).toFixed(1)},${Y(g[i]).toFixed(1)}`); }
        water += `<polygon points="${top.concat(bot).join(" ")}" fill="#3182BD" fill-opacity="0.75"/>`;
      }
      run = [];
    };
    xs.forEach((d, i) => { if (w[i] != null && g[i] != null) run.push(i); else flush(); }); flush();
    const river = xs.map((d, i) => p.permanent_water[i] && g[i] != null ?
      `<line x1="${X(d).toFixed(1)}" x2="${X(d).toFixed(1)}" y1="${Y(g[i]).toFixed(1)}" y2="${(Y(g[i]) + 4).toFixed(1)}" stroke="#5A6E82" stroke-width="2"/>` : "").join("");
    const km = p.length_m / 1000;
    $("profile-chart").innerHTML = `
      <svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Cross-section of ground and flood water along the drawn line">
        <path d="${groundFill}" fill="#D9C7A7"/>
        ${water}${river}
        <path d="${groundPath}" fill="none" stroke="#6B4F2A" stroke-width="1.5"/>
        <line x1="${m.l}" x2="${W - m.r}" y1="${H - m.b}" y2="${H - m.b}" stroke="#4A5578"/>
        <text x="${m.l - 4}" y="${Y(hi - pad) + 4}" text-anchor="end" font-size="11" fill="#4A5578">${fmt(hi - pad, 0)} m</text>
        <text x="${m.l - 4}" y="${Y(lo + pad) + 4}" text-anchor="end" font-size="11" fill="#4A5578">${fmt(lo + pad, 0)} m</text>
        <text x="${m.l}" y="${H - 8}" font-size="11" fill="#4A5578">0 km</text>
        <text x="${W - m.r}" y="${H - 8}" text-anchor="end" font-size="11" fill="#4A5578">${fmt(km)} km</text>
        <line id="pf-cursor" x1="0" x2="0" y1="${m.t}" y2="${H - m.b}" stroke="#0B1650" stroke-dasharray="3 2" visibility="hidden"/>
        <rect x="${m.l}" y="${m.t}" width="${W - m.l - m.r}" height="${H - m.t - m.b}" fill="transparent" id="pf-hit"/>
      </svg>`;
    const maxDepth = Math.max(0, ...xs.map((d, i) => w[i] != null && g[i] != null ? w[i] - g[i] : 0));
    const base = maxDepth > 0 ? `Deepest water along the line: ${fmt(maxDepth, 2)} m. Hover the chart for details.` : "No flood water along this line.";
    $("profile-readout").textContent = base;
    const svg = $("profile-chart").querySelector("svg"), cur = svg.querySelector("#pf-cursor");
    svg.querySelector("#pf-hit").addEventListener("pointermove", (ev) => {
      const r = svg.getBoundingClientRect();
      const sx = (ev.clientX - r.left) * (W / r.width);
      const d = Math.min(Math.max((sx - m.l) / (W - m.l - m.r), 0), 1) * p.length_m;
      let k = 0; while (k < xs.length - 1 && xs[k] < d) k++;
      cur.setAttribute("x1", X(xs[k])); cur.setAttribute("x2", X(xs[k])); cur.setAttribute("visibility", "visible");
      const depth = w[k] != null && g[k] != null ? w[k] - g[k] : 0;
      $("profile-readout").textContent = g[k] == null ? "Outside your area" :
        `${fmt(xs[k] / 1000, 2)} km along: ground ${fmt(g[k], 1)} m` + (p.permanent_water[k] ? ", river or lake" : depth > 0 ? `, water ${fmt(depth, 2)} m deep` : ", dry");
    });
    svg.querySelector("#pf-hit").addEventListener("pointerleave", () => { cur.setAttribute("visibility", "hidden"); $("profile-readout").textContent = base; });
  }

  // ---------- compare two images ----------
  let after = null, afterReady = false, swipeX = 0.5;
  function setResultVisibility(hideAll) {
    LAYERS.forEach(cfg => {
      const ids = cfg.kind === "image" ? ["flood"] : [`res-${cfg.id}`, `res-${cfg.id}-pts`];
      const show = !hideAll && state.layerOn[cfg.id] !== false;
      ids.forEach(id => { if (map.getLayer(id)) map.setLayoutProperty(id, "visibility", show ? "visible" : "none"); });
    });
  }
  function setImageLayer(m, id, src, beforeId) {
    if (m.getLayer(id)) m.removeLayer(id);
    if (m.getSource(id)) m.removeSource(id);
    if (!src) return;
    m.addSource(id, src);
    m.addLayer({ id, type: "raster", source: id }, beforeId && m.getLayer(beforeId) ? beforeId : undefined);
    // make sure newly arrived image tiles are drawn straight away
    m.on("sourcedata", (e) => { if (e.sourceId === id && e.tile) m.triggerRepaint(); });
  }
  function imageArea() {
    if (state.bbox) return state.bbox;
    const b = map.getBounds();
    const c = map.getCenter(), h = 0.2;   // no area drawn: use the middle of the view
    return [Math.max(b.getWest(), c.lng - h), Math.max(b.getSouth(), c.lat - h), Math.min(b.getEast(), c.lng + h), Math.min(b.getNorth(), c.lat + h)];
  }
  const SIDE = { left: { req: 0 }, right: { req: 0 } };
  async function loadSide(side) {
    const m = side === "left" ? map : after;
    if (side === "right" && !afterReady) return;
    const srcType = $(`cmp-${side}-src`).value, date = $(`cmp-${side}`).value, status = $(`cmp-${side}-status`);
    const id = `cmp-${side}`, before = side === "left" ? "box-fill" : undefined;
    const req = ++SIDE[side].req;
    if (!date) { status.textContent = "Choose a date."; return; }
    if (srcType === "modis") {
      setImageLayer(m, id, { type: "raster", tiles: gibsTiles(date), tileSize: 256, maxzoom: 9, attribution: GIBS_ATTR }, before);
      status.textContent = `MODIS image for ${niceDate(date)}.`;
    } else {
      status.textContent = "Finding the nearest image…";
      const area = imageArea();
      try {
        const q = new URLSearchParams({ source: srcType, bbox: area.join(","), date });
        const r = await fetch(`api/imagery/search?${q}`); const j = await r.json();
        if (req !== SIDE[side].req) return;               // a newer choice replaced this one
        if (!r.ok) throw new Error(j.detail || "No image found.");
        const water = srcType === "s1" && $("cmp-water").checked;
        setImageLayer(m, id, { type: "raster", tiles: [j.tiles + (water ? "?water=true" : "")], tileSize: 256, minzoom: 8, maxzoom: 15,
                               bounds: area, attribution: srcType === "s2" ? "Sentinel-2: Copernicus / Element 84" : "Sentinel-1: Copernicus / Microsoft Planetary Computer" }, before);
        const off = j.days_from_target === 0 ? "the same day" : `${Math.abs(j.days_from_target)} day${Math.abs(j.days_from_target) === 1 ? "" : "s"} ${j.days_from_target > 0 ? "after" : "before"}`;
        status.textContent = srcType === "s2" ? `Photo from ${niceDate(j.date)} (${off}), about ${fmt(j.cloud_pct, 0)}% cloud.`
                                              : `Radar image from ${niceDate(j.date)} (${off}).`;
        if (map.getZoom() < 8) status.textContent += " Zoom in to see it.";
      } catch (err) {
        if (req !== SIDE[side].req) return;
        setImageLayer(m, id, null); status.textContent = err.message;
      }
    }
    if (side === "right") syncAfterFlood();
  }
  function syncAfterFlood() {
    if (!after || !afterReady) return;
    removeResultLayers(after);
    if (state.result && $("cmp-flood").checked) {
      after.addSource("flood", { type: "image", url: state.result.image_url, coordinates: state.result.corners });
      after.addLayer({ id: "flood", type: "raster", source: "flood", paint: { "raster-opacity": +$("opacity").value, "raster-resampling": "nearest" } });
    }
  }
  function placeSwipe() {
    const rect = $("map").getBoundingClientRect();
    const px = swipeX * rect.width;
    $("swipe").style.left = `${px}px`;
    $("map-after").style.clipPath = `inset(0 0 0 ${px}px)`;
  }
  function syncView() { if (after) after.jumpTo({ center: map.getCenter(), zoom: map.getZoom(), bearing: map.getBearing(), pitch: map.getPitch() }); }
  function setCompare(on) {
    state.compare = on;
    $("compare-btn").setAttribute("aria-pressed", String(on));
    $("compare-bar").hidden = !on; $("map-after").hidden = !on; $("swipe").hidden = !on;
    if (on) {
      setResultVisibility(true);   // the left side shows the plain earlier image
      loadSide("left");
      if (!after) {
        after = new maplibregl.Map({ container: "map-after", interactive: false, attributionControl: false,
          center: map.getCenter(), zoom: map.getZoom(), maxPitch: 75,
          style: { version: 8, sources: { dem: { type: "raster-dem", tiles: [TERRAIN_TILES], encoding: "terrarium", tileSize: 256, maxzoom: 14 } }, layers: [] } });
        after.on("load", () => {
          afterReady = true;
          if (state.terrain) after.setTerrain({ source: "dem", exaggeration: 2 });
          loadSide("right");
        });
        map.on("move", syncView);
      } else loadSide("right");
      after.resize(); syncView(); placeSwipe();
    } else {
      setImageLayer(map, "cmp-left", null);
      setResultVisibility(false);
    }
  }
  $("compare-btn").onclick = () => setCompare(!state.compare);
  ["left", "right"].forEach(side => {
    $(`cmp-${side}`).onchange = () => loadSide(side);
    $(`cmp-${side}-src`).onchange = () => loadSide(side);
  });
  $("cmp-flood").onchange = syncAfterFlood;
  $("cmp-water").onchange = () => ["left", "right"].forEach(side => { if ($(`cmp-${side}-src`).value === "s1") loadSide(side); });
  (() => {
    const handle = $("swipe-handle");
    let dragging = false;
    const move = (clientX) => {
      const rect = $("map").getBoundingClientRect();
      swipeX = Math.min(Math.max((clientX - rect.left) / rect.width, 0.02), 0.98); placeSwipe();
    };
    handle.addEventListener("pointerdown", (e) => { dragging = true; handle.setPointerCapture(e.pointerId); });
    handle.addEventListener("pointermove", (e) => { if (dragging) move(e.clientX); });
    handle.addEventListener("pointerup", () => { dragging = false; });
    handle.addEventListener("keydown", (e) => {
      if (e.key === "ArrowLeft" || e.key === "ArrowRight") { swipeX = Math.min(Math.max(swipeX + (e.key === "ArrowLeft" ? -0.05 : 0.05), 0.02), 0.98); placeSwipe(); e.preventDefault(); }
    });
    window.addEventListener("resize", () => { if (state.compare) { after.resize(); placeSwipe(); } });
  })();

  // ---------- phone layout: the panel is a bottom sheet over a full-screen map ----------
  function sheetMetrics() {
    const hdr = document.querySelector(".brandbar").getBoundingClientRect().height;
    const avail = window.innerHeight - hdr;
    const peek = $("sheet-handle").offsetHeight + document.querySelector(".run-bar").offsetHeight;
    return { hdr, peek, half: Math.max(peek, Math.round(avail * 0.52)), full: Math.max(peek, avail - 10) };
  }
  function applySheet(px) {
    const app = document.querySelector(".app");
    if (!isPhone()) { app.style.removeProperty("--sheet-h"); return; }
    const m = sheetMetrics();
    app.style.setProperty("--hdr", `${m.hdr}px`);
    app.style.setProperty("--sheet-h", `${Math.round(px)}px`);
  }
  function sheetTo(name) {
    if (!isPhone()) return;
    sheet.name = name;
    applySheet(sheetMetrics()[name]);
    const open = name !== "peek";
    $("sheet-label").textContent = open ? "Show more map" : "Show the steps";
    $("sheet-handle").setAttribute("aria-expanded", String(open));
    document.querySelector(".panel").dataset.sheet = name;
  }
  function scrollPanelTo(el) {        // scroll only the panel, never the page
    const box = $("panel-scroll");
    const top = el.getBoundingClientRect().top - box.getBoundingClientRect().top + box.scrollTop - 8;
    box.scrollTo({ top, behavior: reduced() ? "auto" : "smooth" });
  }
  function fitBox(b) {
    if (!b) return;
    const bottom = isPhone() ? sheetMetrics()[sheet.name] : 0;
    map.fitBounds([[b[0], b[1]], [b[2], b[3]]], { padding: { top: 70, left: 24, right: 24, bottom: bottom + 30 },
      duration: reduced() ? 0 : 850, maxZoom: 15 });
  }
  (() => {
    const handle = $("sheet-handle"), panel = document.querySelector(".panel");
    let drag = null;
    handle.addEventListener("pointerdown", (e) => {
      if (!isPhone()) return;
      drag = { y: e.clientY, h: panel.getBoundingClientRect().height, moved: false };
      handle.setPointerCapture(e.pointerId); panel.classList.add("is-dragging");
    });
    handle.addEventListener("pointermove", (e) => {
      if (!drag) return;
      const m = sheetMetrics(), dy = drag.y - e.clientY;
      if (Math.abs(dy) > 6) drag.moved = true;
      if (drag.moved) applySheet(Math.min(Math.max(drag.h + dy, m.peek), m.full));
    });
    const end = () => {
      if (!drag) return;
      panel.classList.remove("is-dragging");
      const m = sheetMetrics(), h = panel.getBoundingClientRect().height, moved = drag.moved; drag = null;
      if (!moved) { sheetTo(sheet.name === "peek" ? "half" : "peek"); return; }
      const nearest = ["peek", "half", "full"].reduce((a, k) => Math.abs(m[k] - h) < Math.abs(m[a] - h) ? k : a, "peek");
      sheetTo(nearest);
    };
    handle.addEventListener("pointerup", end);
    handle.addEventListener("pointercancel", end);
    handle.addEventListener("keydown", (e) => {
      if (e.key === "ArrowUp") { e.preventDefault(); sheetTo(sheet.name === "peek" ? "half" : "full"); }
      if (e.key === "ArrowDown") { e.preventDefault(); sheetTo(sheet.name === "full" ? "half" : "peek"); }
    });
    handle.addEventListener("click", (e) => { if (e.detail === 0) sheetTo(sheet.name === "peek" ? "half" : "peek"); });  // keyboard
    const relayout = () => { if (isPhone()) sheetTo(sheet.name); else applySheet(0); };
    window.addEventListener("resize", relayout);
    phoneQuery.addEventListener("change", relayout);
    relayout();
  })();

  // ---------- tabs: Area · Rain · Results · Report ----------
  function showTab(name) {
    document.querySelectorAll(".tab").forEach(t => {
      const on = t.dataset.tab === name;
      t.setAttribute("aria-selected", String(on)); t.tabIndex = on ? 0 : -1;
    });
    document.querySelectorAll(".pane").forEach(p => { p.hidden = p.id !== `pane-${name}`; });
    $("panel-scroll").scrollTop = 0;
    if (isPhone() && sheet.name === "peek") sheetTo("half");
  }
  document.querySelectorAll(".tab").forEach(t => {
    t.onclick = () => showTab(t.dataset.tab);
    t.onkeydown = (e) => {                                   // arrow keys move between tabs
      const tabs = [...document.querySelectorAll(".tab")], i = tabs.indexOf(t);
      const k = e.key === "ArrowRight" ? 1 : e.key === "ArrowLeft" ? -1 : 0;
      if (k) { e.preventDefault(); const n = tabs[(i + k + tabs.length) % tabs.length]; n.focus(); showTab(n.dataset.tab); }
    };
  });
  document.querySelectorAll("[data-goto]").forEach(b => b.onclick = () => showTab(b.dataset.goto));
  // the PDF is made on the server when asked for; show that something is happening
  $("res-pdf").addEventListener("click", (e) => {
    if ($("res-pdf").getAttribute("href") === "#") { e.preventDefault(); return; }
    const b = $("res-pdf"), label = b.lastChild.textContent;
    b.classList.add("is-busy"); b.lastChild.textContent = "Preparing the PDF…";
    setTimeout(() => { b.classList.remove("is-busy"); b.lastChild.textContent = label; }, 6000);
  });
  setBox(state.bbox);
})();


In [ ]:
# Atlas Geosystems logo files (stored here as text so the notebook is self-contained)
import base64, pathlib
LOGOS = {
    "logo-full.png": "...",
    "logo-mark.png": "...",
}
assets = pathlib.Path("/content/floodcheck/backend/static/assets")
for name, data in LOGOS.items():
    (assets / name).write_bytes(base64.b64decode(data))
print("Logo files written:", sorted(p.name for p in assets.iterdir()))

## 7. Start the app

This starts the server in the background and prints a link. The first run in a new area takes a few minutes, mostly downloading data. The very first run also takes an extra minute while the model engine compiles.

**To try it quickly:** press *Use an example area*, choose *Choose myself* and a rain amount, then *Check for floods*. To add a river, press *Mark where the river enters*, click where the main river enters the rectangle (the *Satellite* background shows it), then *Get real flow for the storm dates*.

In [ ]:
import os, sys, threading, time, urllib.request
os.chdir("/content/floodcheck/backend"); sys.path.insert(0, "/content/floodcheck/backend")
os.environ["FLOOD_RESULTS_DIR"] = "/content/floodcheck/results"
import uvicorn, server

PORT = 8000
if "flood_server" not in globals():
    flood_server = uvicorn.Server(uvicorn.Config(server.app, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=flood_server.run, daemon=True).start()
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=2); break
    except Exception:
        time.sleep(1)
print("Server is running.")

from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Open FloodCheck here (keep this Colab tab open):", url)

If the link doesn't open, run the next cell instead: it shows the app inside the notebook.

**If you edit a file in `backend/` ending in `.py`:** choose *Runtime → Restart session*, then *Run all* again. Edits to the page, styles or script only need a browser refresh.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=820)